In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

# Project location
ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

assert ROOT.exists(), f"Project folder not found: {ROOT}"

# Existing data and experiment paths
FEATURE_PATH = ROOT / "data/processed/features/respiratory_cycle_features.csv"

TRAIN_PATH = ROOT / "data/processed/deep_learning/cnn_baseline/train_cnn_manifest.csv"
VAL_PATH = ROOT / "data/processed/deep_learning/cnn_baseline/validation_cnn_manifest.csv"
TEST_PATH = ROOT / "data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv"

SOURCE_NOTEBOOK = ROOT / "Notebooks 2/05_Lightweight_MultiFeature_CNN.ipynb"

print("Project exists:", ROOT.exists())
print("Original lightweight CNN notebook:", SOURCE_NOTEBOOK.exists())
print("Feature table:", FEATURE_PATH.exists())
print("Train manifest:", TRAIN_PATH.exists())
print("Validation manifest:", VAL_PATH.exists())
print("Test manifest:", TEST_PATH.exists())

print("\nExisting files:")
for p in [FEATURE_PATH, TRAIN_PATH, VAL_PATH, TEST_PATH, SOURCE_NOTEBOOK]:
    if p.exists():
        print(f"{p.relative_to(ROOT)} — {p.stat().st_size:,} bytes")


In [ ]:

# Load the existing manifests (read-only)
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

splits = {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}

required_columns = {
    "patient_id", "recording_id", "diagnosis", "audio_path"
}

for split_name, df in splits.items():
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"{split_name} manifest is missing: {sorted(missing)}")

    print(
        f"{split_name:10s} | "
        f"cycles={len(df):4d} | "
        f"recordings={df['recording_id'].nunique():3d} | "
        f"patients={df['patient_id'].nunique():3d} | "
        f"duplicate paths={df['audio_path'].duplicated().sum()}"
    )

# Verify patient-level separation
patient_sets = {
    name: set(df["patient_id"].astype(str))
    for name, df in splits.items()
}

print("\nPatient overlap checks:")
print("Train ∩ validation:", len(patient_sets["train"] & patient_sets["validation"]))
print("Train ∩ test:", len(patient_sets["train"] & patient_sets["test"]))
print("Validation ∩ test:", len(patient_sets["validation"] & patient_sets["test"]))

# Disease support by cycles, recordings, and patients
all_classes = sorted(
    set(train_df["diagnosis"].dropna().astype(str))
    | set(val_df["diagnosis"].dropna().astype(str))
    | set(test_df["diagnosis"].dropna().astype(str))
)

rows = []
for split_name, df in splits.items():
    for disease in all_classes:
        part = df[df["diagnosis"].astype(str) == disease]
        rows.append({
            "split": split_name,
            "disease": disease,
            "cycles": len(part),
            "recordings": part["recording_id"].nunique(),
            "patients": part["patient_id"].nunique(),
        })

support_df = pd.DataFrame(rows)

print("\nClass support — cycles")
display(support_df.pivot(index="disease", columns="split", values="cycles").fillna(0).astype(int))

print("\nClass support — distinct patients")
display(support_df.pivot(index="disease", columns="split", values="patients").fillna(0).astype(int))


In [ ]:

import re

search_terms = re.compile(
    r"five.?class|5.?class|class_mapping|label_mapping|"
    r"ICBHI|Fraiwan|Bronchiectasis|Bronchiolitis",
    re.IGNORECASE,
)

search_roots = [
    ROOT / "Notebooks 2",
    ROOT / "data/processed",
]

matches = []
for search_root in search_roots:
    if not search_root.exists():
        continue

    for path in search_root.rglob("*"):
        if path.suffix.lower() not in {".py", ".ipynb", ".json", ".csv", ".md", ".txt"}:
            continue

        # Avoid scanning large generated datasets and unrelated files
        if path.stat().st_size > 5_000_000:
            continue

        try:
            content = path.read_text(encoding="utf-8", errors="ignore")
        except OSError:
            continue

        lines = content.splitlines()
        for line_no, line in enumerate(lines, start=1):
            if search_terms.search(line):
                matches.append(
                    (str(path.relative_to(ROOT)), line_no, line.strip()[:220])
                )

for filename, line_no, line in matches[:100]:
    print(f"{filename}:{line_no}: {line}")

print(f"\nTotal matching lines found: {len(matches)}")
print("Showing at most 100 matches.")


In [ ]:

# Rank diseases by patient support across the existing patient-disjoint splits.
# This is an audit only; it does not change any labels or files.

patient_support = pd.DataFrame({
    "train_patients": train_df.groupby("diagnosis")["patient_id"].nunique(),
    "validation_patients": val_df.groupby("diagnosis")["patient_id"].nunique(),
    "test_patients": test_df.groupby("diagnosis")["patient_id"].nunique(),
}).fillna(0).astype(int)

cycle_support = pd.DataFrame({
    "train_cycles": train_df["diagnosis"].value_counts(),
    "validation_cycles": val_df["diagnosis"].value_counts(),
    "test_cycles": test_df["diagnosis"].value_counts(),
}).fillna(0).astype(int)

support_audit = patient_support.join(cycle_support)
support_audit["total_patients"] = (
    support_audit["train_patients"]
    + support_audit["validation_patients"]
    + support_audit["test_patients"]
)
support_audit["total_cycles"] = (
    support_audit["train_cycles"]
    + support_audit["validation_cycles"]
    + support_audit["test_cycles"]
)

support_audit = support_audit.sort_values(
    ["total_patients", "test_patients", "total_cycles"],
    ascending=False
)

display(support_audit)

print("\nCandidate-class checks:")
for minimum_test_patients in [3, 5]:
    eligible = support_audit.index[
        (support_audit["train_patients"] >= 5)
        & (support_audit["validation_patients"] >= 2)
        & (support_audit["test_patients"] >= minimum_test_patients)
    ].tolist()
    print(
        f"At least 5 train, 2 validation, and "
        f"{minimum_test_patients} test patients: {eligible}"
    )


In [ ]:

import json
import re

with open(SOURCE_NOTEBOOK, "r", encoding="utf-8") as f:
    original_nb = json.load(f)

architecture_terms = re.compile(
    r"class\s+\w+.*(?:CNN|Branch|Net|Model)|"
    r"def\s+forward|"
    r"Conv2d|Depthwise|AdaptiveAvgPool|"
    r"Log.?Mel|MFCC|Chroma|"
    r"CrossEntropyLoss|AdamW|"
    r"EarlyStopping|macro.?f1|"
    r"DataLoader|Dataset",
    re.IGNORECASE,
)

for i, cell in enumerate(original_nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if architecture_terms.search(source):
        print(f"\n{'=' * 20} CELL {i} {'=' * 20}")
        print(source[:3500])


In [ ]:

import json

FIVE_CLASS_NOTEBOOK = (
    ROOT / "Notebooks 2" /
    "11_Lightweight_MultiFeature_CNN_5Class_Experiments.ipynb"
)

with open(FIVE_CLASS_NOTEBOOK, "r", encoding="utf-8") as f:
    five_nb = json.load(f)

for i, cell in enumerate(five_nb.get("cells", [])):
    outputs = cell.get("outputs", [])
    if not outputs:
        continue

    source = "".join(cell.get("source", []))
    output_text = []

    for out in outputs:
        if "text" in out:
            output_text.extend(out["text"])
        for item in out.get("data", {}).values():
            if isinstance(item, list):
                output_text.extend(
                    x for x in item if isinstance(x, str)
                )
            elif isinstance(item, str):
                output_text.append(item)

    text = "".join(output_text).strip()

    if text and any(
        term in (source + "\n" + text).lower()
        for term in [
            "five-class", "5-class", "macro-f1",
            "balanced accuracy", "test_accuracy",
            "class mapping", "target classes",
            "best epoch", "test metrics",
        ]
    ):
        print(f"\n{'=' * 18} CELL {i} {'=' * 18}")
        print("CODE:")
        print(source[:1200])
        print("\nRECORDED OUTPUT:")
        print(text[:2500])


In [ ]:

import json

FIVE_CLASS_NOTEBOOK = (
    ROOT / "Notebooks 2" /
    "11_Lightweight_MultiFeature_CNN_5Class_Experiments.ipynb"
)

with open(FIVE_CLASS_NOTEBOOK, "r", encoding="utf-8") as f:
    five_nb = json.load(f)

print("Notebook:", FIVE_CLASS_NOTEBOOK.name)
print("Total cells:", len(five_nb.get("cells", [])))

for i, cell in enumerate(five_nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    lines = [line.strip() for line in source.splitlines() if line.strip()]

    if cell.get("cell_type") == "markdown":
        headings = [line for line in lines if line.startswith("#")]
        if headings:
            print(f"\n[Markdown cell {i}]")
            print("\n".join(headings[:5]))

    elif cell.get("cell_type") == "code":
        print(f"\n[Code cell {i}]")
        print("\n".join(lines[:5])[:500])

        for output in cell.get("outputs", []):
            text_parts = []

            if "text" in output:
                text_parts.append("".join(output["text"]))

            for value in output.get("data", {}).values():
                if isinstance(value, list):
                    text_parts.extend(
                        item for item in value if isinstance(item, str)
                    )
                elif isinstance(value, str):
                    text_parts.append(value)

            recorded = "\n".join(text_parts).strip()
            if recorded:
                print("Recorded output:")
                print(recorded[:1200])


In [ ]:

CANDIDATE_CLASSES = [
    "COPD",
    "Healthy",
    "URTI",
    "Bronchiectasis",
    "Pneumonia",
]

combined_df = pd.concat(
    [
        train_df.assign(current_split="train"),
        val_df.assign(current_split="validation"),
        test_df.assign(current_split="test"),
    ],
    ignore_index=True,
)

candidate_df = combined_df[
    combined_df["diagnosis"].isin(CANDIDATE_CLASSES)
].copy()

# One row per disease and patient, with their recordings and cycles.
patient_summary = (
    candidate_df.groupby(["diagnosis", "patient_id"], dropna=False)
    .agg(
        current_split=("current_split", "first"),
        split_count=("current_split", "nunique"),
        recordings=("recording_id", "nunique"),
        cycles=("audio_path", "size"),
    )
    .reset_index()
    .sort_values(["diagnosis", "patient_id"])
)

print("Candidate class support:")
display(
    candidate_df.groupby("diagnosis").agg(
        patients=("patient_id", "nunique"),
        recordings=("recording_id", "nunique"),
        cycles=("audio_path", "size"),
    ).reindex(CANDIDATE_CLASSES)
)

print("\nPatient-level detail:")
display(patient_summary)

# Confirm that each patient belongs to only one existing split.
cross_split_patients = patient_summary[
    patient_summary["split_count"] > 1
]

print("\nPatients appearing in multiple splits:", len(cross_split_patients))
if not cross_split_patients.empty:
    display(cross_split_patients)


In [ ]:

# Patient and recording support by disease and existing split.
# Audit only: no resampling, relabelling, or file changes.

support_by_split = (
    candidate_df.groupby(["diagnosis", "current_split"])
    .agg(
        patients=("patient_id", "nunique"),
        recordings=("recording_id", "nunique"),
        cycles=("audio_path", "size"),
    )
    .reset_index()
)

for metric in ["patients", "recordings", "cycles"]:
    print(f"\n{metric.upper()} by disease and split")
    display(
        support_by_split.pivot(
            index="diagnosis",
            columns="current_split",
            values=metric,
        )
        .reindex(CANDIDATE_CLASSES)
        .fillna(0)
        .astype(int)
    )

# Show the actual patient IDs for the two smallest classes.
for disease in ["Bronchiectasis", "Pneumonia"]:
    print(f"\n{disease}: patients by existing split")
    display(
        patient_summary.loc[
            patient_summary["diagnosis"].eq(disease),
            ["patient_id", "current_split", "recordings", "cycles"],
        ].sort_values(["current_split", "patient_id"])
    )


In [ ]:

# Audit disease labels and their existing numeric encoding.
# No labels are changed in this cell.

print("Available manifest columns:")
print(train_df.columns.tolist())

for split_name, df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    print(f"\n{'=' * 12} {split_name.upper()} {'=' * 12}")

    for col in ["diagnosis", "disease_label"]:
        if col in df.columns:
            print(f"\n{col} mapping:")
            display(
                df.groupby(col, dropna=False)
                .agg(
                    cycles=("audio_path", "size"),
                    patients=("patient_id", "nunique"),
                )
                .sort_index()
            )

# Define the five intended classes explicitly.
FIVE_CLASSES = [
    "COPD",
    "Healthy",
    "URTI",
    "Bronchiectasis",
    "Pneumonia",
]

print("\nProposed five-class order:")
for idx, name in enumerate(FIVE_CLASSES):
    print(f"{idx}: {name}")

# Verify that every proposed class occurs in training.
train_classes = set(train_df["diagnosis"].dropna().astype(str))
missing_train_classes = set(FIVE_CLASSES) - train_classes

assert not missing_train_classes, (
    f"Classes absent from training: {sorted(missing_train_classes)}"
)

print("\nAll five proposed classes are present in training.")


In [ ]:

# Build five-class manifests with explicit, contiguous labels.
# Original CSV files remain unchanged.

FIVE_CLASSES = [
    "COPD",
    "Healthy",
    "URTI",
    "Bronchiectasis",
    "Pneumonia",
]

LABEL_MAP = {name: idx for idx, name in enumerate(FIVE_CLASSES)}

five_manifests = {}

for split_name, source_df in {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}.items():
    filtered = source_df[
        source_df["diagnosis"].isin(FIVE_CLASSES)
    ].copy()

    # Remap using the disease name, not the original numeric label.
    filtered["original_disease_label"] = filtered["disease_label"]
    filtered["label"] = filtered["diagnosis"].map(LABEL_MAP).astype(int)

    # Verify that the original numeric labels agree with disease names.
    expected_original_labels = {
        "COPD": 3,
        "Healthy": 4,
        "URTI": 7,
        "Bronchiectasis": 1,
        "Pneumonia": 6,
    }
    for disease, original_label in expected_original_labels.items():
        rows = filtered[filtered["diagnosis"] == disease]
        assert (rows["original_disease_label"] == original_label).all(), (
            f"Unexpected original label for {disease} in {split_name}"
        )

    assert filtered["label"].isin(range(len(FIVE_CLASSES))).all()
    assert filtered["audio_path"].notna().all()
    assert filtered["audio_path"].duplicated().sum() == 0

    five_manifests[split_name] = filtered.reset_index(drop=True)

    print(
        f"{split_name:10s} | "
        f"cycles={len(filtered):4d} | "
        f"patients={filtered['patient_id'].nunique():2d} | "
        f"labels={sorted(filtered['label'].unique().tolist())}"
    )

print("\nFive-class mapping:")
display(pd.DataFrame({
    "new_label": range(len(FIVE_CLASSES)),
    "disease": FIVE_CLASSES,
    "original_disease_label": [3, 4, 7, 1, 6],
}))

print("\nClass support after filtering and remapping:")
for split_name, df in five_manifests.items():
    print(f"\n{split_name.upper()}")
    display(
        df.groupby(["label", "diagnosis"])
        .agg(
            cycles=("audio_path", "size"),
            recordings=("recording_id", "nunique"),
            patients=("patient_id", "nunique"),
        )
        .reset_index()
    )

# Verify no patient occurs in more than one filtered split.
patient_sets = {
    name: set(df["patient_id"].astype(str))
    for name, df in five_manifests.items()
}

assert not (patient_sets["train"] & patient_sets["validation"])
assert not (patient_sets["train"] & patient_sets["test"])
assert not (patient_sets["validation"] & patient_sets["test"])

print("\nPASS: label remapping, paths, and patient separation verified.")


In [ ]:

# Inspect the original lightweight CNN notebook for feature-cache paths.
# This cell only reads notebook code; it does not modify any files.

import nbformat

source_nb = nbformat.read(SOURCE_NOTEBOOK, as_version=4)

keywords = [
    "cache",
    "logmel",
    "mfcc",
    "chroma",
    "feature",
    "processed_path",
    "torch.save",
    "np.save",
    "normal",
]

for cell_idx, cell in enumerate(source_nb.cells):
    if cell.cell_type != "code":
        continue

    code = cell.source
    matched_lines = [
        line for line in code.splitlines()
        if any(word in line.lower() for word in keywords)
    ]

    if matched_lines:
        print(f"\n{'=' * 15} SOURCE CELL {cell_idx} {'=' * 15}")
        print("\n".join(matched_lines[:35]))


In [ ]:

# Inspect existing CNN cache/config files without modifying them.

BASELINE_DIR = ROOT / "data/processed/deep_learning/cnn_baseline"
MULTIFEATURE_DIR = ROOT / "data/processed/deep_learning/multifeature_cnn"

possible_dirs = [
    BASELINE_DIR,
    MULTIFEATURE_DIR,
    ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn",
]

for folder in possible_dirs:
    print(f"\n{'=' * 12} {folder.relative_to(ROOT)} {'=' * 12}")
    if not folder.exists():
        print("Directory does not exist.")
        continue

    for path in sorted(folder.rglob("*")):
        if path.is_file() and (
            path.suffix.lower() in {".npz", ".json", ".pt", ".pth"}
            or "feature_cache" in str(path).lower()
        ):
            print(f"{path.relative_to(ROOT)} | {path.stat().st_size / (1024**2):.2f} MiB")


In [ ]:

# Inspect the existing training feature cache safely.
# Do not load the model or change any files yet.

CACHE_DIR = (
    ROOT
    / "data/processed/deep_learning/lightweight_multifeature_cnn/feature_cache"
)
TRAIN_CACHE_PATH = CACHE_DIR / "train_features.npz"

assert TRAIN_CACHE_PATH.exists(), (
    f"Training cache not found: {TRAIN_CACHE_PATH}"
)

with np.load(TRAIN_CACHE_PATH, allow_pickle=False) as cache:
    print("Cache file:", TRAIN_CACHE_PATH)
    print("Stored keys:", cache.files)

    for key in cache.files:
        arr = cache[key]
        print(
            f"{key:20s} | "
            f"shape={arr.shape} | "
            f"dtype={arr.dtype} | "
            f"size={arr.nbytes / (1024**2):.2f} MiB"
        )

        if arr.ndim == 1 and arr.size <= 10:
            print("  values:", arr.tolist())

        if arr.ndim == 1 and arr.dtype.kind in {"U", "S"}:
            print("  first entries:", arr[:3].tolist())


In [ ]:

# Verify cache alignment against the original training manifest.
# This reads the trusted local cache; it does not modify any files.

with np.load(TRAIN_CACHE_PATH, allow_pickle=True) as cache:
    cached_paths = cache["paths"]
    cached_labels = cache["labels"]

print("Cached paths shape:", cached_paths.shape)
print("Cached labels shape:", cached_labels.shape)
print("Manifest rows:", len(train_df))

print("\nFirst 3 cached paths:")
for path in cached_paths[:3]:
    print(repr(path))

print("\nFirst 3 manifest paths:")
for path in train_df["audio_path"].head(3):
    print(repr(path))

# Compare path strings exactly first.
manifest_paths = train_df["audio_path"].astype(str).to_numpy()
cache_paths = np.asarray(cached_paths).astype(str)

print("\nExact path-order match:",
      np.array_equal(cache_paths, manifest_paths))

print("Cached labels match original manifest:",
      np.array_equal(cached_labels, train_df["disease_label"].to_numpy()))

# Show examples if paths differ.
if not np.array_equal(cache_paths, manifest_paths):
    print("\nFirst differing path positions:")
    mismatch = np.flatnonzero(cache_paths != manifest_paths)
    for idx in mismatch[:5]:
        print(
            f"Index {idx}:\n"
            f"  cache:    {cache_paths[idx]}\n"
            f"  manifest: {manifest_paths[idx]}"
        )


In [ ]:

# Create filtered five-class feature data in memory.
# Original cache files and manifests remain unchanged.

CACHE_FILES = {
    "train": CACHE_DIR / "train_features.npz",
    "validation": CACHE_DIR / "validation_features.npz",
    "test": CACHE_DIR / "test_features.npz",
}

five_feature_data = {}

for split_name, manifest in five_manifests.items():
    cache_path = CACHE_FILES[split_name]

    with np.load(cache_path, allow_pickle=True) as cache:
        cached_paths = np.asarray(cache["paths"]).astype(str)
        cached_labels = cache["labels"]
        logmel = cache["logmel"]
        mfcc = cache["mfcc"]
        chroma = cache["chroma"]

    manifest_paths = manifest["audio_path"].astype(str).to_numpy()

    # Check alignment against the original manifest before filtering.
    original_manifest = splits[split_name]
    assert np.array_equal(
        cached_paths,
        original_manifest["audio_path"].astype(str).to_numpy()
    ), f"{split_name}: cache path order does not match manifest"

    assert np.array_equal(
        cached_labels,
        original_manifest["disease_label"].to_numpy()
    ), f"{split_name}: cache labels do not match manifest"

    # Filter original cache rows in the same order as five_manifests.
    path_to_index = {path: idx for idx, path in enumerate(cached_paths)}
    selected_indices = np.array(
        [path_to_index[path] for path in manifest_paths],
        dtype=np.int64,
    )

    labels = manifest["label"].to_numpy(dtype=np.int64)

    data = {
        "logmel": logmel[selected_indices].astype(np.float32, copy=False),
        "mfcc": mfcc[selected_indices].astype(np.float32, copy=False),
        "chroma": chroma[selected_indices].astype(np.float32, copy=False),
        "labels": labels,
        "paths": manifest_paths,
    }

    assert len(data["labels"]) == len(manifest)
    assert np.array_equal(data["paths"], manifest_paths)
    assert np.array_equal(
        data["labels"],
        manifest["diagnosis"].map(LABEL_MAP).to_numpy(dtype=np.int64)
    )

    for feature_name in ["logmel", "mfcc", "chroma"]:
        assert data[feature_name].shape[0] == len(manifest)
        assert np.isfinite(data[feature_name]).all(), (
            f"{split_name}: non-finite values in {feature_name}"
        )

    five_feature_data[split_name] = data

    print(
        f"{split_name:10s} | "
        f"samples={len(labels):4d} | "
        f"labels={np.bincount(labels, minlength=5).tolist()} | "
        f"features aligned and finite"
    )

print("\nPASS: all five-class feature arrays are aligned and valid.")
print("Feature shapes:")
for name in ["logmel", "mfcc", "chroma"]:
    print(f"{name:8s}: {five_feature_data['train'][name].shape}")


In [ ]:

# Inspect saved configuration from the original lightweight multi-feature CNN.
# Read-only: no files are modified.

import json

CONFIG_PATHS = [
    ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn/feature_config.json",
    ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn/normalization_stats.json",
    ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn/training_config.json",
    ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn/architecture.json",
]

for path in CONFIG_PATHS:
    print(f"\n{'=' * 15} {path.name} {'=' * 15}")

    if not path.exists():
        print("Not found.")
        continue

    with open(path, "r") as f:
        config = json.load(f)

    print(json.dumps(config, indent=2))


In [ ]:

# Recalculate normalization from five-class training data only.
# This updates in-memory arrays, not the original cache files.

normalization_5class = {}

for feature_name in ["logmel", "mfcc", "chroma"]:
    train_array = five_feature_data["train"][feature_name]

    # Use float64 for stable statistics, then store normalized data as float32.
    mean = float(train_array.mean(dtype=np.float64))
    std = float(train_array.std(dtype=np.float64))

    assert np.isfinite(mean)
    assert np.isfinite(std) and std > 0, (
        f"Invalid standard deviation for {feature_name}: {std}"
    )

    normalization_5class[feature_name] = {
        "mean": mean,
        "std": std,
    }

    for split_name in ["train", "validation", "test"]:
        arr = five_feature_data[split_name][feature_name]
        five_feature_data[split_name][feature_name] = (
            (arr - mean) / std
        ).astype(np.float32)

    print(
        f"{feature_name:8s} | "
        f"mean={mean:.6f} | std={std:.6f}"
    )

# Verify the normalized training features are approximately zero-mean,
# unit-standard-deviation, and all splits remain finite.
for feature_name in ["logmel", "mfcc", "chroma"]:
    arr = five_feature_data["train"][feature_name]
    print(
        f"Train {feature_name:8s} | "
        f"mean={arr.mean():.5f} | std={arr.std():.5f} | "
        f"finite={np.isfinite(arr).all()}"
    )

for split_name in ["validation", "test"]:
    for feature_name in ["logmel", "mfcc", "chroma"]:
        assert np.isfinite(
            five_feature_data[split_name][feature_name]
        ).all()

print("\nPASS: five-class training-only normalization completed.")


In [ ]:

# Diagnose whether the existing feature cache was already normalized.
# Read-only: reloads the original cached training arrays for inspection.

with np.load(TRAIN_CACHE_PATH, allow_pickle=True) as cache:
    original_cached_features = {
        name: cache[name].astype(np.float32, copy=True)
        for name in ["logmel", "mfcc", "chroma"]
    }

with open(
    ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn/normalization_stats.json",
    "r",
) as f:
    original_stats = json.load(f)

print("Original cache statistics vs saved normalization parameters:\n")

for name, arr in original_cached_features.items():
    print(f"{name}:")
    print(
        f"  cached mean/std: "
        f"{arr.mean(dtype=np.float64):.6f} / "
        f"{arr.std(dtype=np.float64):.6f}"
    )
    print(
        f"  saved mean/std:  "
        f"{original_stats[name]['mean']:.6f} / "
        f"{original_stats[name]['std']:.6f}"
    )


In [ ]:

# Restore five-class arrays from the original, already-normalized cache.
# This replaces only the in-memory five_feature_data variables.

five_feature_data = {}

for split_name, manifest in five_manifests.items():
    with np.load(CACHE_FILES[split_name], allow_pickle=True) as cache:
        cached_paths = np.asarray(cache["paths"]).astype(str)
        cached_labels = cache["labels"]

        original_manifest = splits[split_name]
        original_paths = original_manifest["audio_path"].astype(str).to_numpy()

        assert np.array_equal(cached_paths, original_paths), (
            f"{split_name}: cache paths are not aligned with the manifest"
        )
        assert np.array_equal(
            cached_labels,
            original_manifest["disease_label"].to_numpy()
        ), f"{split_name}: cached labels do not match the original manifest"

        selected_indices = np.flatnonzero(
            original_manifest["diagnosis"].isin(FIVE_CLASSES).to_numpy()
        )

        labels = manifest["diagnosis"].map(LABEL_MAP).to_numpy(dtype=np.int64)
        selected_paths = cached_paths[selected_indices]

        assert np.array_equal(
            selected_paths,
            manifest["audio_path"].astype(str).to_numpy()
        ), f"{split_name}: filtered cache order mismatch"

        five_feature_data[split_name] = {
            "logmel": cache["logmel"][selected_indices].astype(np.float32),
            "mfcc": cache["mfcc"][selected_indices].astype(np.float32),
            "chroma": cache["chroma"][selected_indices].astype(np.float32),
            "labels": labels,
            "paths": selected_paths,
        }

    print(
        f"{split_name:10s} | "
        f"samples={len(labels):4d} | "
        f"labels={np.bincount(labels, minlength=5).tolist()}"
    )

print("\nRestored feature statistics:")
for feature_name in ["logmel", "mfcc", "chroma"]:
    arr = five_feature_data["train"][feature_name]
    print(
        f"{feature_name:8s} | "
        f"mean={arr.mean():.6f} | std={arr.std():.6f}"
    )

print("\nPASS: original normalized features restored and labels aligned.")


In [ ]:

# Define the five-class version of the lightweight multi-feature CNN.
# Architecture follows the existing lightweight CNN design.

import torch
import torch.nn as nn

NUM_CLASSES = 5
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Training device:", DEVICE)


class CompactFeatureBranch(nn.Module):
    def __init__(self, input_channels):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Depthwise separable convolution: 8 -> 16 channels
            nn.Conv2d(
                8, 8, kernel_size=3, padding=1, groups=8
            ),
            nn.Conv2d(8, 16, kernel_size=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Depthwise separable convolution: 16 -> 24 channels
            nn.Conv2d(
                16, 16, kernel_size=3, padding=1, groups=16
            ),
            nn.Conv2d(16, 24, kernel_size=1),
            nn.BatchNorm2d(24),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(24, 16),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
        )

    def forward(self, x):
        return self.projection(self.features(x))


class LightweightMultiFeatureCNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()

        self.logmel_branch = CompactFeatureBranch(1)
        self.mfcc_branch = CompactFeatureBranch(1)
        self.chroma_branch = CompactFeatureBranch(1)

        self.classifier = nn.Sequential(
            nn.Linear(48, 24),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(24, num_classes),
        )

    def forward(self, logmel, mfcc, chroma):
        features = torch.cat(
            [
                self.logmel_branch(logmel),
                self.mfcc_branch(mfcc),
                self.chroma_branch(chroma),
            ],
            dim=1,
        )
        return self.classifier(features)


model = LightweightMultiFeatureCNN(NUM_CLASSES).to(DEVICE)

# Verify the model's input/output shapes and parameter count.
with torch.no_grad():
    dummy_logmel = torch.zeros(2, 1, 64, 157, device=DEVICE)
    dummy_mfcc = torch.zeros(2, 1, 13, 157, device=DEVICE)
    dummy_chroma = torch.zeros(2, 1, 12, 157, device=DEVICE)

    output = model(dummy_logmel, dummy_mfcc, dummy_chroma)

parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Output shape:", tuple(output.shape))
print("Trainable parameters:", parameter_count)
print("Expected output classes:", NUM_CLASSES)

assert output.shape == (2, 5)
assert parameter_count < 10000

print("\nPASS: five-class CNN architecture verified.")


In [ ]:

from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 64
NUM_WORKERS = 0


class CachedRespiratoryDataset(Dataset):
    def __init__(self, data, augment=False):
        self.data = data
        self.augment = augment

    def __len__(self):
        return len(self.data["labels"])

    def __getitem__(self, idx):
        logmel = self.data["logmel"][idx].copy()
        mfcc = self.data["mfcc"][idx].copy()
        chroma = self.data["chroma"][idx].copy()
        label = int(self.data["labels"][idx])

        if self.augment:
            # Apply the same temporal shift to all feature branches.
            shift = np.random.randint(-8, 9)
            logmel = np.roll(logmel, shift, axis=-1)
            mfcc = np.roll(mfcc, shift, axis=-1)
            chroma = np.roll(chroma, shift, axis=-1)

            logmel += np.random.normal(
                0, 0.015, logmel.shape
            ).astype(np.float32)
            mfcc += np.random.normal(
                0, 0.010, mfcc.shape
            ).astype(np.float32)
            chroma += np.random.normal(
                0, 0.010, chroma.shape
            ).astype(np.float32)

        return (
            torch.from_numpy(logmel.astype(np.float32)),
            torch.from_numpy(mfcc.astype(np.float32)),
            torch.from_numpy(chroma.astype(np.float32)),
            torch.tensor(label, dtype=torch.long),
        )


train_dataset = CachedRespiratoryDataset(
    five_feature_data["train"], augment=True
)
val_dataset = CachedRespiratoryDataset(
    five_feature_data["validation"], augment=False
)
test_dataset = CachedRespiratoryDataset(
    five_feature_data["test"], augment=False
)

generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    generator=generator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

# Square-root inverse-frequency weights, normalized to mean 1.
class_counts = np.bincount(
    five_feature_data["train"]["labels"], minlength=NUM_CLASSES
)
class_weights_np = 1.0 / np.sqrt(class_counts.astype(np.float64))
class_weights_np /= class_weights_np.mean()

class_weights = torch.tensor(
    class_weights_np, dtype=torch.float32, device=DEVICE
)

print("Class order:", FIVE_CLASSES)
print("Training counts:", class_counts.tolist())
print("Class weights:", np.round(class_weights_np, 4).tolist())
print("Batches — train:", len(train_loader),
      "| validation:", len(val_loader),
      "| test:", len(test_loader))

# Check one batch without updating model weights.
batch = next(iter(train_loader))
x_logmel, x_mfcc, x_chroma, y = batch

print("\nBatch shapes:")
print("Log-Mel:", tuple(x_logmel.shape))
print("MFCC:", tuple(x_mfcc.shape))
print("Chroma:", tuple(x_chroma.shape))
print("Labels:", tuple(y.shape))
print("Labels in batch:", sorted(y.unique().tolist()))

assert x_logmel.shape[1:] == (1, 64, 157)
assert x_mfcc.shape[1:] == (1, 13, 157)
assert x_chroma.shape[1:] == (1, 12, 157)
assert y.min().item() >= 0 and y.max().item() < NUM_CLASSES

print("\nPASS: datasets, loaders, and class weights are ready.")


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
import copy
import random

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

# Training configuration
MAX_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

best_val_macro_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0
best_model_state = None
history = []

def evaluate_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for logmel, mfcc, chroma, labels in loader:
            logmel = logmel.to(device)
            mfcc = mfcc.to(device)
            chroma = chroma.to(device)
            labels = labels.to(device)

            logits = model(logmel, mfcc, chroma)
            loss = criterion(logits, labels)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            predictions = logits.argmax(dim=1)
            all_targets.extend(labels.cpu().numpy().tolist())
            all_predictions.extend(predictions.cpu().numpy().tolist())

    metrics = {
        "loss": total_loss / total_samples,
        "accuracy": accuracy_score(all_targets, all_predictions),
        "balanced_accuracy": balanced_accuracy_score(
            all_targets, all_predictions
        ),
        "macro_f1": f1_score(
            all_targets,
            all_predictions,
            labels=list(range(NUM_CLASSES)),
            average="macro",
            zero_division=0,
        ),
        "targets": np.asarray(all_targets),
        "predictions": np.asarray(all_predictions),
    }
    return metrics

print("Loss: weighted cross-entropy")
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Scheduler: ReduceLROnPlateau, maximizing validation macro-F1")
print("Early-stopping patience:", EARLY_STOPPING_PATIENCE)
print("Maximum epochs:", MAX_EPOCHS)

print("\nPASS: training components are configured.")


In [ ]:

# Train the five-class lightweight multi-feature CNN.
# Saves the best checkpoint and epoch history in a separate experiment folder.

import time

EXPERIMENT_5CLASS_DIR = (
    ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn_5class"
)
EXPERIMENT_5CLASS_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_5CLASS_PATH = (
    EXPERIMENT_5CLASS_DIR / "best_lightweight_multifeature_cnn_5class.pt"
)

# Reset tracking before the run.
best_val_macro_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0
best_model_state = None
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.time()
    model.train()

    running_loss = 0.0
    seen = 0

    for logmel, mfcc, chroma, labels in train_loader:
        logmel = logmel.to(DEVICE)
        mfcc = mfcc.to(DEVICE)
        chroma = chroma.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(logmel, mfcc, chroma)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        seen += batch_size

    train_loss = running_loss / seen
    val_metrics = evaluate_model(model, val_loader, criterion, DEVICE)

    scheduler.step(val_metrics["macro_f1"])
    current_lr = optimizer.param_groups[0]["lr"]

    record = {
        "epoch": epoch,
        "train_loss": float(train_loss),
        "val_loss": float(val_metrics["loss"]),
        "val_accuracy": float(val_metrics["accuracy"]),
        "val_balanced_accuracy": float(val_metrics["balanced_accuracy"]),
        "val_macro_f1": float(val_metrics["macro_f1"]),
        "learning_rate": float(current_lr),
    }
    history.append(record)

    improved = val_metrics["macro_f1"] > best_val_macro_f1

    if improved:
        best_val_macro_f1 = float(val_metrics["macro_f1"])
        best_epoch = epoch
        epochs_without_improvement = 0
        best_model_state = copy.deepcopy(model.state_dict())

        torch.save(
            {
                "model_state_dict": best_model_state,
                "num_classes": NUM_CLASSES,
                "class_names": FIVE_CLASSES,
                "best_epoch": best_epoch,
                "best_val_macro_f1": best_val_macro_f1,
                "seed": SEED,
            },
            BEST_MODEL_5CLASS_PATH,
        )
        status = "BEST"
    else:
        epochs_without_improvement += 1
        status = ""

    elapsed = time.time() - epoch_start

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
        f"val_bal_acc={val_metrics['balanced_accuracy']:.4f} | "
        f"lr={current_lr:.2e} | "
        f"{elapsed:.1f}s {status}"
    )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"\nEarly stopping at epoch {epoch}; "
            f"best epoch was {best_epoch}."
        )
        break

# Restore the best validation-selected weights.
if best_model_state is None:
    raise RuntimeError("Training did not produce a valid checkpoint.")

model.load_state_dict(best_model_state)

history_df = pd.DataFrame(history)
history_path = EXPERIMENT_5CLASS_DIR / "training_history.csv"
history_df.to_csv(history_path, index=False)

print("\nTraining complete.")
print("Best epoch:", best_epoch)
print(f"Best validation macro-F1: {best_val_macro_f1:.4f}")
print("Checkpoint:", BEST_MODEL_5CLASS_PATH)
print("History:", history_path)


In [ ]:

from sklearn.metrics import classification_report, confusion_matrix

# Load the best saved five-class checkpoint.
checkpoint = torch.load(
    BEST_MODEL_5CLASS_PATH,
    map_location=DEVICE,
    weights_only=True,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

val_metrics = evaluate_model(model, val_loader, criterion, DEVICE)

print("Best checkpoint epoch:", checkpoint["best_epoch"])
print(f"Best validation macro-F1: {checkpoint['best_val_macro_f1']:.4f}")
print(f"Validation accuracy: {val_metrics['accuracy']:.4f}")
print(f"Validation balanced accuracy: {val_metrics['balanced_accuracy']:.4f}")
print(f"Validation macro-F1: {val_metrics['macro_f1']:.4f}")

print("\nClassification report:")
print(
    classification_report(
        val_metrics["targets"],
        val_metrics["predictions"],
        labels=list(range(NUM_CLASSES)),
        target_names=FIVE_CLASSES,
        digits=4,
        zero_division=0,
    )
)

print("Confusion matrix (rows=true class, columns=predicted class):")
display(
    pd.DataFrame(
        confusion_matrix(
            val_metrics["targets"],
            val_metrics["predictions"],
            labels=list(range(NUM_CLASSES)),
        ),
        index=[f"True {name}" for name in FIVE_CLASSES],
        columns=[f"Pred {name}" for name in FIVE_CLASSES],
    )
)


In [ ]:

# Quantify validation prediction bias and per-class recall.

targets = val_metrics["targets"]
predictions = val_metrics["predictions"]

print("Validation class counts:")
print("True:     ", np.bincount(targets, minlength=NUM_CLASSES).tolist())
print("Predicted:", np.bincount(predictions, minlength=NUM_CLASSES).tolist())

print("\nRecall by class:")
for class_id, disease in enumerate(FIVE_CLASSES):
    mask = targets == class_id
    recall = (predictions[mask] == class_id).mean() if mask.any() else np.nan
    predicted_count = int((predictions == class_id).sum())

    print(
        f"{disease:16s} | "
        f"true={int(mask.sum()):3d} | "
        f"predicted={predicted_count:3d} | "
        f"recall={recall:.4f}"
    )

print("\nPASS: validation prediction distribution inspected.")


In [ ]:

# Experiment 2: calculate effective-number class weights.
# This cell does not train or modify the baseline model.

BETA = 0.999

counts = np.bincount(
    five_feature_data["train"]["labels"],
    minlength=NUM_CLASSES,
).astype(np.float64)

effective_numbers = 1.0 - np.power(BETA, counts)
effective_weights = (1.0 - BETA) / effective_numbers

# Normalize weights to mean 1.
effective_weights /= effective_weights.mean()

print("Class weighting comparison:\n")
print(f"{'Disease':18s} {'Count':>7s} {'Current':>10s} {'Effective':>10s}")

for i, disease in enumerate(FIVE_CLASSES):
    print(
        f"{disease:18s} "
        f"{int(counts[i]):7d} "
        f"{class_weights_np[i]:10.4f} "
        f"{effective_weights[i]:10.4f}"
    )

print("\nEffective-number beta:", BETA)
print("Mean effective weight:", effective_weights.mean())

assert np.isfinite(effective_weights).all()
assert np.all(effective_weights > 0)
print("\nPASS: effective-number weights calculated.")


In [ ]:

# Experiment 2: train with effective-number class weights.
# Keeps the baseline model and checkpoint untouched.

EXPERIMENT_2_DIR = (
    ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn_5class_effective_weights"
)
EXPERIMENT_2_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_EXP2_PATH = (
    EXPERIMENT_2_DIR / "best_lightweight_multifeature_cnn_5class.pt"
)

# Fresh model and optimizer: no weights carried over from baseline training.
torch.manual_seed(SEED)
np.random.seed(SEED)

model_exp2 = LightweightMultiFeatureCNN(NUM_CLASSES).to(DEVICE)

effective_class_weights = torch.tensor(
    effective_weights,
    dtype=torch.float32,
    device=DEVICE,
)

criterion_exp2 = nn.CrossEntropyLoss(weight=effective_class_weights)

optimizer_exp2 = torch.optim.AdamW(
    model_exp2.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler_exp2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_exp2,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

best_f1_exp2 = -1.0
best_epoch_exp2 = 0
patience_exp2 = 0
best_state_exp2 = None
history_exp2 = []

for epoch in range(1, MAX_EPOCHS + 1):
    start = time.time()
    model_exp2.train()

    total_loss = 0.0
    total_samples = 0

    for logmel, mfcc, chroma, labels in train_loader:
        logmel = logmel.to(DEVICE)
        mfcc = mfcc.to(DEVICE)
        chroma = chroma.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer_exp2.zero_grad(set_to_none=True)
        logits = model_exp2(logmel, mfcc, chroma)
        loss = criterion_exp2(logits, labels)
        loss.backward()
        optimizer_exp2.step()

        n = labels.size(0)
        total_loss += loss.item() * n
        total_samples += n

    train_loss = total_loss / total_samples
    val = evaluate_model(model_exp2, val_loader, criterion_exp2, DEVICE)

    scheduler_exp2.step(val["macro_f1"])
    lr = optimizer_exp2.param_groups[0]["lr"]

    history_exp2.append({
        "epoch": epoch,
        "train_loss": float(train_loss),
        "val_loss": float(val["loss"]),
        "val_macro_f1": float(val["macro_f1"]),
        "val_balanced_accuracy": float(val["balanced_accuracy"]),
        "val_accuracy": float(val["accuracy"]),
        "learning_rate": float(lr),
    })

    if val["macro_f1"] > best_f1_exp2:
        best_f1_exp2 = float(val["macro_f1"])
        best_epoch_exp2 = epoch
        patience_exp2 = 0
        best_state_exp2 = copy.deepcopy(model_exp2.state_dict())

        torch.save({
            "model_state_dict": best_state_exp2,
            "num_classes": NUM_CLASSES,
            "class_names": FIVE_CLASSES,
            "best_epoch": best_epoch_exp2,
            "best_val_macro_f1": best_f1_exp2,
            "seed": SEED,
            "class_weights": effective_weights.tolist(),
            "weighting_method": "effective_number",
            "beta": BETA,
        }, BEST_MODEL_EXP2_PATH)

        status = "BEST"
    else:
        patience_exp2 += 1
        status = ""

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_macro_f1={val['macro_f1']:.4f} | "
        f"val_bal_acc={val['balanced_accuracy']:.4f} | "
        f"lr={lr:.2e} | "
        f"{time.time() - start:.1f}s {status}"
    )

    if patience_exp2 >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

history_exp2_df = pd.DataFrame(history_exp2)
history_exp2_df.to_csv(EXPERIMENT_2_DIR / "training_history.csv", index=False)

print("\nEffective-weighting experiment complete.")
print("Best epoch:", best_epoch_exp2)
print(f"Best validation macro-F1: {best_f1_exp2:.4f}")
print("Baseline validation macro-F1: 0.4505")
print("Checkpoint:", BEST_MODEL_EXP2_PATH)


In [ ]:

# Evaluate Experiment 2 on validation data only.
# Does not train the model or touch the test set.

model_exp2.load_state_dict(
    torch.load(
        BEST_MODEL_EXP2_PATH,
        map_location=DEVICE,
        weights_only=True,
    )["model_state_dict"]
)

model_exp2.eval()

val_exp2 = evaluate_model(
    model_exp2,
    val_loader,
    criterion_exp2,
    DEVICE,
)

print(f"Validation accuracy: {val_exp2['accuracy']:.4f}")
print(f"Validation balanced accuracy: {val_exp2['balanced_accuracy']:.4f}")
print(f"Validation macro-F1: {val_exp2['macro_f1']:.4f}")

print("\nClassification report:")
print(
    classification_report(
        val_exp2["targets"],
        val_exp2["predictions"],
        labels=list(range(NUM_CLASSES)),
        target_names=FIVE_CLASSES,
        digits=4,
        zero_division=0,
    )
)

print("Confusion matrix (rows=true, columns=predicted):")
display(
    pd.DataFrame(
        confusion_matrix(
            val_exp2["targets"],
            val_exp2["predictions"],
            labels=list(range(NUM_CLASSES)),
        ),
        index=[f"True {name}" for name in FIVE_CLASSES],
        columns=[f"Pred {name}" for name in FIVE_CLASSES],
    )
)


In [ ]:

from torch.utils.data import WeightedRandomSampler

# Experiment 3: class-balanced sampling with ordinary cross-entropy.
# This cell only prepares and audits the sampler; it does not train.

train_labels = five_feature_data["train"]["labels"]
train_counts = np.bincount(train_labels, minlength=NUM_CLASSES)

# Inverse-frequency sample weights.
sample_weights = 1.0 / train_counts[train_labels]

sampler_generator = torch.Generator().manual_seed(SEED)

weighted_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_labels),
    replacement=True,
    generator=sampler_generator,
)

# Audit the expected class distribution using the sampler's actual draws.
sampled_indices = list(iter(weighted_sampler))
sampled_labels = train_labels[np.asarray(sampled_indices, dtype=np.int64)]
sampled_counts = np.bincount(sampled_labels, minlength=NUM_CLASSES)

print("Original training counts:")
for name, count in zip(FIVE_CLASSES, train_counts):
    print(f"{name:16s}: {count:4d}")

print("\nCounts in one simulated sampled epoch:")
for name, count in zip(FIVE_CLASSES, sampled_counts):
    print(f"{name:16s}: {count:4d}")

print("\nOriginal proportions:",
      np.round(train_counts / train_counts.sum(), 3).tolist())
print("Sampled proportions:",
      np.round(sampled_counts / sampled_counts.sum(), 3).tolist())

assert len(sampled_indices) == len(train_labels)
assert sampled_counts.sum() == len(train_labels)

print("\nPASS: weighted sampler configured and audited.")


In [ ]:

# Experiment 3: weighted sampling + ordinary cross-entropy.
# Setup only; training happens in the next cell.

EXPERIMENT_3_DIR = (
    ROOT / "data/processed/deep_learning/"
    "lightweight_multifeature_cnn_5class_weighted_sampler"
)
EXPERIMENT_3_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_EXP3_PATH = (
    EXPERIMENT_3_DIR / "best_lightweight_multifeature_cnn_5class.pt"
)
HISTORY_EXP3_PATH = EXPERIMENT_3_DIR / "training_history.csv"

# Reproducible model initialization.
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model_exp3 = LightweightMultiFeatureCNN(NUM_CLASSES).to(DEVICE)

# Use ordinary cross-entropy; balancing is handled by the sampler.
criterion_exp3 = nn.CrossEntropyLoss()

# Preserve the existing dataset and all validation/test loaders.
train_loader_exp3 = DataLoader(
    train_dataset,
    batch_size=64,
    sampler=weighted_sampler,
    num_workers=0,
    pin_memory=False,
)

print(f"Device: {DEVICE}")
print(f"Train samples per epoch: {len(train_loader_exp3.sampler)}")
print(f"Training batches per epoch: {len(train_loader_exp3)}")
print(f"Model parameters: {sum(p.numel() for p in model_exp3.parameters() if p.requires_grad):,}")
print(f"Loss: {criterion_exp3}")
print(f"Checkpoint directory: {EXPERIMENT_3_DIR}")

assert len(train_loader_exp3.dataset) == len(train_dataset)
assert train_loader_exp3.sampler is weighted_sampler
assert not isinstance(criterion_exp3, nn.CrossEntropyLoss) or criterion_exp3.weight is None

print("PASS: Experiment 3 setup is ready.")


In [ ]:

# Experiment 3: weighted random sampling + ordinary cross-entropy.
# Saves the best model based on validation macro-F1.

MAX_EPOCHS = 60
PATIENCE = 10

optimizer_exp3 = torch.optim.AdamW(
    model_exp3.parameters(),
    lr=0.001,
    weight_decay=0.0001,
)

scheduler_exp3 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_exp3,
    mode="min",
    factor=0.5,
    patience=3,
)

best_macro_f1_exp3 = -1.0
best_epoch_exp3 = 0
epochs_without_improvement_exp3 = 0
history_exp3 = []

for epoch in range(1, MAX_EPOCHS + 1):
    model_exp3.train()
    running_loss = 0.0
    total_samples = 0

    for batch in train_loader_exp3:
        logmel, mfcc, chroma, labels = [
            tensor.to(DEVICE) for tensor in batch
        ]

        optimizer_exp3.zero_grad(set_to_none=True)

        logits = model_exp3(logmel, mfcc, chroma)
        loss = criterion_exp3(logits, labels)
        loss.backward()
        optimizer_exp3.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

    train_loss = running_loss / total_samples

    val_metrics = evaluate_model(
        model_exp3,
        val_loader,
        criterion_exp3,
        DEVICE,
    )

    scheduler_exp3.step(val_metrics["loss"])

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "learning_rate": optimizer_exp3.param_groups[0]["lr"],
    }
    history_exp3.append(row)

    improved = val_metrics["macro_f1"] > best_macro_f1_exp3

    if improved:
        best_macro_f1_exp3 = val_metrics["macro_f1"]
        best_epoch_exp3 = epoch
        epochs_without_improvement_exp3 = 0

        torch.save(
            {
                "model_state_dict": model_exp3.state_dict(),
                "epoch": epoch,
                "val_macro_f1": best_macro_f1_exp3,
                "num_classes": NUM_CLASSES,
                "classes": FIVE_CLASSES,
                "seed": SEED,
                "experiment": "weighted_random_sampler",
            },
            BEST_MODEL_EXP3_PATH,
        )
    else:
        epochs_without_improvement_exp3 += 1

    if epoch == 1 or improved or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
            f"val_bal_acc={val_metrics['balanced_accuracy']:.4f}"
        )

    if epochs_without_improvement_exp3 >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

pd.DataFrame(history_exp3).to_csv(HISTORY_EXP3_PATH, index=False)

print("\nExperiment 3 training complete.")
print(f"Best epoch: {best_epoch_exp3}")
print(f"Best validation macro-F1: {best_macro_f1_exp3:.4f}")
print(f"Checkpoint saved: {BEST_MODEL_EXP3_PATH}")
print(f"History saved: {HISTORY_EXP3_PATH}")


In [ ]:

# Evaluate Experiment 3's best checkpoint on validation only.
# The test set remains untouched.

checkpoint_exp3 = torch.load(
    BEST_MODEL_EXP3_PATH,
    map_location=DEVICE,
    weights_only=True,
)

model_exp3.load_state_dict(checkpoint_exp3["model_state_dict"])
model_exp3.eval()

val_exp3 = evaluate_model(
    model_exp3,
    val_loader,
    criterion_exp3,
    DEVICE,
)

print(f"Best checkpoint epoch: {checkpoint_exp3['epoch']}")
print(f"Validation accuracy: {val_exp3['accuracy']:.4f}")
print(f"Validation balanced accuracy: {val_exp3['balanced_accuracy']:.4f}")
print(f"Validation macro-F1: {val_exp3['macro_f1']:.4f}")

print("\nClassification report:")
print(
    classification_report(
        val_exp3["targets"],
        val_exp3["predictions"],
        labels=list(range(NUM_CLASSES)),
        target_names=FIVE_CLASSES,
        digits=4,
        zero_division=0,
    )
)

print("Confusion matrix (rows=true, columns=predicted):")
display(
    pd.DataFrame(
        confusion_matrix(
            val_exp3["targets"],
            val_exp3["predictions"],
            labels=list(range(NUM_CLASSES)),
        ),
        index=[f"True {name}" for name in FIVE_CLASSES],
        columns=[f"Pred {name}" for name in FIVE_CLASSES],
    )
)


In [ ]:

# Save validation results for the three experiments.
# These are reported checkpoint-selection metrics, not test results.

comparison_exp123 = pd.DataFrame([
    {
        "experiment": "Exp1_baseline",
        "best_epoch": 27,
        "val_accuracy": 0.7779,
        "val_balanced_accuracy": 0.4431,
        "val_macro_f1": 0.4505,
        "sampling": "Natural",
        "loss": "Inverse-square-root class weights",
    },
    {
        "experiment": "Exp2_effective_weights",
        "best_epoch": 26,
        "val_accuracy": 0.7080,
        "val_balanced_accuracy": 0.4100,
        "val_macro_f1": 0.4201,
        "sampling": "Natural",
        "loss": "Effective-number class weights",
    },
    {
        "experiment": "Exp3_weighted_sampler",
        "best_epoch": 8,
        "val_accuracy": 0.6719,
        "val_balanced_accuracy": 0.4538,
        "val_macro_f1": 0.3779,
        "sampling": "Inverse-frequency weighted sampler",
        "loss": "Ordinary cross-entropy",
    },
])

comparison_path = EXPERIMENT_3_DIR / "experiment_comparison.csv"
comparison_exp123.to_csv(comparison_path, index=False)

display(comparison_exp123)
print(f"\nComparison saved to: {comparison_path}")


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma

        if alpha is None:
            self.alpha = None
        else:
            self.register_buffer(
                "alpha",
                torch.as_tensor(alpha, dtype=torch.float32)
            )

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=1)
        log_pt = log_probs.gather(
            1, targets.unsqueeze(1)
        ).squeeze(1)

        pt = log_pt.exp()
        loss = -((1.0 - pt) ** self.gamma) * log_pt

        if self.alpha is not None:
            loss = loss * self.alpha[targets]

        return loss.mean()


# Smoke test: check the loss is finite and supports backpropagation.
focal_test = FocalLoss(gamma=2.0).to(DEVICE)
test_logits = torch.randn(8, NUM_CLASSES, device=DEVICE, requires_grad=True)
test_targets = torch.randint(0, NUM_CLASSES, (8,), device=DEVICE)

test_loss = focal_test(test_logits, test_targets)
test_loss.backward()

print(f"Focal loss test value: {test_loss.item():.4f}")
print(f"Loss is finite: {torch.isfinite(test_loss).item()}")
print(f"Gradients available: {test_logits.grad is not None}")
print(f"Focal loss gamma: {focal_test.gamma}")

assert torch.isfinite(test_loss).item()
assert test_logits.grad is not None

print("PASS: focal loss implementation is working.")


In [ ]:

# Experiment 4: focal loss, natural sampling.
# Setup only; training will be in the next cell.

EXPERIMENT_4_DIR = (
    ROOT / "data/processed/deep_learning/"
    "lightweight_multifeature_cnn_5class_focal_loss"
)
EXPERIMENT_4_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_EXP4_PATH = (
    EXPERIMENT_4_DIR / "best_lightweight_multifeature_cnn_5class.pt"
)
HISTORY_EXP4_PATH = EXPERIMENT_4_DIR / "training_history.csv"

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model_exp4 = LightweightMultiFeatureCNN(NUM_CLASSES).to(DEVICE)

criterion_exp4 = FocalLoss(gamma=2.0).to(DEVICE)

# Natural class distribution: no weighted sampler.
train_loader_exp4 = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)

print(f"Device: {DEVICE}")
print(f"Training samples per epoch: {len(train_loader_exp4.dataset)}")
print(f"Training batches per epoch: {len(train_loader_exp4)}")
print(
    "Trainable parameters:",
    sum(p.numel() for p in model_exp4.parameters() if p.requires_grad),
)
print(f"Loss: FocalLoss(gamma={criterion_exp4.gamma})")
print(f"Sampler: {type(train_loader_exp4.sampler).__name__}")
print(f"Checkpoint directory: {EXPERIMENT_4_DIR}")

assert len(train_loader_exp4.dataset) == len(train_dataset)
assert isinstance(train_loader_exp4.sampler, torch.utils.data.RandomSampler)

print("PASS: Experiment 4 setup is ready.")


In [ ]:

# Experiment 4: focal loss, natural sampling.
# Select the best checkpoint using validation macro-F1.

MAX_EPOCHS = 60
PATIENCE = 10

optimizer_exp4 = torch.optim.AdamW(
    model_exp4.parameters(),
    lr=0.001,
    weight_decay=0.0001,
)

scheduler_exp4 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_exp4,
    mode="min",
    factor=0.5,
    patience=3,
)

best_macro_f1_exp4 = -1.0
best_epoch_exp4 = 0
epochs_without_improvement_exp4 = 0
history_exp4 = []

for epoch in range(1, MAX_EPOCHS + 1):
    model_exp4.train()
    running_loss = 0.0
    total_samples = 0

    for batch in train_loader_exp4:
        logmel, mfcc, chroma, labels = [
            tensor.to(DEVICE) for tensor in batch
        ]

        optimizer_exp4.zero_grad(set_to_none=True)

        logits = model_exp4(logmel, mfcc, chroma)
        loss = criterion_exp4(logits, labels)
        loss.backward()
        optimizer_exp4.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

    train_loss = running_loss / total_samples

    val_metrics = evaluate_model(
        model_exp4,
        val_loader,
        criterion_exp4,
        DEVICE,
    )

    scheduler_exp4.step(val_metrics["loss"])

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "learning_rate": optimizer_exp4.param_groups[0]["lr"],
    }
    history_exp4.append(row)

    improved = val_metrics["macro_f1"] > best_macro_f1_exp4

    if improved:
        best_macro_f1_exp4 = val_metrics["macro_f1"]
        best_epoch_exp4 = epoch
        epochs_without_improvement_exp4 = 0

        torch.save(
            {
                "model_state_dict": model_exp4.state_dict(),
                "epoch": epoch,
                "val_macro_f1": best_macro_f1_exp4,
                "num_classes": NUM_CLASSES,
                "classes": FIVE_CLASSES,
                "seed": SEED,
                "experiment": "focal_loss_gamma_2",
            },
            BEST_MODEL_EXP4_PATH,
        )
    else:
        epochs_without_improvement_exp4 += 1

    if epoch == 1 or improved or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
            f"val_bal_acc={val_metrics['balanced_accuracy']:.4f}"
        )

    if epochs_without_improvement_exp4 >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

pd.DataFrame(history_exp4).to_csv(HISTORY_EXP4_PATH, index=False)

print("\nExperiment 4 training complete.")
print(f"Best epoch: {best_epoch_exp4}")
print(f"Best validation macro-F1: {best_macro_f1_exp4:.4f}")
print(f"Checkpoint saved: {BEST_MODEL_EXP4_PATH}")
print(f"History saved: {HISTORY_EXP4_PATH}")


In [ ]:

# Evaluate Experiment 4's best checkpoint on validation only.
# The test set remains untouched.

checkpoint_exp4 = torch.load(
    BEST_MODEL_EXP4_PATH,
    map_location=DEVICE,
    weights_only=True,
)

model_exp4.load_state_dict(checkpoint_exp4["model_state_dict"])
model_exp4.eval()

val_exp4 = evaluate_model(
    model_exp4,
    val_loader,
    criterion_exp4,
    DEVICE,
)

print(f"Best checkpoint epoch: {checkpoint_exp4['epoch']}")
print(f"Validation accuracy: {val_exp4['accuracy']:.4f}")
print(f"Validation balanced accuracy: {val_exp4['balanced_accuracy']:.4f}")
print(f"Validation macro-F1: {val_exp4['macro_f1']:.4f}")

print("\nClassification report:")
print(
    classification_report(
        val_exp4["targets"],
        val_exp4["predictions"],
        labels=list(range(NUM_CLASSES)),
        target_names=FIVE_CLASSES,
        digits=4,
        zero_division=0,
    )
)

print("Confusion matrix (rows=true, columns=predicted):")
display(
    pd.DataFrame(
        confusion_matrix(
            val_exp4["targets"],
            val_exp4["predictions"],
            labels=list(range(NUM_CLASSES)),
        ),
        index=[f"True {name}" for name in FIVE_CLASSES],
        columns=[f"Pred {name}" for name in FIVE_CLASSES],
    )
)


In [ ]:

import json

print("Feature cache directory:")
print(CACHE_DIR)

print("\nCandidate normalization/statistics files:")
for path in sorted(CACHE_DIR.parent.rglob("*")):
    if path.is_file() and any(
        word in path.name.lower()
        for word in ("norm", "stat", "scaler", "mean", "std")
    ):
        print(path)

print("\nExisting normalization statistics object:")
if "normalization_stats" in globals():
    print(type(normalization_stats).__name__)
    print(normalization_stats)
elif "normalization_stats_path" in globals():
    print("Path:", normalization_stats_path)
    if normalization_stats_path.exists():
        print(normalization_stats_path.read_text()[:5000])
    else:
        print("The recorded path does not exist.")
else:
    print("No variable named normalization_stats was found.")


In [ ]:

import json

NORMALIZATION_STATS_PATH = CACHE_DIR.parent / "normalization_stats.json"

print("Statistics file exists:", NORMALIZATION_STATS_PATH.exists())
print("Statistics file:", NORMALIZATION_STATS_PATH)

with open(NORMALIZATION_STATS_PATH, "r") as f:
    normalization_stats_exp = json.load(f)

print("\nNormalization statistics:")
print(json.dumps(normalization_stats_exp, indent=2))


In [ ]:

# Audit cache metadata and preprocessing references.
# This cell does not alter any files.

print("=== Cached feature files ===")
for split_name, cache_path in CACHE_FILES.items():
    print(f"\n{split_name}: {cache_path}")
    with np.load(cache_path, allow_pickle=True) as cache:
        for key in cache.files:
            arr = cache[key]
            print(
                f"  {key}: shape={arr.shape}, dtype={arr.dtype}, "
                f"mean={float(np.mean(arr)):.5f}, "
                f"std={float(np.std(arr)):.5f}"
                if arr.dtype.kind in "fiu"
                else f"  {key}: shape={arr.shape}, dtype={arr.dtype}"
            )

print("\n=== Candidate preprocessing scripts ===")
for search_root in [
    ROOT / "Notebooks 2",
    ROOT / "scripts",
    ROOT / "src",
]:
    if search_root.exists():
        for path in sorted(search_root.rglob("*.py")):
            try:
                content = path.read_text(errors="ignore").lower()
            except OSError:
                continue

            if any(term in content for term in [
                "normalization_stats",
                "logmel_mean",
                "feature_cache",
                "standard deviation",
                "standard deviation",
                "mfcc_mean",
            ]):
                print(path)


In [ ]:

# Search notebook code cells for feature extraction and normalization.
# Read-only audit; no training or file modification.

import json

notebook_candidates = [
    ROOT / "Notebooks 2" / "11_Lightweight_MultiFeature_CNN_5Class_Experiments.ipynb",
    ROOT / "Notebooks 2" / "10_Lightweight_MultiFeature_CNN.ipynb",
]

for notebook_path in notebook_candidates:
    print(f"\nNotebook: {notebook_path}")
    print("Exists:", notebook_path.exists())

    if not notebook_path.exists():
        continue

    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook_json = json.load(f)

    terms = (
        "normalization_stats",
        "feature_cache",
        "np.savez",
        "logmel",
        "mfcc",
        "chroma",
        "mean(",
        "std(",
        "normalize",
    )

    for cell_idx, cell in enumerate(notebook_json.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        source_lower = source.lower()

        if any(term in source_lower for term in terms):
            print(f"\n--- Code cell {cell_idx} ---")
            print(source[:4500])


In [ ]:

# Experiment 5 preprocessing:
# Re-normalize the filtered features using five-class training data only.
# The original .npz cache files remain unchanged.

normalization_5class = {}

for feature_name in ["logmel", "mfcc", "chroma"]:
    # Calculate statistics exclusively from the five-class training subset.
    train_array = five_feature_data["train"][feature_name]

    mean = float(train_array.mean(dtype=np.float64))
    std = float(train_array.std(dtype=np.float64))

    assert np.isfinite(mean)
    assert np.isfinite(std) and std > 0

    normalization_5class[feature_name] = {
        "mean": mean,
        "std": std,
    }

    # Apply the same training-derived statistics to every split.
    for split_name in ["train", "validation", "test"]:
        arr = five_feature_data[split_name][feature_name]
        five_feature_data[split_name][feature_name] = (
            (arr - mean) / std
        ).astype(np.float32)

    print(
        f"{feature_name:8s} | "
        f"training mean={mean:.6f} | "
        f"training std={std:.6f}"
    )

print("\nVerification:")
for feature_name in ["logmel", "mfcc", "chroma"]:
    arr = five_feature_data["train"][feature_name]
    print(
        f"Train {feature_name:8s} | "
        f"mean={arr.mean():.5f} | "
        f"std={arr.std():.5f} | "
        f"finite={np.isfinite(arr).all()}"
    )

for split_name in ["validation", "test"]:
    for feature_name in ["logmel", "mfcc", "chroma"]:
        assert np.isfinite(
            five_feature_data[split_name][feature_name]
        ).all()

print("\nPASS: five-class training-only normalization applied.")


In [ ]:

# Experiment 5: rebuild datasets and loaders after five-class normalization.
# No model training in this cell.

from torch.utils.data import DataLoader

BATCH_SIZE = 64
NUM_WORKERS = 0

train_dataset_exp5 = CachedRespiratoryDataset(
    five_feature_data["train"], augment=True
)
val_dataset_exp5 = CachedRespiratoryDataset(
    five_feature_data["validation"], augment=False
)
test_dataset_exp5 = CachedRespiratoryDataset(
    five_feature_data["test"], augment=False
)

generator_exp5 = torch.Generator().manual_seed(SEED)

train_loader_exp5 = DataLoader(
    train_dataset_exp5,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    generator=generator_exp5,
)
val_loader_exp5 = DataLoader(
    val_dataset_exp5,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)
test_loader_exp5 = DataLoader(
    test_dataset_exp5,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

# Preserve the same baseline weighting strategy.
counts_exp5 = np.bincount(
    five_feature_data["train"]["labels"],
    minlength=NUM_CLASSES,
).astype(np.float64)

weights_exp5 = 1.0 / np.sqrt(counts_exp5)
weights_exp5 /= weights_exp5.mean()

class_weights_exp5 = torch.tensor(
    weights_exp5, dtype=torch.float32, device=DEVICE
)

print("Training samples:", len(train_dataset_exp5))
print("Validation samples:", len(val_dataset_exp5))
print("Test samples:", len(test_dataset_exp5))
print("Training batches:", len(train_loader_exp5))
print("Class order:", FIVE_CLASSES)
print("Training counts:", counts_exp5.astype(int).tolist())
print("Class weights:", np.round(weights_exp5, 4).tolist())

# Confirm that the refreshed datasets use the normalized arrays.
for feature_name in ["logmel", "mfcc", "chroma"]:
    assert np.isfinite(
        five_feature_data["train"][feature_name]
    ).all()

assert len(train_dataset_exp5) == len(five_manifests["train"])
assert len(val_dataset_exp5) == len(five_manifests["validation"])
assert len(test_dataset_exp5) == len(five_manifests["test"])

print("\nPASS: Experiment 5 datasets and loaders are ready.")


In [ ]:

# Experiment 5: baseline architecture with five-class training-only normalization.
# Setup only; training happens in the next cell.

EXPERIMENT_5_DIR = (
    ROOT / "data/processed/deep_learning/"
    "lightweight_multifeature_cnn_5class_trainonly_norm"
)
EXPERIMENT_5_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_EXP5_PATH = (
    EXPERIMENT_5_DIR / "best_lightweight_multifeature_cnn_5class.pt"
)
HISTORY_EXP5_PATH = EXPERIMENT_5_DIR / "training_history.csv"
NORM_STATS_EXP5_PATH = EXPERIMENT_5_DIR / "normalization_stats.json"

# Save the training-derived statistics for reproducibility.
with open(NORM_STATS_EXP5_PATH, "w") as f:
    json.dump(normalization_5class, f, indent=2)

torch.manual_seed(SEED)
np.random.seed(SEED)

model_exp5 = LightweightMultiFeatureCNN(NUM_CLASSES).to(DEVICE)

criterion_exp5 = nn.CrossEntropyLoss(weight=class_weights_exp5)

optimizer_exp5 = torch.optim.AdamW(
    model_exp5.parameters(),
    lr=0.001,
    weight_decay=0.0001,
)

scheduler_exp5 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_exp5,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

print("Device:", DEVICE)
print("Trainable parameters:", sum(
    p.numel() for p in model_exp5.parameters() if p.requires_grad
))
print("Loss: weighted cross-entropy")
print("Learning rate:", optimizer_exp5.param_groups[0]["lr"])
print("Weight decay:", 0.0001)
print("Scheduler: ReduceLROnPlateau, maximizing validation macro-F1")
print("Checkpoint:", BEST_MODEL_EXP5_PATH)
print("Normalization statistics:", NORM_STATS_EXP5_PATH)

assert sum(
    p.numel() for p in model_exp5.parameters() if p.requires_grad
) == 5405
assert criterion_exp5.weight is not None

print("\nPASS: Experiment 5 model and optimizer are ready.")


In [ ]:

# Experiment 5: baseline training with five-class training-only normalization.

import time
import copy

MAX_EPOCHS = 60
PATIENCE = 10

best_macro_f1_exp5 = -1.0
best_epoch_exp5 = 0
epochs_without_improvement_exp5 = 0
best_state_exp5 = None
history_exp5 = []

for epoch in range(1, MAX_EPOCHS + 1):
    start_time = time.time()
    model_exp5.train()

    running_loss = 0.0
    total_samples = 0

    for logmel, mfcc, chroma, labels in train_loader_exp5:
        logmel = logmel.to(DEVICE)
        mfcc = mfcc.to(DEVICE)
        chroma = chroma.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer_exp5.zero_grad(set_to_none=True)

        logits = model_exp5(logmel, mfcc, chroma)
        loss = criterion_exp5(logits, labels)
        loss.backward()
        optimizer_exp5.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

    train_loss = running_loss / total_samples

    val_exp5_epoch = evaluate_model(
        model_exp5,
        val_loader_exp5,
        criterion_exp5,
        DEVICE,
    )

    scheduler_exp5.step(val_exp5_epoch["macro_f1"])
    current_lr = optimizer_exp5.param_groups[0]["lr"]

    history_exp5.append({
        "epoch": epoch,
        "train_loss": float(train_loss),
        "val_loss": float(val_exp5_epoch["loss"]),
        "val_accuracy": float(val_exp5_epoch["accuracy"]),
        "val_balanced_accuracy": float(
            val_exp5_epoch["balanced_accuracy"]
        ),
        "val_macro_f1": float(val_exp5_epoch["macro_f1"]),
        "learning_rate": float(current_lr),
    })

    improved = val_exp5_epoch["macro_f1"] > best_macro_f1_exp5

    if improved:
        best_macro_f1_exp5 = float(val_exp5_epoch["macro_f1"])
        best_epoch_exp5 = epoch
        epochs_without_improvement_exp5 = 0
        best_state_exp5 = copy.deepcopy(model_exp5.state_dict())

        torch.save({
            "model_state_dict": best_state_exp5,
            "num_classes": NUM_CLASSES,
            "class_names": FIVE_CLASSES,
            "best_epoch": best_epoch_exp5,
            "best_val_macro_f1": best_macro_f1_exp5,
            "seed": SEED,
            "normalization": normalization_5class,
            "experiment": "baseline_trainonly_normalization",
        }, BEST_MODEL_EXP5_PATH)

        status = "BEST"
    else:
        epochs_without_improvement_exp5 += 1
        status = ""

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_exp5_epoch['loss']:.4f} | "
        f"val_macro_f1={val_exp5_epoch['macro_f1']:.4f} | "
        f"val_bal_acc={val_exp5_epoch['balanced_accuracy']:.4f} | "
        f"lr={current_lr:.2e} | "
        f"{time.time() - start_time:.1f}s {status}"
    )

    if epochs_without_improvement_exp5 >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

if best_state_exp5 is None:
    raise RuntimeError("No valid best checkpoint was produced.")

model_exp5.load_state_dict(best_state_exp5)

pd.DataFrame(history_exp5).to_csv(
    HISTORY_EXP5_PATH,
    index=False,
)

print("\nExperiment 5 training complete.")
print("Best epoch:", best_epoch_exp5)
print(f"Best validation macro-F1: {best_macro_f1_exp5:.4f}")
print("Checkpoint:", BEST_MODEL_EXP5_PATH)
print("History:", HISTORY_EXP5_PATH)


In [ ]:

# Evaluate Experiment 5's best checkpoint on validation only.
# The test set remains untouched.

checkpoint_exp5 = torch.load(
    BEST_MODEL_EXP5_PATH,
    map_location=DEVICE,
    weights_only=True,
)

model_exp5.load_state_dict(checkpoint_exp5["model_state_dict"])
model_exp5.eval()

val_exp5 = evaluate_model(
    model_exp5,
    val_loader_exp5,
    criterion_exp5,
    DEVICE,
)

print(f"Best checkpoint epoch: {checkpoint_exp5['best_epoch']}")
print(f"Validation accuracy: {val_exp5['accuracy']:.4f}")
print(f"Validation balanced accuracy: {val_exp5['balanced_accuracy']:.4f}")
print(f"Validation macro-F1: {val_exp5['macro_f1']:.4f}")

print("\nClassification report:")
print(
    classification_report(
        val_exp5["targets"],
        val_exp5["predictions"],
        labels=list(range(NUM_CLASSES)),
        target_names=FIVE_CLASSES,
        digits=4,
        zero_division=0,
    )
)

print("Confusion matrix (rows=true, columns=predicted):")
display(
    pd.DataFrame(
        confusion_matrix(
            val_exp5["targets"],
            val_exp5["predictions"],
            labels=list(range(NUM_CLASSES)),
        ),
        index=[f"True {name}" for name in FIVE_CLASSES],
        columns=[f"Pred {name}" for name in FIVE_CLASSES],
    )
)


In [ ]:

# Record the five experiments in one reproducible comparison table.
# Uses validation results only; does not access the test loader.

comparison_exp12345 = pd.DataFrame([
    {
        "experiment": "Exp1_baseline",
        "best_epoch": 27,
        "val_accuracy": 0.7779,
        "val_balanced_accuracy": 0.4431,
        "val_macro_f1": 0.4505,
        "preprocessing": "Original cached features",
        "loss": "Inverse-square-root weighted cross-entropy",
    },
    {
        "experiment": "Exp2_effective_weights",
        "best_epoch": 26,
        "val_accuracy": 0.7080,
        "val_balanced_accuracy": 0.4100,
        "val_macro_f1": 0.4201,
        "preprocessing": "Original cached features",
        "loss": "Effective-number weighted cross-entropy",
    },
    {
        "experiment": "Exp3_weighted_sampler",
        "best_epoch": 8,
        "val_accuracy": 0.6719,
        "val_balanced_accuracy": 0.4538,
        "val_macro_f1": 0.3779,
        "preprocessing": "Original cached features",
        "loss": "Ordinary cross-entropy with weighted sampler",
    },
    {
        "experiment": "Exp4_focal_loss",
        "best_epoch": 15,
        "val_accuracy": 0.7621,
        "val_balanced_accuracy": 0.4274,
        "val_macro_f1": 0.4040,
        "preprocessing": "Original cached features",
        "loss": "Focal loss, gamma=2",
    },
    {
        "experiment": "Exp5_trainonly_norm",
        "best_epoch": 21,
        "val_accuracy": 0.7351,
        "val_balanced_accuracy": 0.5194,
        "val_macro_f1": 0.4548,
        "preprocessing": "Five-class training-only normalization",
        "loss": "Inverse-square-root weighted cross-entropy",
    },
])

comparison_path_exp5 = EXPERIMENT_5_DIR / "experiment_comparison.csv"
comparison_exp12345.to_csv(comparison_path_exp5, index=False)

display(
    comparison_exp12345.sort_values(
        "val_macro_f1", ascending=False
    ).reset_index(drop=True)
)

print("\nSaved comparison:", comparison_path_exp5)


In [ ]:

# Inventory existing model artifacts across the repository.
# Read-only: this cell does not load models, train, or evaluate the test set.

from pathlib import Path

MODEL_SUFFIXES = {".pt", ".pth", ".pkl", ".joblib", ".onnx", ".keras", ".h5"}
CONFIG_SUFFIXES = {".json", ".yaml", ".yml"}

print("Repository root:", ROOT)

print("\n=== Saved model artifacts ===")
model_files = []

for path in sorted(ROOT.rglob("*")):
    if not path.is_file():
        continue

    if any(part.startswith(".") for part in path.parts):
        continue

    if path.suffix.lower() in MODEL_SUFFIXES:
        # Avoid listing duplicate checkpoint files in obvious environments.
        if any(part in {"venv", ".venv", "node_modules"} for part in path.parts):
            continue
        model_files.append(path)

for path in model_files:
    print(f"{path.relative_to(ROOT)} | {path.stat().st_size / (1024**2):.2f} MiB")

print(f"\nTotal saved model artifacts: {len(model_files)}")

print("\n=== Model-related configuration and result files ===")
keywords = (
    "model", "metrics", "result", "score", "report",
    "feature", "scaler", "transform", "label", "config",
    "experiment", "comparison",
)

config_files = []

for path in sorted(ROOT.rglob("*")):
    if not path.is_file():
        continue
    if path.suffix.lower() not in CONFIG_SUFFIXES and path.suffix.lower() != ".csv":
        continue
    if any(part in {"venv", ".venv", "node_modules", ".git"} for part in path.parts):
        continue
    if any(word in path.name.lower() for word in keywords):
        config_files.append(path)

for path in config_files:
    print(path.relative_to(ROOT))

print(f"\nTotal related configuration/result files: {len(config_files)}")


In [ ]:

# Step 2: Inspect saved PyTorch checkpoint metadata.
# Read-only audit. No model training or test evaluation.

from pathlib import Path
import torch

checkpoint_paths = [
    p for p in sorted(ROOT.rglob("*.pt"))
    if p.is_file()
    and not any(
        part in {"venv", ".venv", "node_modules", ".git"}
        for part in p.parts
    )
]

print(f"Found {len(checkpoint_paths)} PyTorch checkpoints.\n")

for path in checkpoint_paths:
    print("=" * 90)
    print("FILE:", path.relative_to(ROOT))
    print(f"SIZE: {path.stat().st_size / (1024**2):.3f} MiB")

    try:
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=True,
        )

        if isinstance(checkpoint, dict):
            print("TOP-LEVEL KEYS:", list(checkpoint.keys())[:30])

            for key in [
                "class_names",
                "classes",
                "num_classes",
                "best_epoch",
                "best_val_macro_f1",
                "seed",
                "experiment",
                "input_dim",
                "feature_names",
            ]:
                if key in checkpoint:
                    value = checkpoint[key]
                    if isinstance(value, (list, tuple)) and len(value) > 20:
                        value = list(value[:20]) + ["..."]
                    print(f"{key}: {value}")

            state = checkpoint.get(
                "model_state_dict",
                checkpoint.get("state_dict"),
            )

            if isinstance(state, dict):
                print("STATE DICT TENSOR COUNT:", len(state))
                print(
                    "FIRST TENSOR SHAPES:",
                    [
                        (k, tuple(v.shape))
                        for k, v in list(state.items())[:5]
                        if torch.is_tensor(v)
                    ],
                )
        else:
            print("OBJECT TYPE:", type(checkpoint).__name__)
            print("Metadata not stored as a dictionary.")

    except Exception as exc:
        print("SAFE LOAD NOT AVAILABLE:", type(exc).__name__, str(exc)[:250])


In [ ]:

# Step 3: Inspect traditional ML artifacts and feature compatibility.
# Read-only audit; no training or test evaluation.

import joblib
from pathlib import Path

ml_dirs = [
    ROOT / "models" / "traditional_ml",
    ROOT / "models" / "traditional_ml_patient_disjoint",
]

for folder in ml_dirs:
    print("\n" + "=" * 80)
    print("DIRECTORY:", folder.relative_to(ROOT))

    for filename in [
        "label_encoder.joblib",
        "scaler.joblib",
        "logistic_regression.joblib",
        "random_forest.joblib",
        "svm.joblib",
    ]:
        path = folder / filename

        if not path.exists():
            print(f"\n{filename}: NOT FOUND")
            continue

        try:
            obj = joblib.load(path)
            print(f"\n{filename}: {type(obj).__name__}")

            if hasattr(obj, "classes_"):
                print("Classes:", obj.classes_.tolist())

            if hasattr(obj, "n_features_in_"):
                print("Input feature count:", obj.n_features_in_)

            if hasattr(obj, "feature_names_in_"):
                print("Feature names:", obj.feature_names_in_.tolist())

            if hasattr(obj, "mean_"):
                print("Scaler mean shape:", obj.mean_.shape)

            if hasattr(obj, "scale_"):
                print("Scaler scale shape:", obj.scale_.shape)

            if hasattr(obj, "n_estimators"):
                print("Number of trees:", obj.n_estimators)

        except Exception as exc:
            print("Inspection error:", type(exc).__name__, str(exc)[:200])

print("\n=== XGBoost artifact ===")
xgb_path = (
    ROOT / "reports" / "research_audit" / "common_patient_split"
    / "xgboost" / "xgboost_disease_model.joblib"
)
print("Exists:", xgb_path.exists())
if xgb_path.exists():
    print("Size (MiB):", round(xgb_path.stat().st_size / (1024**2), 3))


In [ ]:

# Step 4: Locate existing ML feature tables and evaluation reports.
# Read-only file inventory; does not load or evaluate models.

from pathlib import Path

search_roots = [
    ROOT / "data" / "processed",
    ROOT / "reports",
    ROOT / "models",
]

terms = (
    "test", "evaluation", "metrics", "classification_report",
    "feature", "prediction", "patient_disjoint", "xgboost",
)

extensions = {".csv", ".parquet", ".json", ".txt", ".joblib"}

found = set()

for base in search_roots:
    if not base.exists():
        continue

    for path in base.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix.lower() not in extensions:
            continue
        if any(
            part in {"venv", ".venv", "node_modules", ".git"}
            for part in path.parts
        ):
            continue
        if any(term in path.name.lower() for term in terms):
            found.add(path)

for path in sorted(found):
    print(path.relative_to(ROOT))

print(f"\nTotal matching files: {len(found)}")


In [ ]:

# Step 5: Audit existing common-patient test comparison reports.
# Read-only; no model training or new predictions.

import pandas as pd

audit_dir = (
    ROOT / "reports" / "research_audit"
    / "common_patient_split" / "final_comparison"
)

report_names = [
    "deduplicated_common_test_leaderboard.csv",
    "complete_common_test_leaderboard.csv",
    "overall_metrics_with_xgboost.csv",
    "complete_common_test_alignment_checks.csv",
    "recomputed_additional_model_metrics.csv",
]

for filename in report_names:
    path = audit_dir / filename

    print("\n" + "=" * 90)
    print("REPORT:", filename)

    if not path.exists():
        print("NOT FOUND")
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.to_string(index=False, max_rows=40))


In [ ]:

# Step 6: Audit five-class test-split compatibility.
# Read-only: this does not generate predictions or evaluate models.

import pandas as pd

test_manifest_path = (
    ROOT / "data" / "processed" / "deep_learning"
    / "cnn_baseline" / "test_cnn_manifest.csv"
)

five_test = five_manifests["test"].copy()
common_test = pd.read_csv(test_manifest_path)

print("=== Five-class test manifest ===")
print("Rows:", len(five_test))
print("Columns:", five_test.columns.tolist())
print(five_test.head(3).to_string(index=False))

print("\n=== Existing common test manifest ===")
print("Rows:", len(common_test))
print("Columns:", common_test.columns.tolist())
print(common_test.head(3).to_string(index=False))

print("\n=== Path-column candidates ===")
for name, df in [
    ("five_class_test", five_test),
    ("common_test", common_test),
]:
    candidates = [
        c for c in df.columns
        if any(k in c.lower() for k in ["path", "file", "recording", "wav"])
    ]
    print(name, candidates)


In [ ]:

# Step 7: Verify exact five-class/common-test alignment.
# Read-only; does not evaluate any model.

import pandas as pd

five_test = five_manifests["test"].copy()
common_test = pd.read_csv(test_manifest_path)

# Normalize paths only for comparison; do not modify source manifests.
five_paths = set(five_test["audio_path"].astype(str))
common_paths = set(common_test["audio_path"].astype(str))

print("=== Exact audio-path overlap ===")
print("Five-class unique paths:", len(five_paths))
print("Common-test unique paths:", len(common_paths))
print("Paths shared:", len(five_paths & common_paths))
print("Five-class paths missing from common test:", len(five_paths - common_paths))
print("Common-test paths excluded from five-class set:", len(common_paths - five_paths))

print("\n=== Five-class test label distribution ===")
print(
    five_test["diagnosis"]
    .value_counts()
    .reindex(FIVE_CLASSES, fill_value=0)
    .to_string()
)

print("\n=== Excluded classes in common test ===")
excluded = common_test[~common_test["audio_path"].astype(str).isin(five_paths)]
print("Excluded rows:", len(excluded))
print(excluded["diagnosis"].value_counts().to_string())

# Confirm patient separation across the canonical splits.
train_patients = set(five_manifests["train"]["patient_id"].astype(str))
val_patients = set(five_manifests["validation"]["patient_id"].astype(str))
test_patients = set(five_test["patient_id"].astype(str))

print("\n=== Patient-disjoint check ===")
print("Train/validation overlap:", len(train_patients & val_patients))
print("Train/test overlap:", len(train_patients & test_patients))
print("Validation/test overlap:", len(val_patients & test_patients))


In [ ]:

# Step 8: Check existing prediction-file compatibility with the five-class test set.
# Read-only; no new predictions or model evaluation.

import pandas as pd
from pathlib import Path

prediction_files = {
    "lightweight_multifeature_cnn":
        ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn/test_predictions.csv",
    "multifeature_cnn":
        ROOT / "data/processed/deep_learning/multifeature_cnn/multifeature_cnn_test_predictions.csv",
    "regularized_multifeature_cnn":
        ROOT / "models/deep_learning/regularized_multifeature_cnn/test_predictions.csv",
    "cnn_lstm":
        ROOT / "data/processed/deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv",
    "logistic_regression":
        ROOT / "data/processed/classical_ml/classical_ml_predictions.csv",
    "xgboost":
        ROOT / "reports/research_audit/common_patient_split/xgboost/xgboost_test_predictions.csv",
}

five_path_set = set(five_test["audio_path"].astype(str))

for model_name, path in prediction_files.items():
    print("\n" + "=" * 80)
    print(model_name)

    if not path.exists():
        print("Prediction file not found:", path.relative_to(ROOT))
        continue

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    path_cols = [
        c for c in df.columns
        if "audio_path" in c.lower() or c.lower() in {"path", "filepath", "file_path"}
    ]
    true_cols = [
        c for c in df.columns
        if any(k in c.lower() for k in ["true", "actual", "target", "y_true", "label"])
    ]

    print("Candidate path columns:", path_cols)
    print("Candidate true-label columns:", true_cols)

    if path_cols:
        model_paths = set(df[path_cols[0]].astype(str))
        print("Paths shared with five-class cohort:", len(model_paths & five_path_set))
        print("Five-class paths absent from predictions:", len(five_path_set - model_paths))
    else:
        print("Path-level alignment cannot be verified from this file alone.")

    print(df.head(2).to_string(index=False))


In [ ]:

# Step 9: Identify traditional ML test predictions and their split provenance.
# Read-only audit.

from pathlib import Path
import pandas as pd

candidate_paths = [
    ROOT / "data/processed/classical_ml/classical_ml_predictions.csv",
    ROOT / "reports/research_audit/common_patient_split/test_predictions.csv",
    ROOT / "reports/research_audit/common_patient_split/xgboost/xgboost_test_predictions.csv",
    ROOT / "reports/research_audit/common_patient_split/xgboost/xgboost_test_metrics.json",
]

for path in candidate_paths:
    print("\n" + "=" * 85)
    print(path.relative_to(ROOT))

    if not path.exists():
        print("NOT FOUND")
        continue

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())

        for col in ["official_split", "cnn_split", "split", "patient_id"]:
            if col in df.columns:
                print(f"{col} distribution:")
                print(df[col].value_counts(dropna=False).head(12).to_string())

        print(df.head(3).to_string(index=False))

    elif path.suffix.lower() == ".json":
        print(path.read_text()[:5000])


In [ ]:

# Step 10: Audit existing common-test predictions by model.
# Read-only: no model inference or training.

import pandas as pd

pred_path = (
    ROOT / "reports" / "research_audit"
    / "common_patient_split" / "test_predictions.csv"
)

pred_df = pd.read_csv(pred_path)
manifest_df = pd.read_csv(test_manifest_path)

print("=== Rows per model ===")
print(pred_df.groupby("model_id").size().sort_values(ascending=False).to_string())

print("\n=== Canonical test manifest ===")
print("Rows:", len(manifest_df))
print("Unique patient/recording/location groups:",
      manifest_df[["patient_id", "recording_id", "chest_location"]]
      .drop_duplicates().shape[0])

print("\n=== Truth-label sequence checks ===")
canonical_truth = manifest_df["disease_label"].astype(str).tolist()

for model_id, group in pred_df.groupby("model_id", sort=False):
    group = group.reset_index(drop=True)

    sequence_match = (
        len(group) == len(canonical_truth)
        and group["true_label"].astype(str).tolist() == canonical_truth
    )

    print(
        f"{model_id}: rows={len(group)}, "
        f"truth_sequence_matches={sequence_match}"
    )

print("\n=== Distinct model IDs ===")
print(sorted(pred_df["model_id"].dropna().unique().tolist()))


In [ ]:

# Step 11 (corrected): Recompute five-class metrics from saved predictions.
# No retraining or new inference.

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)

pred_path = (
    ROOT / "reports" / "research_audit"
    / "common_patient_split" / "test_predictions.csv"
)

pred_df = pd.read_csv(pred_path)
manifest_df = pd.read_csv(test_manifest_path)

FIVE_CLASSES = [
    "COPD", "Healthy", "URTI", "Bronchiectasis", "Pneumonia"
]
FIVE_LABELS = [3, 4, 7, 1, 6]

mask = manifest_df["diagnosis"].isin(FIVE_CLASSES).to_numpy()
five_manifest = manifest_df.loc[mask].reset_index(drop=True)
y_true = five_manifest["disease_label"].astype(int).to_numpy()

assert len(five_manifest) == 1542

summary_rows = []
per_class_rows = []

for model_id, group in pred_df.groupby("model_id", sort=True):
    group = group.reset_index(drop=True)

    assert len(group) == len(manifest_df)
    assert (
        group["true_label"].astype(int).tolist()
        == manifest_df["disease_label"].astype(int).tolist()
    )

    group_five = group.loc[mask].reset_index(drop=True)
    y_pred = group_five["predicted_label"].astype(int).to_numpy()

    summary_rows.append({
        "model": model_id,
        "test_cycles": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy_5class": recall_score(
            y_true, y_pred, labels=FIVE_LABELS,
            average="macro", zero_division=0
        ),
        "macro_f1_5class": f1_score(
            y_true, y_pred, labels=FIVE_LABELS,
            average="macro", zero_division=0
        ),
        "weighted_f1_5class": f1_score(
            y_true, y_pred, labels=FIVE_LABELS,
            average="weighted", zero_division=0
        ),
    })

    report = classification_report(
        y_true, y_pred,
        labels=FIVE_LABELS,
        target_names=FIVE_CLASSES,
        output_dict=True,
        zero_division=0,
    )

    for class_name in FIVE_CLASSES:
        per_class_rows.append({
            "model": model_id,
            "class": class_name,
            "precision": report[class_name]["precision"],
            "recall": report[class_name]["recall"],
            "f1": report[class_name]["f1-score"],
            "support": report[class_name]["support"],
        })

summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values("macro_f1_5class", ascending=False)
    .reset_index(drop=True)
)
per_class_df = pd.DataFrame(per_class_rows)

print("=== Five-class test leaderboard ===")
print(summary_df.round(4).to_string(index=False))

print("\n=== Per-class results ===")
print(per_class_df.round(4).to_string(index=False))


In [ ]:
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Load the selected Experiment 5 checkpoint
checkpoint = torch.load(
    BEST_MODEL_EXP5_PATH,
    map_location=device,
    weights_only=False,
)

model_exp5_test = LightweightMultiFeatureCNN(NUM_CLASSES).to(device)
model_exp5_test.load_state_dict(checkpoint["model_state_dict"])
model_exp5_test.eval()

# Evaluate once on the untouched test loader
test_loss, y_true_exp5, y_pred_exp5 = evaluate_model(
    model_exp5_test,
    test_loader_exp5,
    criterion,
    device,
)

y_true_exp5 = np.asarray(y_true_exp5, dtype=int)
y_pred_exp5 = np.asarray(y_pred_exp5, dtype=int)

print("=== Experiment 5: Five-class test results ===")
print("Checkpoint:", BEST_MODEL_EXP5_PATH)
print("Best validation macro-F1:", checkpoint.get("best_val_macro_f1"))
print("Test cycles:", len(y_true_exp5))
print("Test loss:", round(float(test_loss), 4))
print("Accuracy:", round(accuracy_score(y_true_exp5, y_pred_exp5), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_true_exp5, y_pred_exp5), 4),
)
print(
    "Macro-F1:",
    round(f1_score(y_true_exp5, y_pred_exp5, average="macro", zero_division=0), 4),
)

print("\n=== Per-class test report ===")
print(
    classification_report(
        y_true_exp5,
        y_pred_exp5,
        labels=list(range(NUM_CLASSES)),
        target_names=FIVE_CLASSES,
        zero_division=0,
        digits=4,
    )
)

print("=== Confusion matrix ===")
print(confusion_matrix(y_true_exp5, y_pred_exp5, labels=list(range(NUM_CLASSES))))

In [ ]:
import torch

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)
print("Experiment 5 checkpoint exists:", BEST_MODEL_EXP5_PATH.exists()
      if hasattr(BEST_MODEL_EXP5_PATH, "exists")
      else __import__("os").path.exists(BEST_MODEL_EXP5_PATH))

In [ ]:
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Load the selected Experiment 5 checkpoint
checkpoint = torch.load(
    BEST_MODEL_EXP5_PATH,
    map_location=device,
    weights_only=False,
)

model_exp5_test = LightweightMultiFeatureCNN(NUM_CLASSES).to(device)
model_exp5_test.load_state_dict(checkpoint["model_state_dict"])
model_exp5_test.eval()

# Evaluate on the test set
test_loss, y_true_exp5, y_pred_exp5 = evaluate_model(
    model_exp5_test,
    test_loader_exp5,
    criterion,
    device,
)

y_true_exp5 = np.asarray(y_true_exp5, dtype=int)
y_pred_exp5 = np.asarray(y_pred_exp5, dtype=int)

print("=== Experiment 5: Five-class test results ===")
print("Checkpoint:", BEST_MODEL_EXP5_PATH)
print("Best validation macro-F1:", checkpoint.get("best_val_macro_f1"))
print("Test cycles:", len(y_true_exp5))
print("Test loss:", round(float(test_loss), 4))
print("Accuracy:", round(accuracy_score(y_true_exp5, y_pred_exp5), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_true_exp5, y_pred_exp5), 4),
)
print(
    "Macro-F1:",
    round(f1_score(y_true_exp5, y_pred_exp5, average="macro", zero_division=0), 4),
)

print("\n=== Per-class test report ===")
print(
    classification_report(
        y_true_exp5,
        y_pred_exp5,
        labels=list(range(NUM_CLASSES)),
        target_names=FIVE_CLASSES,
        zero_division=0,
        digits=4,
    )
)

print("=== Confusion matrix ===")
print(confusion_matrix(y_true_exp5, y_pred_exp5, labels=list(range(NUM_CLASSES))))

In [ ]:
import inspect

print(inspect.getsource(evaluate_model))

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Evaluate the selected Experiment 5 model
test_metrics_exp5 = evaluate_model(
    model_exp5_test,
    test_loader_exp5,
    criterion,
    device,
)

y_true_exp5 = test_metrics_exp5["targets"]
y_pred_exp5 = test_metrics_exp5["predictions"]

print("=== Experiment 5: Five-class test results ===")
print("Checkpoint:", BEST_MODEL_EXP5_PATH)
print("Best validation macro-F1:", checkpoint.get("best_val_macro_f1"))
print("Test cycles:", len(y_true_exp5))
print("Test loss:", round(float(test_metrics_exp5["loss"]), 4))
print("Accuracy:", round(float(test_metrics_exp5["accuracy"]), 4))
print("Balanced accuracy:", round(float(test_metrics_exp5["balanced_accuracy"]), 4))
print("Macro-F1:", round(float(test_metrics_exp5["macro_f1"]), 4))

print("\n=== Per-class test report ===")
print(
    classification_report(
        y_true_exp5,
        y_pred_exp5,
        labels=list(range(NUM_CLASSES)),
        target_names=FIVE_CLASSES,
        zero_division=0,
        digits=4,
    )
)

print("=== Confusion matrix ===")
print(
    confusion_matrix(
        y_true_exp5,
        y_pred_exp5,
        labels=list(range(NUM_CLASSES)),
    )
)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

MANIFEST_DIR = (
    PROJECT_ROOT
    / "data/processed/deep_learning/cnn_baseline"
)

manifest_paths = {
    "train": MANIFEST_DIR / "train_cnn_manifest.csv",
    "validation": MANIFEST_DIR / "validation_cnn_manifest.csv",
    "test": MANIFEST_DIR / "test_cnn_manifest.csv",
}

for split, path in manifest_paths.items():
    print(f"\n=== {split.upper()} ===")

    if not path.exists():
        print("MISSING:", path)
        continue

    df = pd.read_csv(path)

    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    label_col = (
        "diagnosis" if "diagnosis" in df.columns
        else "disease_label"
    )

    print("\nClass counts:")
    print(df[label_col].value_counts().sort_index())

    if "patient_id" in df.columns:
        print("Unique patients:", df["patient_id"].nunique())

In [ ]:
import pandas as pd

train_df = pd.read_csv(manifest_paths["train"])
val_df = pd.read_csv(manifest_paths["validation"])
test_df = pd.read_csv(manifest_paths["test"])

CLASS_NAMES = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI",
]

LABEL_MAP_8 = {
    name: index for index, name in enumerate(CLASS_NAMES)
}

print("=== CLASS MAPPING ===")
for name, index in LABEL_MAP_8.items():
    print(f"{index}: {name}")

print("\n=== PATIENT OVERLAP CHECK ===")
train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])
test_patients = set(test_df["patient_id"])

print("Train ∩ Validation:", len(train_patients & val_patients))
print("Train ∩ Test:", len(train_patients & test_patients))
print("Validation ∩ Test:", len(val_patients & test_patients))

print("\n=== CLASS SUPPORT BY SPLIT ===")
for split, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    counts = df["diagnosis"].value_counts()
    print(f"\n{split}:")
    for name in CLASS_NAMES:
        print(f"  {name:<18} {counts.get(name, 0)}")

missing_val = sorted(set(CLASS_NAMES) - set(val_df["diagnosis"]))
print("\nClasses absent from validation:", missing_val)

In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

search_dirs = [
    PROJECT_ROOT / "data/processed/deep_learning",
    PROJECT_ROOT / "data/processed",
]

print("=== FEATURE CACHE FILES ===")

seen = set()
for root in search_dirs:
    if not root.exists():
        print("Missing directory:", root)
        continue

    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in {
            ".npy", ".npz", ".pt", ".pth", ".pkl", ".pickle", ".json"
        }:
            if path not in seen:
                seen.add(path)
                print(path.relative_to(PROJECT_ROOT))

In [ ]:
from pathlib import Path
import numpy as np

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data/processed/deep_learning/lightweight_multifeature_cnn/feature_cache"
)

for split in ["train", "validation", "test"]:
    path = CACHE_DIR / f"{split}_features.npz"

    print(f"\n{'=' * 20} {split.upper()} {'=' * 20}")

    with np.load(path, allow_pickle=True) as data:
        print("Keys:", data.files)

        for key in data.files:
            arr = data[key]
            print(
                f"{key}: shape={arr.shape}, dtype={arr.dtype}"
            )

            if key.lower() in {"y", "labels", "targets", "disease_label"}:
                print("  Unique labels:", np.unique(arr, return_counts=True))

In [ ]:
from pathlib import Path
import json

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

paths_to_inspect = [
    PROJECT_ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn/normalization_stats.json",
    PROJECT_ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn_5class_trainonly_norm/normalization_stats.json",
    PROJECT_ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn/feature_config.json",
    PROJECT_ROOT / "data/processed/deep_learning/lightweight_multifeature_cnn/architecture.json",
]

for path in paths_to_inspect:
    print(f"\n{'=' * 18} {path.name} {'=' * 18}")
    print("Path:", path)
    if path.exists():
        try:
            print(json.dumps(json.loads(path.read_text()), indent=2))
        except Exception as exc:
            print("Could not parse JSON:", exc)
    else:
        print("FILE NOT FOUND")

In [ ]:
import json
import numpy as np
from pathlib import Path

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data/processed/deep_learning/lightweight_multifeature_cnn/feature_cache"
)

EXP8_DIR = (
    PROJECT_ROOT
    / "data/processed/deep_learning/lightweight_multifeature_cnn_8class_trainonly_norm"
)
EXP8_DIR.mkdir(parents=True, exist_ok=True)

# Fit normalization statistics on training data only.
normalization_stats_8class = {}

with np.load(CACHE_DIR / "train_features.npz", allow_pickle=True) as data:
    for feature_name in ["logmel", "mfcc", "chroma"]:
        X = data[feature_name].astype(np.float64)

        mean = float(X.mean())
        std = float(X.std())

        if not np.isfinite(mean) or not np.isfinite(std) or std <= 0:
            raise ValueError(f"Invalid normalization statistics for {feature_name}")

        normalization_stats_8class[feature_name] = {
            "mean": mean,
            "std": std,
        }

        print(
            f"{feature_name}: mean={mean:.8f}, std={std:.8f}"
        )

stats_path = EXP8_DIR / "normalization_stats.json"
stats_path.write_text(
    json.dumps(normalization_stats_8class, indent=2)
)

print("\nSaved train-only normalization statistics to:")
print(stats_path)

In [ ]:
import numpy as np
from pathlib import Path

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data/processed/deep_learning/lightweight_multifeature_cnn/feature_cache"
)

EXP8_DIR = (
    PROJECT_ROOT
    / "data/processed/deep_learning/lightweight_multifeature_cnn_8class_trainonly_norm"
)

FEATURE_NAMES = ["logmel", "mfcc", "chroma"]
normalization_stats_8class = __import__("json").loads(
    (EXP8_DIR / "normalization_stats.json").read_text()
)

for split in ["train", "validation", "test"]:
    source_path = CACHE_DIR / f"{split}_features.npz"
    output_path = EXP8_DIR / f"{split}_features_normalized.npz"

    with np.load(source_path, allow_pickle=True) as data:
        output = {}

        for name in FEATURE_NAMES:
            mean = normalization_stats_8class[name]["mean"]
            std = normalization_stats_8class[name]["std"]

            output[name] = (
                (data[name] - mean) / std
            ).astype(np.float32)

        output["labels"] = data["labels"].copy()
        output["paths"] = data["paths"].copy()

    np.savez_compressed(output_path, **output)

    print(f"\n{split.upper()}: saved {output_path}")
    for name in FEATURE_NAMES:
        arr = output[name]
        print(
            f"  {name}: shape={arr.shape}, "
            f"mean={arr.mean():.4f}, std={arr.std():.4f}"
        )
    print("  labels:", len(output["labels"]))
    print("  unique classes:", np.unique(output["labels"]).tolist())

In [ ]:
import inspect
import torch

print("=== DEVICE ===")
print("device defined:", "device" in globals())
if "device" in globals():
    print("device:", device)

print("\n=== MODEL CLASS ===")
print("LightweightMultiFeatureCNN defined:",
      "LightweightMultiFeatureCNN" in globals())

if "LightweightMultiFeatureCNN" in globals():
    print(inspect.getsource(LightweightMultiFeatureCNN))

print("\n=== TRAINING UTILITIES ===")
for name in [
    "evaluate_model",
    "criterion",
    "NUM_CLASSES",
    "train_loader_exp5",
    "test_loader_exp5",
]:
    print(f"{name}: {'defined' if name in globals() else 'not defined'}")

print("\n=== CHECKPOINT AND OUTPUT DIRECTORY ===")
EXP8_DIR = (
    PROJECT_ROOT
    / "data/processed/deep_learning/lightweight_multifeature_cnn_8class_trainonly_norm"
)
print("Output directory exists:", EXP8_DIR.exists())
print("Existing files:", sorted(p.name for p in EXP8_DIR.iterdir()))

In [ ]:
import torch

print("=== DEVICE ===")
print("device:", device)

print("\n=== MODEL ARCHITECTURE ===")
model_check = LightweightMultiFeatureCNN(8).to(device)
print(model_check)
print("Number of classes:", 8)
print(
    "Total parameters:",
    sum(p.numel() for p in model_check.parameters())
)
print(
    "Trainable parameters:",
    sum(p.numel() for p in model_check.parameters() if p.requires_grad)
)

print("\n=== TRAINING UTILITIES ===")
for name in [
    "evaluate_model",
    "criterion",
    "NUM_CLASSES",
    "train_loader_exp5",
    "test_loader_exp5",
]:
    print(f"{name}: {'defined' if name in globals() else 'not defined'}")

print("\n=== EIGHT-CLASS EXPERIMENT FILES ===")
print("Output directory:", EXP8_DIR)
print("Existing files:", sorted(p.name for p in EXP8_DIR.iterdir()))

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

CLASS_NAMES_8 = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI",
]

class MultiFeatureDataset8Class(Dataset):
    def __init__(self, npz_path):
        with np.load(npz_path, allow_pickle=True) as data:
            self.logmel = torch.from_numpy(data["logmel"].copy()).float()
            self.mfcc = torch.from_numpy(data["mfcc"].copy()).float()
            self.chroma = torch.from_numpy(data["chroma"].copy()).float()
            self.labels = torch.from_numpy(data["labels"].copy()).long()
            self.paths = data["paths"].copy()

        assert len(self.logmel) == len(self.labels) == len(self.paths)
        assert len(self.mfcc) == len(self.labels)
        assert len(self.chroma) == len(self.labels)
        assert self.labels.min().item() >= 0
        assert self.labels.max().item() < 8

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.logmel[idx],
            self.mfcc[idx],
            self.chroma[idx],
            self.labels[idx],
        )

train_ds_8 = MultiFeatureDataset8Class(
    EXP8_DIR / "train_features_normalized.npz"
)
val_ds_8 = MultiFeatureDataset8Class(
    EXP8_DIR / "validation_features_normalized.npz"
)
test_ds_8 = MultiFeatureDataset8Class(
    EXP8_DIR / "test_features_normalized.npz"
)

BATCH_SIZE_8 = 32
SEED_8 = 42

generator_8 = torch.Generator().manual_seed(SEED_8)

train_loader_8 = DataLoader(
    train_ds_8,
    batch_size=BATCH_SIZE_8,
    shuffle=True,
    generator=generator_8,
    num_workers=0,
)
val_loader_8 = DataLoader(
    val_ds_8,
    batch_size=BATCH_SIZE_8,
    shuffle=False,
    num_workers=0,
)
test_loader_8 = DataLoader(
    test_ds_8,
    batch_size=BATCH_SIZE_8,
    shuffle=False,
    num_workers=0,
)

# Weighted cross-entropy weights computed from training data only.
train_labels_8 = train_ds_8.labels.numpy()
class_counts_8 = np.bincount(train_labels_8, minlength=8)

class_weights_8 = len(train_labels_8) / (
    8 * np.maximum(class_counts_8, 1)
)
class_weights_8 = torch.tensor(
    class_weights_8, dtype=torch.float32, device=device
)

print("Dataset sizes:", len(train_ds_8), len(val_ds_8), len(test_ds_8))
print("\nTraining class counts:")
for i, name in enumerate(CLASS_NAMES_8):
    print(
        f"{i}: {name:<18} count={class_counts_8[i]:4d} "
        f"weight={class_weights_8[i].item():.4f}"
    )

batch = next(iter(train_loader_8))
print("\nBatch shapes:", [tuple(x.shape) for x in batch])
print("Device:", device)

In [ ]:
import torch
import torch.nn as nn
import json
from pathlib import Path

# Reproducibility
torch.manual_seed(SEED_8)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED_8)

NUM_CLASSES_8 = 8
EPOCHS_8 = 60
PATIENCE_8 = 10
LEARNING_RATE_8 = 1e-3
WEIGHT_DECAY_8 = 1e-4

# Fresh model: do not reuse the five-class checkpoint.
model_8 = LightweightMultiFeatureCNN(NUM_CLASSES_8).to(device)

criterion_8 = nn.CrossEntropyLoss(weight=class_weights_8)

optimizer_8 = torch.optim.AdamW(
    model_8.parameters(),
    lr=LEARNING_RATE_8,
    weight_decay=WEIGHT_DECAY_8,
)

scheduler_8 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_8,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

BEST_MODEL_8_PATH = EXP8_DIR / "best_lightweight_multifeature_cnn_8class.pt"
HISTORY_8_PATH = EXP8_DIR / "training_history.csv"
CONFIG_8_PATH = EXP8_DIR / "training_config.json"

config_8 = {
    "experiment": "Eight-class Experiment 5 adaptation",
    "num_classes": NUM_CLASSES_8,
    "class_names": CLASS_NAMES_8,
    "seed": SEED_8,
    "batch_size": BATCH_SIZE_8,
    "epochs_max": EPOCHS_8,
    "early_stopping_patience": PATIENCE_8,
    "learning_rate": LEARNING_RATE_8,
    "weight_decay": WEIGHT_DECAY_8,
    "loss": "weighted_cross_entropy",
    "normalization": "training split only, second-stage normalization",
    "optimizer": "AdamW",
    "scheduler": "ReduceLROnPlateau monitoring validation macro-F1",
    "train_class_counts": class_counts_8.tolist(),
    "class_weights": class_weights_8.detach().cpu().tolist(),
    "parameter_count": sum(p.numel() for p in model_8.parameters()),
    "device": str(device),
}

CONFIG_8_PATH.write_text(json.dumps(config_8, indent=2))

print("Model initialized.")
print("Parameters:", config_8["parameter_count"])
print("Loss:", criterion_8)
print("Optimizer:", optimizer_8.__class__.__name__)
print("Scheduler:", scheduler_8.__class__.__name__)
print("Checkpoint path:", BEST_MODEL_8_PATH)
print("Config saved:", CONFIG_8_PATH.exists())

In [ ]:
import time
import copy
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, recall_score

history_8 = []
best_val_macro_f1_8 = -1.0
best_epoch_8 = 0
epochs_without_improvement_8 = 0
start_time_8 = time.time()

def run_epoch_8(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    total_samples = 0
    all_targets = []
    all_predictions = []

    with torch.set_grad_enabled(training):
        for logmel, mfcc, chroma, labels in loader:
            logmel = logmel.to(device)
            mfcc = mfcc.to(device)
            chroma = chroma.to(device)
            labels = labels.to(device)

            if training:
                optimizer.zero_grad(set_to_none=True)

            logits = model(logmel, mfcc, chroma)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            all_targets.extend(labels.detach().cpu().numpy().tolist())
            all_predictions.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())

    y_true = np.asarray(all_targets, dtype=int)
    y_pred = np.asarray(all_predictions, dtype=int)

    return {
        "loss": total_loss / total_samples,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": recall_score(
            y_true,
            y_pred,
            labels=list(range(NUM_CLASSES_8)),
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=list(range(NUM_CLASSES_8)),
            average="macro",
            zero_division=0,
        ),
        "targets": y_true,
        "predictions": y_pred,
    }

for epoch in range(1, EPOCHS_8 + 1):
    train_metrics = run_epoch_8(
        model_8, train_loader_8, criterion_8, optimizer_8
    )
    val_metrics = run_epoch_8(
        model_8, val_loader_8, criterion_8
    )

    scheduler_8.step(val_metrics["macro_f1"])

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "learning_rate": optimizer_8.param_groups[0]["lr"],
    }
    history_8.append(row)

    improved = val_metrics["macro_f1"] > best_val_macro_f1_8

    if improved:
        best_val_macro_f1_8 = val_metrics["macro_f1"]
        best_epoch_8 = epoch
        epochs_without_improvement_8 = 0

        torch.save(
            {
                "model_state_dict": model_8.state_dict(),
                "num_classes": NUM_CLASSES_8,
                "class_names": CLASS_NAMES_8,
                "best_epoch": best_epoch_8,
                "best_val_macro_f1": best_val_macro_f1_8,
                "seed": SEED_8,
                "normalization": normalization_stats_8class,
                "experiment": "Eight-class Experiment 5 adaptation",
                "config": config_8,
            },
            BEST_MODEL_8_PATH,
        )
    else:
        epochs_without_improvement_8 += 1

    pd.DataFrame(history_8).to_csv(HISTORY_8_PATH, index=False)

    print(
        f"Epoch {epoch:02d}/{EPOCHS_8} | "
        f"train loss={train_metrics['loss']:.4f} | "
        f"val loss={val_metrics['loss']:.4f} | "
        f"val acc={val_metrics['accuracy']:.4f} | "
        f"val macro-F1={val_metrics['macro_f1']:.4f} | "
        f"best={best_val_macro_f1_8:.4f} (epoch {best_epoch_8})"
    )

    if epochs_without_improvement_8 >= PATIENCE_8:
        print(f"Early stopping at epoch {epoch}.")
        break

print

In [ ]:
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Load the best checkpoint selected by validation macro-F1
checkpoint_8 = torch.load(
    BEST_MODEL_8_PATH,
    map_location=device,
    weights_only=False,
)

best_model_8 = LightweightMultiFeatureCNN(NUM_CLASSES_8).to(device)
best_model_8.load_state_dict(checkpoint_8["model_state_dict"])
best_model_8.eval()

val_metrics_8 = run_epoch_8(
    best_model_8,
    val_loader_8,
    criterion_8,
)

y_true_val_8 = val_metrics_8["targets"]
y_pred_val_8 = val_metrics_8["predictions"]

print("=== BEST 8-CLASS CHECKPOINT ===")
print("Best epoch:", checkpoint_8["best_epoch"])
print("Best validation macro-F1:", round(checkpoint_8["best_val_macro_f1"], 4))
print("Validation accuracy:", round(val_metrics_8["accuracy"], 4))
print("Validation balanced accuracy:", round(val_metrics_8["balanced_accuracy"], 4))

print("\n=== PER-CLASS VALIDATION REPORT ===")
print(
    classification_report(
        y_true_val_8,
        y_pred_val_8,
        labels=list(range(NUM_CLASSES_8)),
        target_names=CLASS_NAMES_8,
        zero_division=0,
        digits=4,
    )
)

print("=== VALIDATION CONFUSION MATRIX ===")
print(
    confusion_matrix(
        y_true_val_8,
        y_pred_val_8,
        labels=list(range(NUM_CLASSES_8)),
    )
)

In [ ]:
import numpy as np
import torch

# Effective-number weighting: training labels only
beta_8 = 0.999

counts = class_counts_8.astype(np.float64)

effective_num = 1.0 - np.power(beta_8, counts)
weights_effective_8 = (1.0 - beta_8) / effective_num

# Normalize weights so their mean is 1.
weights_effective_8 = weights_effective_8 / weights_effective_8.mean()

weights_effective_8_tensor = torch.tensor(
    weights_effective_8,
    dtype=torch.float32,
    device=device,
)

print("=== EFFECTIVE-NUMBER WEIGHTS ===")
for i, name in enumerate(CLASS_NAMES_8):
    print(
        f"{name:<18} count={int(counts[i]):4d} "
        f"weight={weights_effective_8[i]:.4f}"
    )

print("\nOriginal inverse-frequency weight range:",
      round(float(class_weights_8.min()), 4),
      "to",
      round(float(class_weights_8.max()), 4))

print("Effective-number weight range:",
      round(float(weights_effective_8_tensor.min()), 4),
      "to",
      round(float(weights_effective_8_tensor.max()), 4))

In [ ]:
import time
import json
import pandas as pd
import torch
import torch.nn as nn

EXP8B_DIR = EXP8_DIR / "effective_number_weights"
EXP8B_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_8B_PATH = (
    EXP8B_DIR / "best_lightweight_multifeature_cnn_8class.pt"
)
HISTORY_8B_PATH = EXP8B_DIR / "training_history.csv"
CONFIG_8B_PATH = EXP8B_DIR / "training_config.json"

# Fresh model and optimizer for a fair independent experiment
torch.manual_seed(SEED_8)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED_8)

model_8b = LightweightMultiFeatureCNN(NUM_CLASSES_8).to(device)

criterion_8b = nn.CrossEntropyLoss(
    weight=weights_effective_8_tensor
)

optimizer_8b = torch.optim.AdamW(
    model_8b.parameters(),
    lr=LEARNING_RATE_8,
    weight_decay=WEIGHT_DECAY_8,
)

scheduler_8b = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_8b,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

history_8b = []
best_val_macro_f1_8b = -1.0
best_epoch_8b = 0
patience_count_8b = 0
start_8b = time.time()

config_8b = {
    **config_8,
    "experiment": "Eight-class effective-number weighted CE",
    "loss": "effective-number weighted cross-entropy",
    "beta": beta_8,
    "class_weights": weights_effective_8.tolist(),
}
CONFIG_8B_PATH.write_text(json.dumps(config_8b, indent=2))

for epoch in range(1, EPOCHS_8 + 1):
    train_metrics = run_epoch_8(
        model_8b, train_loader_8, criterion_8b, optimizer_8b
    )
    val_metrics = run_epoch_8(
        model_8b, val_loader_8, criterion_8b
    )

    scheduler_8b.step(val_metrics["macro_f1"])

    history_8b.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "learning_rate": optimizer_8b.param_groups[0]["lr"],
    })

    if val_metrics["macro_f1"] > best_val_macro_f1_8b:
        best_val_macro_f1_8b = val_metrics["macro_f1"]
        best_epoch_8b = epoch
        patience_count_8b = 0

        torch.save({
            "model_state_dict": model_8b.state_dict(),
            "num_classes": NUM_CLASSES_8,
            "class_names": CLASS_NAMES_8,
            "best_epoch": best_epoch_8b,
            "best_val_macro_f1": best_val_macro_f1_8b,
            "seed": SEED_8,
            "normalization": normalization_stats_8class,
            "experiment": "Eight-class effective-number weighted CE",
            "config": config_8b,
        }, BEST_MODEL_8B_PATH)
    else:
        patience_count_8b += 1

    pd.DataFrame(history_8b).to_csv(HISTORY_8B_PATH, index=False)

    print(
        f"Epoch {epoch:02d}/{EPOCHS_8} | "
        f"train loss={train_metrics['loss']:.4f} | "
        f"val loss={val_metrics['loss']:.4f} | "
        f"val acc={val_metrics['accuracy']:.4f} | "
        f"val macro-F1={val_metrics['macro_f1']:.4f} | "
        f"best={best_val_macro_f1_8b:.4f} (epoch {best_epoch_8b})"
    )

    if patience_count_8b >= PATIENCE_8:
        print(f"Early stopping at epoch {epoch}.")
        break

print("\n=== EXPERIMENT 8B COMPLETE ===")
print("Best epoch:", best_epoch_8b)
print("Best validation macro-F1:", round(best_val_macro_f1_8b, 4))
print("Checkpoint exists:", BEST_MODEL_8B_PATH.exists())
print("History saved:", HISTORY_8B_PATH.exists())
print("Elapsed minutes:", round((time.time() - start_8b) / 60, 2))

In [ ]:
import torch
from sklearn.metrics import classification_report, confusion_matrix

checkpoint_8b = torch.load(
    BEST_MODEL_8B_PATH,
    map_location=device,
    weights_only=False,
)

best_model_8b = LightweightMultiFeatureCNN(NUM_CLASSES_8).to(device)
best_model_8b.load_state_dict(checkpoint_8b["model_state_dict"])
best_model_8b.eval()

val_metrics_8b = run_epoch_8(
    best_model_8b,
    val_loader_8,
    criterion_8b,
)

y_true_val_8b = val_metrics_8b["targets"]
y_pred_val_8b = val_metrics_8b["predictions"]

print("=== EXPERIMENT 8B: BEST VALIDATION CHECKPOINT ===")
print("Best epoch:", checkpoint_8b["best_epoch"])
print("Best validation macro-F1:", round(checkpoint_8b["best_val_macro_f1"], 4))
print("Validation accuracy:", round(val_metrics_8b["accuracy"], 4))
print("Validation balanced accuracy:", round(val_metrics_8b["balanced_accuracy"], 4))

print("\n=== PER-CLASS VALIDATION REPORT ===")
print(classification_report(
    y_true_val_8b,
    y_pred_val_8b,
    labels=list(range(NUM_CLASSES_8)),
    target_names=CLASS_NAMES_8,
    zero_division=0,
    digits=4,
))

print("=== VALIDATION CONFUSION MATRIX ===")
print(confusion_matrix(
    y_true_val_8b,
    y_pred_val_8b,
    labels=list(range(NUM_CLASSES_8)),
))

In [ ]:
from collections import Counter
import numpy as np
import torch

labels_8 = np.asarray(train_ds_8.labels, dtype=np.int64)
counts_8 = np.bincount(labels_8, minlength=NUM_CLASSES_8)

sample_weights_8 = 1.0 / np.maximum(counts_8[labels_8], 1)

print("=== EXPERIMENT 8C: SAMPLER READINESS CHECK ===")
print("Training samples:", len(labels_8))
print("Class counts:", dict(zip(CLASS_NAMES_8, counts_8.tolist())))
print("Sample weights shape:", sample_weights_8.shape)
print("Sample weights finite:", bool(np.isfinite(sample_weights_8).all()))
print("Sample weights positive:", bool((sample_weights_8 > 0).all()))
print("Unique sample weights:", len(np.unique(sample_weights_8)))
print("Current training loader uses shuffle:", getattr(train_loader_8, "sampler", None).__class__.__name__)

In [ ]:

from pathlib import Path
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import torch
import json

# Separate output directory; do not overwrite 8A or 8B.
EXP8C_DIR = EXP8_DIR / "balanced_sampler"
EXP8C_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_8C_PATH = EXP8C_DIR / "best_lightweight_multifeature_cnn_8class.pt"
HISTORY_8C_PATH = EXP8C_DIR / "training_history.json"
CONFIG_8C_PATH = EXP8C_DIR / "experiment_config.json"

# Build weights from training labels only.
labels_8c = np.asarray(train_ds_8.labels, dtype=np.int64)
counts_8c = np.bincount(labels_8c, minlength=NUM_CLASSES_8)

sample_weights_8c = 1.0 / counts_8c[labels_8c]

sampler_8c = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights_8c, dtype=torch.double),
    num_samples=len(sample_weights_8c),
    replacement=True,
    generator=torch.Generator().manual_seed(SEED_8),
)

train_loader_8c = DataLoader(
    train_ds_8,
    batch_size=BATCH_SIZE_8,
    sampler=sampler_8c,
    num_workers=0,
)

# Validation must remain deterministic and unbalanced.
val_loader_8c = DataLoader(
    val_ds_8,
    batch_size=BATCH_SIZE_8,
    shuffle=False,
    num_workers=0,
)

print("=== EXPERIMENT 8C: CONFIGURATION ===")
print("Training samples:", len(train_ds_8))
print("Class counts:", dict(zip(CLASS_NAMES_8, counts_8c.tolist())))
print("Sampler:", type(train_loader_8c.sampler).__name__)
print("Replacement:", train_loader_8c.sampler.replacement)
print("Samples drawn per epoch:", train_loader_8c.sampler.num_samples)
print("Validation sampler:", type(val_loader_8c.sampler).__name__)
print("Output directory:", EXP8C_DIR)
print("Checkpoint path:", BEST_MODEL_8C_PATH)


In [ ]:

import torch
import torch.nn as nn
import json

# Reproducibility
torch.manual_seed(SEED_8)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_8)

# Same architecture as Experiments 8A and 8B
model_8c = LightweightMultiFeatureCNN(NUM_CLASSES_8).to(device)

# No class weights: balancing is handled by WeightedRandomSampler
criterion_8c = nn.CrossEntropyLoss()

optimizer_8c = torch.optim.AdamW(
    model_8c.parameters(),
    lr=LEARNING_RATE_8,
    weight_decay=WEIGHT_DECAY_8,
)

scheduler_8c = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_8c,
    mode="max",
    factor=0.5,
    patience=3,
)

config_8c = {
    "experiment": "8C_balanced_sampler",
    "num_classes": NUM_CLASSES_8,
    "class_names": CLASS_NAMES_8,
    "seed": SEED_8,
    "batch_size": BATCH_SIZE_8,
    "epochs": EPOCHS_8,
    "early_stopping_patience": PATIENCE_8,
    "learning_rate": LEARNING_RATE_8,
    "weight_decay": WEIGHT_DECAY_8,
    "loss": "CrossEntropyLoss without class weights",
    "sampler": "WeightedRandomSampler with replacement",
    "training_samples_per_epoch": len(train_ds_8),
    "architecture": "LightweightMultiFeatureCNN",
}

with open(CONFIG_8C_PATH, "w") as f:
    json.dump(config_8c, f, indent=2)

print("=== EXPERIMENT 8C: MODEL INITIALIZED ===")
print("Device:", device)
print("Trainable parameters:", sum(
    p.numel() for p in model_8c.parameters() if p.requires_grad
))
print("Loss:", type(criterion_8c).__name__)
print("Optimizer:", type(optimizer_8c).__name__)
print("Scheduler:", type(scheduler_8c).__name__)
print("Configuration saved:", CONFIG_8C_PATH)
print("Checkpoint will save to:", BEST_MODEL_8C_PATH)


In [ ]:

import copy
import json
import torch

history_8c = []
best_val_macro_f1_8c = -1.0
best_epoch_8c = 0
epochs_without_improvement_8c = 0

for epoch in range(1, EPOCHS_8 + 1):
    train_metrics_8c = run_epoch_8(
        model_8c,
        train_loader_8c,
        criterion_8c,
        optimizer_8c,
    )

    val_metrics_8c = run_epoch_8(
        model_8c,
        val_loader_8c,
        criterion_8c,
    )

    scheduler_8c.step(val_metrics_8c["macro_f1"])

    epoch_record_8c = {
        "epoch": epoch,
        "train_loss": float(train_metrics_8c["loss"]),
        "train_accuracy": float(train_metrics_8c["accuracy"]),
        "train_macro_f1": float(train_metrics_8c["macro_f1"]),
        "val_loss": float(val_metrics_8c["loss"]),
        "val_accuracy": float(val_metrics_8c["accuracy"]),
        "val_balanced_accuracy": float(val_metrics_8c["balanced_accuracy"]),
        "val_macro_f1": float(val_metrics_8c["macro_f1"]),
        "learning_rate": float(optimizer_8c.param_groups[0]["lr"]),
    }
    history_8c.append(epoch_record_8c)

    improved_8c = (
        val_metrics_8c["macro_f1"] > best_val_macro_f1_8c
    )

    if improved_8c:
        best_val_macro_f1_8c = float(val_metrics_8c["macro_f1"])
        best_epoch_8c = epoch
        epochs_without_improvement_8c = 0

        torch.save(
            {
                "model_state_dict": model_8c.state_dict(),
                "best_epoch": best_epoch_8c,
                "best_val_macro_f1": best_val_macro_f1_8c,
                "class_names": CLASS_NAMES_8,
                "num_classes": NUM_CLASSES_8,
                "experiment": "8C_balanced_sampler",
            },
            BEST_MODEL_8C_PATH,
        )
    else:
        epochs_without_improvement_8c += 1

    print(
        f"Epoch {epoch:02d}/{EPOCHS_8} | "
        f"Train loss: {train_metrics_8c['loss']:.4f} | "
        f"Train macro-F1: {train_metrics_8c['macro_f1']:.4f} | "
        f"Val loss: {val_metrics_8c['loss']:.4f} | "
        f"Val macro-F1: {val_metrics_8c['macro_f1']:.4f} | "
        f"Val accuracy: {val_metrics_8c['accuracy']:.4f}"
        + (" | BEST" if improved_8c else "")
    )

    with open(HISTORY_8C_PATH, "w") as f:
        json.dump(history_8c, f, indent=2)

    if epochs_without_improvement_8c >= PATIENCE_8:
        print(
            f"\nEarly stopping at epoch {epoch}. "
            f"Best epoch: {best_epoch_8c}."
        )
        break

print("\n=== EXPERIMENT 8C TRAINING SUMMARY ===")
print("Best epoch:", best_epoch_8c)
print("Best validation macro-F1:", round(best_val_macro_f1_8c, 4))
print("Checkpoint saved:", BEST_MODEL_8C_PATH)
print("History saved:", HISTORY_8C_PATH)


In [ ]:

import torch
from sklearn.metrics import classification_report, confusion_matrix

checkpoint_8c = torch.load(
    BEST_MODEL_8C_PATH,
    map_location=device,
    weights_only=False,
)

best_model_8c = LightweightMultiFeatureCNN(NUM_CLASSES_8).to(device)
best_model_8c.load_state_dict(checkpoint_8c["model_state_dict"])
best_model_8c.eval()

val_metrics_8c = run_epoch_8(
    best_model_8c,
    val_loader_8c,
    criterion_8c,
)

y_true_val_8c = val_metrics_8c["targets"]
y_pred_val_8c = val_metrics_8c["predictions"]

print("=== EXPERIMENT 8C: BEST VALIDATION CHECKPOINT ===")
print("Best epoch:", checkpoint_8c["best_epoch"])
print("Best validation macro-F1:", round(checkpoint_8c["best_val_macro_f1"], 4))
print("Validation accuracy:", round(val_metrics_8c["accuracy"], 4))
print("Validation balanced accuracy:", round(val_metrics_8c["balanced_accuracy"], 4))

print("\n=== PER-CLASS VALIDATION REPORT ===")
print(classification_report(
    y_true_val_8c,
    y_pred_val_8c,
    labels=list(range(NUM_CLASSES_8)),
    target_names=CLASS_NAMES_8,
    zero_division=0,
    digits=4,
))

print("=== VALIDATION CONFUSION MATRIX ===")
print(confusion_matrix(
    y_true_val_8c,
    y_pred_val_8c,
    labels=list(range(NUM_CLASSES_8)),
))


In [ ]:

from pathlib import Path
import json

combined_dir = (
    PROJECT_ROOT
    / "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1"
)

print("=== PREVIOUS COMBINED DATASET EXPERIMENT ===")
print("Directory exists:", combined_dir.exists())
print("Directory:", combined_dir)

if combined_dir.exists():
    print("\n=== FILES AND SUBDIRECTORIES ===")
    for path in sorted(combined_dir.rglob("*")):
        if path.is_file():
            print(f"{path.relative_to(combined_dir)} ({path.stat().st_size:,} bytes)")
        elif path.is_dir():
            print(f"{path.relative_to(combined_dir)}/")

    print("\n=== MODEL / RESULT ARTIFACTS ===")
    keywords = (
        "best", "checkpoint", "model", "history",
        "result", "metric", "manifest", "config",
        "classification", "confusion"
    )

    for path in sorted(combined_dir.rglob("*")):
        if path.is_file() and any(k in path.name.lower() for k in keywords):
            print(path)
else:
    print(
        "\nThe expected directory was not found. "
        "We will locate the existing experiment artifacts next."
    )


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = (
    PROJECT_ROOT
    / "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1"
)

result_files = [
    ROOT / "experiment_b_feature_cache/experiment_b_v1_strict/evaluation/experiment_b_test_summary.csv",
    ROOT / "experiment_b_feature_cache/experiment_b_v1_strict/evaluation/experiment_b_test_summary.json",
    ROOT / "experiment_c1_icbhi_only/evaluation_c1_normalized/summary.csv",
    ROOT / "experiment_c2_icbhi_fraiwan_high_capacity/c2_compact_multifeature_training_history.json",
]

for path in result_files:
    print("\n" + "=" * 80)
    print(path.relative_to(ROOT))
    print("Exists:", path.exists())

    if not path.exists():
        continue

    try:
        if path.suffix == ".csv":
            print(pd.read_csv(path).to_string(index=False))
        elif path.suffix == ".json":
            with open(path, "r") as f:
                data = json.load(f)
            if isinstance(data, list):
                print(pd.DataFrame(data).tail(5).to_string(index=False))
            else:
                print(json.dumps(data, indent=2))
    except Exception as exc:
        print("Could not parse this file:", repr(exc))


In [ ]:
import numpy as np
from pathlib import Path

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

CACHE_DIR = (
    PROJECT_ROOT
    / "data/processed/deep_learning/lightweight_multifeature_cnn/feature_cache"
)

EXP8_DIR = (
    PROJECT_ROOT
    / "data/processed/deep_learning/lightweight_multifeature_cnn_8class_trainonly_norm"
)

FEATURE_NAMES = ["logmel", "mfcc", "chroma"]
normalization_stats_8class = __import__("json").loads(
    (EXP8_DIR / "normalization_stats.json").read_text()
)

for split in ["train", "validation", "test"]:
    source_path = CACHE_DIR / f"{split}_features.npz"
    output_path = EXP8_DIR / f"{split}_features_normalized.npz"

    with np.load(source_path, allow_pickle=True) as data:
        output = {}

        for name in FEATURE_NAMES:
            mean = normalization_stats_8class[name]["mean"]
            std = normalization_stats_8class[name]["std"]

            output[name] = (
                (data[name] - mean) / std
            ).astype(np.float32)

        output["labels"] = data["labels"].copy()
        output["paths"] = data["paths"].copy()

    np.savez_compressed(output_path, **output)

    print(f"\n{split.upper()}: saved {output_path}")
    for name in FEATURE_NAMES:
        arr = output[name]
        print(
            f"  {name}: shape={arr.shape}, "
            f"mean={arr.mean():.4f}, std={arr.std():.4f}"
        )
    print("  labels:", len(output["labels"]))
    print("  unique classes:", np.unique(output["labels"]).tolist())

In [ ]:

import pandas as pd

ROOT_COMBINED = (
    PROJECT_ROOT
    / "data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1"
)

manifest_paths = {
    "Combined training": ROOT_COMBINED / "experiment_b_train_manifest.csv",
    "ICBHI test": ROOT_COMBINED / "experiment_b_icbhi_test_manifest.csv",
    "Fraiwan external test": ROOT_COMBINED / "experiment_b_fraiwan_test_manifest.csv",
}

for name, path in manifest_paths.items():
    print("\n" + "=" * 80)
    print(name)
    print("File exists:", path.exists())

    if not path.exists():
        continue

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    for col in df.columns:
        low = col.lower()
        if any(key in low for key in [
            "source", "dataset", "patient", "subject",
            "label", "class", "disease", "recording"
        ]):
            print(f"\n{col} — {df[col].nunique(dropna=True)} unique values")
            if any(key in low for key in [
                "source", "dataset", "label", "class", "disease"
            ]):
                print(df[col].value_counts(dropna=False).to_string())


In [ ]:

import pandas as pd
import re

train_df = pd.read_csv(ROOT_COMBINED / "experiment_b_train_manifest.csv")
icbhi_test_df = pd.read_csv(ROOT_COMBINED / "experiment_b_icbhi_test_manifest.csv")
fraiwan_test_df = pd.read_csv(ROOT_COMBINED / "experiment_b_fraiwan_test_manifest.csv")

fraiwan_all = pd.concat(
    [
        train_df[train_df["source_dataset"].str.lower() == "fraiwan"],
        fraiwan_test_df,
    ],
    ignore_index=True,
)

print("=== FRAIWAN RECORDING AND PATIENT COUNTS ===")
print("Train recordings:", int((train_df["source_dataset"].str.lower() == "fraiwan").sum()))
print("External test recordings:", len(fraiwan_test_df))
print("Combined unique patient IDs:", fraiwan_all["patient_id"].nunique())

print("\n=== EXTERNAL TEST PATIENT GROUPS ===")
print(
    fraiwan_test_df.groupby("patient_id")["source_recording_id"]
    .agg(["count", lambda s: " | ".join(s.astype(str))])
    .to_string()
)

print("\n=== FILENAME PREFIX COUNTS ===")
fraiwan_all = fraiwan_all.copy()
fraiwan_all["prefix"] = fraiwan_all["source_recording_id"].astype(str).str.extract(
    r"^([A-Za-z]+)"
)[0]

print(fraiwan_all.groupby(["split", "prefix"]).size().to_string())

print("\n=== POSSIBLE DUPLICATE RECORDING NAMES ===")
fraiwan_all["normalized_name"] = (
    fraiwan_all["source_recording_id"]
    .astype(str)
    .str.lower()
    .str.replace(r"^(bp|dp|ep)", "", regex=True)
)

duplicates = fraiwan_all[
    fraiwan_all.duplicated("normalized_name", keep=False)
].sort_values("normalized_name")

if duplicates.empty:
    print("No duplicates found by prefix-normalized filename.")
else:
    print(
        duplicates[
            ["source_dataset", "source_recording_id", "patient_id", "split", "normalized_name"]
        ].to_string(index=False)
    )

print("\n=== PATIENT OVERLAP BETWEEN FRAIWAN TRAIN AND EXTERNAL TEST ===")
train_fraiwan_patients = set(
    train_df.loc[
        train_df["source_dataset"].str.lower() == "fraiwan", "patient_id"
    ].dropna()
)
test_fraiwan_patients = set(fraiwan_test_df["patient_id"].dropna())

print("Overlapping patient IDs:", sorted(train_fraiwan_patients & test_fraiwan_patients))


In [ ]:

from pathlib import Path

search_roots = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT,
]

seen = set()

print("=== FRAIWAN AUDIO FILE LOCATOR ===")

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        name = path.name.lower()
        if (
            name.endswith(".wav")
            and name.startswith(("bp", "dp", "ep"))
            and any(term in str(path).lower() for term in [
                "fraiwan", "combined_icbhi_fraiwan", "hf_lung"
            ])
        ):
            resolved = str(path.resolve())
            if resolved not in seen:
                seen.add(resolved)
                print(path)

print("\nTotal matching WAV files:", len(seen))


In [ ]:

import pandas as pd

train_df = pd.read_csv(ROOT_COMBINED / "experiment_b_train_manifest.csv")
fraiwan_test_df = pd.read_csv(
    ROOT_COMBINED / "experiment_b_fraiwan_test_manifest.csv"
)

fraiwan_train = train_df[
    train_df["source_dataset"].str.lower() == "fraiwan"
].copy()

fraiwan_all = pd.concat(
    [fraiwan_train, fraiwan_test_df],
    ignore_index=True
)

print("=== FRAIWAN COUNTS ===")
print("Training recordings:", len(fraiwan_train))
print("External test recordings:", len(fraiwan_test_df))
print("Unique patient IDs across both:", fraiwan_all["patient_id"].nunique())

print("\n=== EXTERNAL TEST RECORDINGS PER PATIENT ===")
print(
    fraiwan_test_df.groupby("patient_id")["source_recording_id"]
    .agg(lambda s: " | ".join(s.astype(str)))
    .to_string()
)

print("\n=== PREFIX-NORMALIZED NAME MATCHES ===")
fraiwan_all["normalized_name"] = (
    fraiwan_all["source_recording_id"]
    .astype(str)
    .str.lower()
    .str.replace(r"^(bp|dp|ep)", "", regex=True)
)

matches = fraiwan_all[
    fraiwan_all.duplicated("normalized_name", keep=False)
].sort_values("normalized_name")

if matches.empty:
    print("No matching names after removing BP/DP/EP prefixes.")
else:
    print(
        matches[
            ["source_recording_id", "patient_id", "split", "normalized_name"]
        ].to_string(index=False)
    )

print("\n=== PATIENT OVERLAP ===")
train_patients = set(fraiwan_train["patient_id"].dropna())
test_patients = set(fraiwan_test_df["patient_id"].dropna())
overlap = train_patients & test_patients

print("Overlapping patient IDs:", sorted(overlap))
print("Patient-disjoint:", len(overlap) == 0)


In [ ]:

from pathlib import Path
import hashlib
import pandas as pd

train_df = pd.read_csv(
    ROOT_COMBINED / "experiment_b_train_manifest.csv"
)
test_df = pd.read_csv(
    ROOT_COMBINED / "experiment_b_fraiwan_test_manifest.csv"
)

fraiwan_train = train_df[
    train_df["source_dataset"].str.lower() == "fraiwan"
]
fraiwan_all = pd.concat(
    [fraiwan_train, test_df],
    ignore_index=True
)

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

rows = []

for _, row in fraiwan_all.iterrows():
    path = Path(str(row["recording_path"]))

    # Support relative paths if the manifest stores them that way.
    if not path.is_absolute():
        path = PROJECT_ROOT / path

    result = {
        "source_recording_id": row["source_recording_id"],
        "patient_id": row["patient_id"],
        "split": row["split"],
        "path_exists": path.is_file(),
        "file_size_bytes": None,
        "sha256": None,
    }

    if path.is_file():
        result["file_size_bytes"] = path.stat().st_size
        result["sha256"] = sha256_file(path)

    rows.append(result)

audit_df = pd.DataFrame(rows)

print("=== FILE AVAILABILITY ===")
print(audit_df["path_exists"].value_counts(dropna=False).to_string())

print("\n=== RECORDING VARIANT AUDIT ===")
print(
    audit_df[
        ["source_recording_id", "patient_id", "split",
         "path_exists", "file_size_bytes"]
    ].to_string(index=False)
)

existing = audit_df[audit_df["sha256"].notna()].copy()
print("\n=== IDENTICAL FILE HASHES ===")
hash_groups = existing.groupby("sha256").filter(
    lambda group: len(group) > 1
)

if hash_groups.empty:
    print("No byte-identical files found among accessible manifest recordings.")
else:
    print(
        hash_groups[
            ["source_recording_id", "patient_id", "split", "sha256"]
        ].sort_values("sha256").to_string(index=False)
    )


In [ ]:

# Uses audit_df and existing hashes from the previous cell.

valid_hashes = audit_df.dropna(subset=["sha256"]).copy()

print("=== DUPLICATE AUDIO SUMMARY ===")

duplicate_groups = (
    valid_hashes.groupby("sha256")
    .agg(
        file_count=("sha256", "size"),
        filenames=("source_recording_id", lambda s: list(s)),
        patient_ids=("patient_id", lambda s: sorted(set(s))),
        splits=("split", lambda s: sorted(set(s))),
    )
    .reset_index()
)

duplicate_groups = duplicate_groups[
    duplicate_groups["file_count"] > 1
].sort_values("file_count", ascending=False)

if duplicate_groups.empty:
    print("No duplicate audio files found.")
else:
    print("Number of duplicate hash groups:", len(duplicate_groups))
    print(
        duplicate_groups[
            ["file_count", "filenames", "patient_ids", "splits"]
        ].to_string(index=False)
    )

print("\n=== UNIQUE AUDIO COUNTS ===")
print("Manifest rows:", len(audit_df))
print("Files with available hashes:", len(valid_hashes))
print("Unique audio contents:", valid_hashes["sha256"].nunique())
print(
    "Duplicate excess rows:",
    len(valid_hashes) - valid_hashes["sha256"].nunique()
)

print("\n=== CROSS-SPLIT DUPLICATE CHECK ===")
if duplicate_groups.empty:
    print("No byte-identical duplicate groups to check.")
else:
    cross_split = duplicate_groups[
        duplicate_groups["splits"].apply(lambda x: len(x) > 1)
    ]
    if cross_split.empty:
        print("No byte-identical audio found across different splits.")
    else:
        print(cross_split[["filenames", "patient_ids", "splits"]].to_string(index=False))


In [ ]:

from pathlib import Path
import pandas as pd

train_path = ROOT_COMBINED / "experiment_b_train_manifest.csv"
train_df = pd.read_csv(train_path)

mask = train_df["source_recording_id"].astype(str).str.match(
    r"^(BP9|DP9)_", case=False
)

columns = [
    c for c in [
        "source_dataset",
        "source_recording_id",
        "patient_id",
        "diagnosis",
        "split",
        "recording_path",
    ]
    if c in train_df.columns
]

print("=== BP9 / DP9 MANIFEST ROWS ===")
print(train_df.loc[mask, columns].to_string(index=False))

print("\n=== EXPERIMENT B ARTIFACTS ===")
for path in sorted(ROOT_COMBINED.rglob("*")):
    if path.is_file() and any(
        term in path.name.lower()
        for term in [
            "experiment_b", "cache", "feature", "window",
            "checkpoint", "model"
        ]
    ):
        print(path.relative_to(ROOT_COMBINED))


In [ ]:

from pathlib import Path
import pandas as pd

cache_dir = ROOT_COMBINED / "experiment_b_feature_cache"
metadata_path = cache_dir / "experiment_b_v1_strict" / "train_window_metadata.csv"

if not metadata_path.exists():
    print("Strict training-window metadata not found:", metadata_path)
else:
    meta = pd.read_csv(metadata_path)

    print("=== TRAIN WINDOW METADATA ===")
    print("Rows:", len(meta))
    print("Columns:", meta.columns.tolist())

    # Search all text columns for the BP9/DP9 recording names.
    text_cols = meta.select_dtypes(include=["object", "string"]).columns
    mask = pd.Series(False, index=meta.index)

    for col in text_cols:
        values = meta[col].astype(str)
        mask |= values.str.contains(
            r"BP9_Asthma|DP9_Asthma",
            case=False,
            regex=True,
            na=False,
        )

    matches = meta.loc[mask]

    print("\n=== BP9 / DP9 CACHED WINDOW ROWS ===")
    print("Matching rows:", len(matches))

    if matches.empty:
        print("No BP9/DP9 filenames found in the cached metadata.")
    else:
        print(matches.to_string(index=False))

        print("\n=== MATCHES BY RECORDING IDENTIFIER ===")
        for col in text_cols:
            values = matches[col].astype(str)
            selected = values[
                values.str.contains(
                    r"BP9_Asthma|DP9_Asthma",
                    case=False,
                    regex=True,
                    na=False,
                )
            ]
            if not selected.empty:
                print(f"\nColumn: {col}")
                print(selected.value_counts().to_string())


In [ ]:

import numpy as np
from pathlib import Path

strict_dir = (
    ROOT_COMBINED
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

for filename in [
    "train_features.npz",
    "validation_features.npz",
    "icbhi_test_features.npz",
    "fraiwan_test_features.npz",
]:
    path = strict_dir / filename
    print(f"\n=== {filename} ===")

    if not path.exists():
        print("File not found")
        continue

    with np.load(path, allow_pickle=False) as data:
        for key in data.files:
            arr = data[key]
            print(
                f"{key}: shape={arr.shape}, dtype={arr.dtype}"
            )


In [ ]:

import numpy as np
import pandas as pd

strict_dir = (
    ROOT_COMBINED
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

meta = pd.read_csv(strict_dir / "train_window_metadata.csv")

with np.load(strict_dir / "train_features.npz", allow_pickle=False) as data:
    features = {key: data[key] for key in data.files}

assert len(meta) == len(features["labels"]), (
    f"Metadata rows ({len(meta)}) do not match feature rows "
    f"({len(features['labels'])})."
)

recording_ids = meta["source_recording_id"].astype(str)

bp_mask = recording_ids.str.startswith("BP9_Asthma")
dp_mask = recording_ids.str.startswith("DP9_Asthma")

bp_idx = np.flatnonzero(bp_mask.to_numpy())
dp_idx = np.flatnonzero(dp_mask.to_numpy())

print("BP9 row indices:", bp_idx.tolist())
print("DP9 row indices:", dp_idx.tolist())

print("\n=== FEATURE EQUIVALENCE ===")
if len(bp_idx) != len(dp_idx):
    print("Different window counts; do not remove rows automatically.")
else:
    for name in ["logmel", "mfcc", "chroma", "labels"]:
        bp = features[name][bp_idx]
        dp = features[name][dp_idx]
        print(
            f"{name}: exact_equal={np.array_equal(bp, dp)}"
        )

print("\n=== PROPOSED ROW REMOVAL ===")
print("Training rows before:", len(meta))
print("DP9 rows to exclude:", len(dp_idx))
print("Training rows after:", len(meta) - len(dp_idx))


In [ ]:

from pathlib import Path
import shutil
import numpy as np
import pandas as pd

source_dir = (
    ROOT_COMBINED
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

corrected_dir = (
    ROOT_COMBINED
    / "experiment_b_v1_strict_deduplicated"
)

if corrected_dir.exists():
    raise FileExistsError(
        f"Destination already exists; refusing to overwrite: {corrected_dir}"
    )

meta = pd.read_csv(source_dir / "train_window_metadata.csv")

assert len(meta) == 2224, f"Unexpected training metadata rows: {len(meta)}"

recording_ids = meta["source_recording_id"].astype(str)
remove_mask = recording_ids.str.startswith("DP9_Asthma")
keep_indices = np.flatnonzero(~remove_mask.to_numpy())

assert int(remove_mask.sum()) == 4, (
    f"Expected 4 duplicate DP9 windows; found {int(remove_mask.sum())}"
)

corrected_dir.mkdir(parents=True)

# Save corrected training arrays and metadata.
with np.load(source_dir / "train_features.npz", allow_pickle=False) as data:
    corrected_arrays = {
        key: data[key][keep_indices]
        for key in data.files
    }

np.savez_compressed(
    corrected_dir / "train_features.npz",
    **corrected_arrays
)

meta.iloc[keep_indices].to_csv(
    corrected_dir / "train_window_metadata.csv",
    index=False
)

# Copy validation and test files unchanged.
for filename in [
    "validation_features.npz",
    "validation_window_metadata.csv",
    "icbhi_test_features.npz",
    "icbhi_test_window_metadata.csv",
    "fraiwan_test_features.npz",
    "fraiwan_test_window_metadata.csv",
]:
    shutil.copy2(source_dir / filename, corrected_dir / filename)

print("Corrected cache created:", corrected_dir)
print("Original training windows:", len(meta))
print("Removed duplicate windows:", int(remove_mask.sum()))
print("Corrected training windows:", len(keep_indices))

with np.load(corrected_dir / "train_features.npz", allow_pickle=False) as data:
    print("Saved feature-array shapes:")
    for key in data.files:
        print(f"  {key}: {data[key].shape}")

print(
    "Validation/test artifacts copied unchanged:",
    all((corrected_dir / f).exists() for f in [
        "validation_features.npz",
        "icbhi_test_features.npz",
        "fraiwan_test_features.npz",
    ])
)


In [ ]:

import json
from pathlib import Path

strict_dir = (
    ROOT_COMBINED
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

config_path = strict_dir / "checkpoints" / "experiment_b_lightweight_cnn_config.json"
history_path = strict_dir / "checkpoints" / "experiment_b_lightweight_cnn_history.csv"
normalization_path = ROOT_COMBINED / "experiment_b_feature_cache" / "experiment_b_normalization_stats.json"

print("=== MODEL CONFIGURATION ===")
if config_path.exists():
    with open(config_path, "r") as f:
        config = json.load(f)
    print(json.dumps(config, indent=2))
else:
    print("Config file not found:", config_path)

print("\n=== NORMALIZATION SETTINGS ===")
if normalization_path.exists():
    with open(normalization_path, "r") as f:
        stats = json.load(f)
    print(json.dumps(stats, indent=2))
else:
    print("Normalization file not found:", normalization_path)

print("\n=== TRAINING HISTORY ===")
if history_path.exists():
    import pandas as pd
    history = pd.read_csv(history_path)
    print("Columns:", history.columns.tolist())
    print("History rows:", len(history))
    print(history.tail(5).to_string(index=False))
else:
    print("History file not found:", history_path)


In [ ]:

from pathlib import Path
import numpy as np

# Use the ROOT_COMBINED variable already defined in your notebook.
corrected_cache = Path(ROOT_COMBINED) / "experiment_b_v1_strict_deduplicated"

print("Corrected cache:", corrected_cache)
print("Cache exists:", corrected_cache.exists())

required_files = [
    "train_features.npz",
    "val_features.npz",
    "icbhi_test_features.npz",
    "fraiwan_test_features.npz",
]

for filename in required_files:
    path = corrected_cache / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

if not corrected_cache.exists():
    raise FileNotFoundError(
        f"Corrected cache not found: {corrected_cache}"
    )

print("\nCache directory contents:")
for path in sorted(corrected_cache.iterdir()):
    print(path.name)


In [ ]:

from pathlib import Path
import numpy as np

corrected_cache = Path(ROOT_COMBINED) / "experiment_b_v1_strict_deduplicated"

feature_files = {
    "train": corrected_cache / "train_features.npz",
    "validation": corrected_cache / "validation_features.npz",
    "ICBHI_test": corrected_cache / "icbhi_test_features.npz",
    "Fraiwan_test": corrected_cache / "fraiwan_test_features.npz",
}

for split, filepath in feature_files.items():
    print(f"\n{'=' * 20} {split} {'=' * 20}")

    with np.load(filepath, allow_pickle=True) as data:
        print("Available arrays:", data.files)

        for key in data.files:
            arr = data[key]
            print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

        if "labels" in data.files:
            labels, counts = np.unique(data["labels"], return_counts=True)
            print("Label distribution:", dict(zip(labels.tolist(), counts.tolist())))


In [ ]:

from pathlib import Path

project_root = Path(PROJECT_ROOT)

search_terms = [
    "LightweightMultiFeatureCNN",
    "Experiment 5",
    "EXPERIMENT 5",
    "FIVE_CLASSES",
    "class_weight_strategy",
    "validation_recording_macro_f1",
]

extensions = {".py", ".ipynb"}

print("=== MATCHES IN PROJECT SOURCE FILES ===")

matches_found = 0

for filepath in project_root.rglob("*"):
    if not filepath.is_file() or filepath.suffix.lower() not in extensions:
        continue

    # Avoid searching large environment and version-control folders.
    if any(part in {".git", ".venv", "venv", "__pycache__"} for part in filepath.parts):
        continue

    try:
        content = filepath.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue

    matched_terms = [term for term in search_terms if term.lower() in content.lower()]

    if matched_terms:
        matches_found += 1
        print(f"\nFile: {filepath.relative_to(project_root)}")
        print("Matches:", ", ".join(matched_terms))

print(f"\nFiles with matches: {matches_found}")


In [ ]:

import json
from pathlib import Path

experiment5_notebook = (
    Path(PROJECT_ROOT)
    / "Notebooks 2"
    / "11_Lightweight_MultiFeature_CNN_5Class_Experiments.ipynb"
)

with experiment5_notebook.open("r", encoding="utf-8") as f:
    notebook_data = json.load(f)

keywords = [
    "EXPERIMENT 5",
    "FIVE_CLASSES",
    "class LightweightMultiFeatureCNN",
    "def train",
    "AdamW",
    "CrossEntropyLoss",
    "ReduceLROnPlateau",
    "early_stopping",
    "augmentation",
    "best_checkpoint",
]

for cell_index, cell in enumerate(notebook_data.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    matched = [
        keyword for keyword in keywords
        if keyword.lower() in source.lower()
    ]

    if matched:
        print(f"\n{'=' * 25} CELL {cell_index} {'=' * 25}")
        print("Matched:", ", ".join(matched))
        print(source[:14000])
        if len(source) > 14000:
            print("\n... [cell truncated]")


In [ ]:

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# --------------------------------------------------
# 1. Reproducibility and paths
# --------------------------------------------------
SEED_EXP5_COMBINED = 42
random.seed(SEED_EXP5_COMBINED)
np.random.seed(SEED_EXP5_COMBINED)
torch.manual_seed(SEED_EXP5_COMBINED)

if torch.backends.mps.is_available():
    DEVICE_EXP5_COMBINED = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE_EXP5_COMBINED = torch.device("cuda")
else:
    DEVICE_EXP5_COMBINED = torch.device("cpu")

CACHE_EXP5_COMBINED = (
    Path(ROOT_COMBINED) / "experiment_b_v1_strict_deduplicated"
)

OUTPUT_EXP5_COMBINED = (
    Path(PROJECT_ROOT)
    / "data/processed/deep_learning"
    / "combined_icbhi_fraiwan_experiment5_config_8class"
)
OUTPUT_EXP5_COMBINED.mkdir(parents=True, exist_ok=True)

CLASS_NAMES_EXP5_COMBINED = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI",
]
NUM_CLASSES_EXP5_COMBINED = len(CLASS_NAMES_EXP5_COMBINED)

SPLIT_FILES_EXP5_COMBINED = {
    "train": "train_features.npz",
    "validation": "validation_features.npz",
    "icbhi_test": "icbhi_test_features.npz",
    "fraiwan_test": "fraiwan_test_features.npz",
}

# --------------------------------------------------
# 2. Load the corrected feature arrays
# --------------------------------------------------
data_exp5_combined = {}

for split_name, filename in SPLIT_FILES_EXP5_COMBINED.items():
    filepath = CACHE_EXP5_COMBINED / filename

    if not filepath.exists():
        raise FileNotFoundError(filepath)

    with np.load(filepath, allow_pickle=True) as saved:
        data_exp5_combined[split_name] = {
            key: saved[key].copy()
            for key in ["logmel", "mfcc", "chroma", "labels", "paths"]
        }

    split = data_exp5_combined[split_name]
    split["labels"] = split["labels"].astype(np.int64)
    split["paths"] = split["paths"].astype(str)

    n = len(split["labels"])

    assert all(
        len(split[name]) == n
        for name in ["logmel", "mfcc", "chroma", "paths"]
    ), f"Array length mismatch in {split_name}"

    assert np.isin(
        split["labels"], np.arange(NUM_CLASSES_EXP5_COMBINED)
    ).all(), f"Unexpected labels in {split_name}"

    print(f"{split_name:12s}: {n:4d} windows")

# --------------------------------------------------
# 3. Fit normalization on training windows only
# --------------------------------------------------
normalization_exp5_combined = {}

for feature_name in ["logmel", "mfcc", "chroma"]:
    train_array = data_exp5_combined["train"][feature_name]

    mean = float(train_array.mean(dtype=np.float64))
    std = float(train_array.std(dtype=np.float64))

    if not np.isfinite(mean) or not np.isfinite(std) or std <= 0:
        raise ValueError(f"Invalid normalization statistics: {feature_name}")

    normalization_exp5_combined[feature_name] = {
        "mean": mean,
        "std": std,
    }

    for split_name in SPLIT_FILES_EXP5_COMBINED:
        arr = data_exp5_combined[split_name][feature_name]
        data_exp5_combined[split_name][feature_name] = (
            (arr - mean) / std
        ).astype(np.float32)

    print(f"{feature_name:8s}: mean={mean:.6f}, std={std:.6f}")

# --------------------------------------------------
# 4. Class weights: inverse square-root frequency
# --------------------------------------------------
train_labels_exp5_combined = data_exp5_combined["train"]["labels"]

class_counts_exp5_combined = np.bincount(
    train_labels_exp5_combined,
    minlength=NUM_CLASSES_EXP5_COMBINED,
).astype(np.int64)

if np.any(class_counts_exp5_combined == 0):
    raise ValueError("At least one class is absent from training.")

class_weights_np_exp5_combined = (
    1.0 / np.sqrt(class_counts_exp5_combined.astype(np.float64))
)
class_weights_np_exp5_combined /= class_weights_np_exp5_combined.mean()

class_weights_exp5_combined = torch.tensor(
    class_weights_np_exp5_combined,
    dtype=torch.float32,
    device=DEVICE_EXP5_COMBINED,
)

print("\nTraining class counts:")
for idx, name in enumerate(CLASS_NAMES_EXP5_COMBINED):
    print(
        f"{idx}: {name:18s} "
        f"count={class_counts_exp5_combined[idx]:4d} "
        f"weight={class_weights_np_exp5_combined[idx]:.4f}"
    )

# --------------------------------------------------
# 5. Dataset with Experiment 5 augmentation
# --------------------------------------------------
class CombinedExperiment5Dataset(Dataset):
    def __init__(self, split_data, augment=False):
        self.data = split_data
        self.augment = augment

    def __len__(self):
        return len(self.data["labels"])

    def __getitem__(self, idx):
        logmel = self.data["logmel"][idx].copy()
        mfcc = self.data["mfcc"][idx].copy()
        chroma = self.data["chroma"][idx].copy()
        label = int(self.data["labels"][idx])

        if self.augment:
            # Apply one shared temporal shift to all three branches.
            shift = np.random.randint(-8, 9)
            logmel = np.roll(logmel, shift, axis=-1)
            mfcc = np.roll(mfcc, shift, axis=-1)
            chroma = np.roll(chroma, shift, axis=-1)

            logmel += np.random.normal(
                0, 0.015, logmel.shape
            ).astype(np.float32)
            mfcc += np.random.normal(
                0, 0.010, mfcc.shape
            ).astype(np.float32)
            chroma += np.random.normal(
                0, 0.010, chroma.shape
            ).astype(np.float32)

        return (
            torch.from_numpy(logmel.astype(np.float32)),
            torch.from_numpy(mfcc.astype(np.float32)),
            torch.from_numpy(chroma.astype(np.float32)),
            torch.tensor(label, dtype=torch.long),
        )

BATCH_SIZE_EXP5_COMBINED = 64

datasets_exp5_combined = {
    name: CombinedExperiment5Dataset(
        data_exp5_combined[name],
        augment=(name == "train"),
    )
    for name in SPLIT_FILES_EXP5_COMBINED
}

generator_exp5_combined = torch.Generator().manual_seed(
    SEED_EXP5_COMBINED
)

loaders_exp5_combined = {
    name: DataLoader(
        dataset,
        batch_size=BATCH_SIZE_EXP5_COMBINED,
        shuffle=(name == "train"),
        num_workers=0,
        generator=(
            generator_exp5_combined if name == "train" else None
        ),
    )
    for name, dataset in datasets_exp5_combined.items()
}

# --------------------------------------------------
# 6. Define the Experiment 5 model architecture
# --------------------------------------------------
class CompactFeatureBranchExp5(nn.Module):
    def __init__(self, input_channels=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(8, 8, kernel_size=3, padding=1, groups=8),
            nn.Conv2d(8, 16, kernel_size=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 16, kernel_size=3, padding=1, groups=16),
            nn.Conv2d(16, 24, kernel_size=1),
            nn.BatchNorm2d(24),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(24, 16),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
        )

    def forward(self, x):
        return self.projection(self.features(x))


class LightweightMultiFeatureCNNExp5(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.logmel_branch = CompactFeatureBranchExp5(1)
        self.mfcc_branch = CompactFeatureBranchExp5(1)
        self.chroma_branch = CompactFeatureBranchExp5(1)
        self.classifier = nn.Sequential(
            nn.Linear(48, 24),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(24, num_classes),
        )

    def forward(self, logmel, mfcc, chroma):
        features = torch.cat([
            self.logmel_branch(logmel),
            self.mfcc_branch(mfcc),
            self.chroma_branch(chroma),
        ], dim=1)
        return self.classifier(features)


model_exp5_combined = LightweightMultiFeatureCNNExp5(
    NUM_CLASSES_EXP5_COMBINED
).to(DEVICE_EXP5_COMBINED)

criterion_exp5_combined = nn.CrossEntropyLoss(
    weight=class_weights_exp5_combined
)

optimizer_exp5_combined = torch.optim.AdamW(
    model_exp5_combined.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

scheduler_exp5_combined = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_exp5_combined,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

# --------------------------------------------------
# 7. Verify shapes and save configuration
# --------------------------------------------------
with torch.no_grad():
    sample = next(iter(loaders_exp5_combined["train"]))
    sample_logmel, sample_mfcc, sample_chroma, sample_labels = sample

    output = model_exp5_combined(
        sample_logmel.to(DEVICE_EXP5_COMBINED),
        sample_mfcc.to(DEVICE_EXP5_COMBINED),
        sample_chroma.to(DEVICE_EXP5_COMBINED),
    )

parameter_count_exp5_combined = sum(
    p.numel()
    for p in model_exp5_combined.parameters()
    if p.requires_grad
)

config_exp5_combined = {
    "experiment": "Experiment 5 configuration adapted to combined ICBHI + Fraiwan",
    "architecture": "LightweightMultiFeatureCNN",
    "class_names": CLASS_NAMES_EXP5_COMBINED,
    "seed": SEED_EXP5_COMBINED,
    "device": str(DEVICE_EXP5_COMBINED),
    "batch_size": BATCH_SIZE_EXP5_COMBINED,
    "max_epochs": 60,
    "early_stopping_patience": 10,
    "optimizer": "AdamW",
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "scheduler": "ReduceLROnPlateau",
    "scheduler_mode": "max",
    "scheduler_factor": 0.5,
    "scheduler_patience": 3,
    "minimum_learning_rate": 1e-6,
    "loss": "inverse-square-root frequency weighted CrossEntropyLoss",
    "normalization": "training windows only",
    "augmentation": {
        "temporal_shift_frames": [-8, 8],
        "logmel_noise_std": 0.015,
        "mfcc_noise_std": 0.010,
        "chroma_noise_std": 0.010,
    },
    "trainable_parameters": parameter_count_exp5_combined,
    "train_class_counts": class_counts_exp5_combined.tolist(),
    "class_weights": class_weights_np_exp5_combined.tolist(),
    "normalization_stats": normalization_exp5_combined,
    "window_counts": {
        name: len(dataset)
        for name, dataset in datasets_exp5_combined.items()
    },
}

config_path_exp5_combined = (
    OUTPUT_EXP5_COMBINED / "training_config.json"
)
with config_path_exp5_combined.open("w", encoding="utf-8") as f:
    json.dump(config_exp5_combined, f, indent=2)

print("\n=== SETUP VERIFICATION ===")
print("Device:", DEVICE_EXP5_COMBINED)
print("Model output shape:", tuple(output.shape))
print("Trainable parameters:", parameter_count_exp5_combined)
print("Batch shapes:", tuple(sample_logmel.shape),
      tuple(sample_mfcc.shape), tuple(sample_chroma.shape))
print("Loss:", criterion_exp5_combined)
print("Output directory:", OUTPUT_EXP5_COMBINED)
print("Configuration saved:", config_path_exp5_combined.exists())

assert output.shape[1] == 8
assert parameter_count_exp5_combined < 10000
assert all(
    np.isfinite(data_exp5_combined[name][feature]).all()
    for name in SPLIT_FILES_EXP5_COMBINED
    for feature in ["logmel", "mfcc", "chroma"]
)

print("\nPASS: eight-class model and training data are ready.")


In [ ]:

import copy
import time
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, recall_score

MAX_EPOCHS_EXP5_COMBINED = 60
PATIENCE_EXP5_COMBINED = 10

BEST_CHECKPOINT_EXP5_COMBINED = (
    OUTPUT_EXP5_COMBINED / "best_lightweight_multifeature_cnn_8class.pt"
)
HISTORY_EXP5_COMBINED = (
    OUTPUT_EXP5_COMBINED / "training_history.csv"
)

def run_epoch_exp5_combined(training):
    model_exp5_combined.train() if training else model_exp5_combined.eval()
    loader = loaders_exp5_combined["train" if training else "validation"]

    total_loss = 0.0
    total_samples = 0
    targets_all = []
    predictions_all = []

    with torch.set_grad_enabled(training):
        for logmel, mfcc, chroma, labels in loader:
            logmel = logmel.to(DEVICE_EXP5_COMBINED)
            mfcc = mfcc.to(DEVICE_EXP5_COMBINED)
            chroma = chroma.to(DEVICE_EXP5_COMBINED)
            labels = labels.to(DEVICE_EXP5_COMBINED)

            if training:
                optimizer_exp5_combined.zero_grad(set_to_none=True)

            logits = model_exp5_combined(logmel, mfcc, chroma)
            loss = criterion_exp5_combined(logits, labels)

            if training:
                loss.backward()
                optimizer_exp5_combined.step()

            n = labels.size(0)
            total_loss += loss.item() * n
            total_samples += n
            targets_all.extend(labels.detach().cpu().numpy().tolist())
            predictions_all.extend(
                logits.argmax(dim=1).detach().cpu().numpy().tolist()
            )

    y_true = np.asarray(targets_all, dtype=int)
    y_pred = np.asarray(predictions_all, dtype=int)

    return {
        "loss": total_loss / total_samples,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": recall_score(
            y_true,
            y_pred,
            labels=list(range(NUM_CLASSES_EXP5_COMBINED)),
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=list(range(NUM_CLASSES_EXP5_COMBINED)),
            average="macro",
            zero_division=0,
        ),
    }

best_macro_f1_exp5_combined = -1.0
best_epoch_exp5_combined = 0
epochs_without_improvement_exp5_combined = 0
best_state_exp5_combined = None
history_exp5_combined = []
start_exp5_combined = time.time()

for epoch in range(1, MAX_EPOCHS_EXP5_COMBINED + 1):
    train_metrics = run_epoch_exp5_combined(training=True)
    val_metrics = run_epoch_exp5_combined(training=False)

    scheduler_exp5_combined.step(val_metrics["macro_f1"])
    current_lr = optimizer_exp5_combined.param_groups[0]["lr"]

    improved = (
        val_metrics["macro_f1"] > best_macro_f1_exp5_combined
    )

    if improved:
        best_macro_f1_exp5_combined = val_metrics["macro_f1"]
        best_epoch_exp5_combined = epoch
        epochs_without_improvement_exp5_combined = 0
        best_state_exp5_combined = copy.deepcopy(
            model_exp5_combined.state_dict()
        )

        torch.save(
            {
                "model_state_dict": best_state_exp5_combined,
                "num_classes": NUM_CLASSES_EXP5_COMBINED,
                "class_names": CLASS_NAMES_EXP5_COMBINED,
                "best_epoch": best_epoch_exp5_combined,
                "best_val_macro_f1": best_macro_f1_exp5_combined,
                "seed": SEED_EXP5_COMBINED,
                "normalization": normalization_exp5_combined,
                "config": config_exp5_combined,
                "experiment": "Experiment 5 config on combined ICBHI + Fraiwan",
            },
            BEST_CHECKPOINT_EXP5_COMBINED,
        )
        status = "BEST"
    else:
        epochs_without_improvement_exp5_combined += 1
        status = ""

    history_exp5_combined.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "learning_rate": current_lr,
        "seconds": time.time() - start_exp5_combined,
    })

    pd.DataFrame(history_exp5_combined).to_csv(
        HISTORY_EXP5_COMBINED, index=False
    )

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS_EXP5_COMBINED} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"train_F1={train_metrics['macro_f1']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_acc={val_metrics['accuracy']:.4f} | "
        f"val_bal_acc={val_metrics['balanced_accuracy']:.4f} | "
        f"val_macro_F1={val_metrics['macro_f1']:.4f} | "
        f"lr={current_lr:.2e} {status}"
    )

    if (
        epochs_without_improvement_exp5_combined
        >= PATIENCE_EXP5_COMBINED
    ):
        print(
            f"\nEarly stopping at epoch {epoch}. "
            f"Best epoch: {best_epoch_exp5_combined}"
        )
        break

if best_state_exp5_combined is None:
    raise RuntimeError("Training did not produce a best checkpoint.")

model_exp5_combined.load_state_dict(best_state_exp5_combined)

print("\n=== TRAINING COMPLETE ===")
print("Best epoch:", best_epoch_exp5_combined)
print(f"Best validation macro-F1: {best_macro_f1_exp5_combined:.4f}")
print("Checkpoint:", BEST_CHECKPOINT_EXP5_COMBINED)
print("History:", HISTORY_EXP5_COMBINED)
print(f"Elapsed minutes: {(time.time() - start_exp5_combined) / 60:.2f}")


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
import pandas as pd
import numpy as np
import torch

# Restore the saved best checkpoint.
checkpoint_exp5_combined = torch.load(
    BEST_CHECKPOINT_EXP5_COMBINED,
    map_location=DEVICE_EXP5_COMBINED,
    weights_only=True,
)

model_exp5_combined.load_state_dict(
    checkpoint_exp5_combined["model_state_dict"]
)
model_exp5_combined.eval()

print("Loaded best epoch:", checkpoint_exp5_combined["best_epoch"])
print(
    "Best validation macro-F1:",
    round(checkpoint_exp5_combined["best_val_macro_f1"], 4),
)

test_results_exp5_combined = {}

for split_name in ["icbhi_test", "fraiwan_test"]:
    loader = loaders_exp5_combined[split_name]
    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for logmel, mfcc, chroma, labels in loader:
            logits = model_exp5_combined(
                logmel.to(DEVICE_EXP5_COMBINED),
                mfcc.to(DEVICE_EXP5_COMBINED),
                chroma.to(DEVICE_EXP5_COMBINED),
            )

            predictions = logits.argmax(dim=1).cpu().numpy()

            all_targets.extend(labels.numpy().tolist())
            all_predictions.extend(predictions.tolist())

    y_true = np.asarray(all_targets, dtype=int)
    y_pred = np.asarray(all_predictions, dtype=int)
    class_ids = list(range(NUM_CLASSES_EXP5_COMBINED))

    results = {
        "targets": y_true,
        "predictions": y_pred,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy_all8": balanced_accuracy_score(
            y_true, y_pred
        ),
        "macro_f1_all8": f1_score(
            y_true,
            y_pred,
            labels=class_ids,
            average="macro",
            zero_division=0,
        ),
        "macro_f1_supported": f1_score(
            y_true,
            y_pred,
            labels=np.unique(y_true).tolist(),
            average="macro",
            zero_division=0,
        ),
    }
    test_results_exp5_combined[split_name] = results

    print(f"\n{'=' * 18} {split_name.upper()} {'=' * 18}")
    print("Windows:", len(y_true))
    print(f"Accuracy:                 {results['accuracy']:.4f}")
    print(f"Balanced accuracy (all 8): {results['balanced_accuracy_all8']:.4f}")
    print(f"Macro-F1 (all 8):          {results['macro_f1_all8']:.4f}")
    print(f"Macro-F1 (supported):      {results['macro_f1_supported']:.4f}")

    print("\nPer-class classification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=class_ids,
            target_names=CLASS_NAMES_EXP5_COMBINED,
            digits=4,
            zero_division=0,
        )
    )

    print("Confusion matrix (rows=true, columns=predicted):")
    display(
        pd.DataFrame(
            confusion_matrix(y_true, y_pred, labels=class_ids),
            index=[f"True {name}" for name in CLASS_NAMES_EXP5_COMBINED],
            columns=[f"Pred {name}" for name in CLASS_NAMES_EXP5_COMBINED],
        )
    )

# Save test predictions and summary in the new experiment directory.
prediction_frames = []
summary_rows = []

for split_name, results in test_results_exp5_combined.items():
    y_true = results["targets"]
    y_pred = results["predictions"]

    prediction_frames.append(pd.DataFrame({
        "test_split": split_name,
        "true_label": y_true,
        "true_class": [
            CLASS_NAMES_EXP5_COMBINED[i] for i in y_true
        ],
        "predicted_label": y_pred,
        "predicted_class": [
            CLASS_NAMES_EXP5_COMBINED[i] for i in y_pred
        ],
    }))

    summary_rows.append({
        "test_split": split_name,
        "windows": len(y_true),
        "accuracy": results["accuracy"],
        "balanced_accuracy_all8": results["balanced_accuracy_all8"],
        "macro_f1_all8": results["macro_f1_all8"],
        "macro_f1_supported": results["macro_f1_supported"],
    })

pd.concat(prediction_frames, ignore_index=True).to_csv(
    OUTPUT_EXP5_COMBINED / "test_predictions.csv",
    index=False,
)
pd.DataFrame(summary_rows).to_csv(
    OUTPUT_EXP5_COMBINED / "test_metrics_summary.csv",
    index=False,
)

print("\nSaved test predictions and summary to:")
print(OUTPUT_EXP5_COMBINED)


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

output_dir = Path(OUTPUT_EXP5_COMBINED)

pred_path = output_dir / "test_predictions.csv"
summary_path = output_dir / "test_metrics_summary.csv"

pred_df = pd.read_csv(pred_path)
summary_df = pd.read_csv(summary_path)

print("=== SAVED SUMMARY ===")
display(summary_df)

print("\n=== PREDICTION FILE AUDIT ===")
print("Rows:", len(pred_df))
print("Columns:", pred_df.columns.tolist())
print("\nRows by split:")
print(pred_df["test_split"].value_counts().to_string())

for split_name, group in pred_df.groupby("test_split"):
    y_true = group["true_label"].to_numpy(dtype=int)
    y_pred = group["predicted_label"].to_numpy(dtype=int)

    print(f"\n=== {split_name.upper()} ===")
    print("Rows:", len(group))
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print(
        "Balanced accuracy (all 8):",
        round(balanced_accuracy_score(y_true, y_pred), 4),
    )
    print(
        "Macro-F1 (all 8):",
        round(
            f1_score(
                y_true, y_pred,
                labels=list(range(8)),
                average="macro",
                zero_division=0,
            ),
            4,
        ),
    )

    print("\nTrue-label counts:")
    print(pd.Series(y_true).value_counts().sort_index().to_dict())

    print("Predicted-label counts:")
    print(pd.Series(y_pred).value_counts().sort_index().to_dict())

print("\nPrediction file:", pred_path.exists())
print("Summary file:", summary_path.exists())


In [ ]:

BETA_EFFECTIVE = 0.999

counts = class_counts_exp5_combined.astype(np.float64)

effective_numbers = 1.0 - np.power(BETA_EFFECTIVE, counts)
effective_weights = (1.0 - BETA_EFFECTIVE) / effective_numbers
effective_weights /= effective_weights.mean()

comparison = pd.DataFrame({
    "class": CLASS_NAMES_EXP5_COMBINED,
    "training_windows": counts.astype(int),
    "inverse_sqrt_weight": class_weights_np_exp5_combined,
    "effective_number_weight": effective_weights,
})

print(comparison.round(4).to_string(index=False))
print("\nBeta:", BETA_EFFECTIVE)
print("Mean effective-number weight:", effective_weights.mean())
print("All weights finite:", np.isfinite(effective_weights).all())
print("All weights positive:", np.all(effective_weights > 0))


In [ ]:

import copy
import json
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

# Separate experiment directory; do not overwrite the baseline.
OUTPUT_EXP5B_COMBINED = (
    Path(PROJECT_ROOT)
    / "data/processed/deep_learning"
    / "combined_icbhi_fraiwan_experiment5B_effective_weights"
)
OUTPUT_EXP5B_COMBINED.mkdir(parents=True, exist_ok=True)

CHECKPOINT_EXP5B_COMBINED = (
    OUTPUT_EXP5B_COMBINED / "best_model_8class.pt"
)
HISTORY_EXP5B_COMBINED = (
    OUTPUT_EXP5B_COMBINED / "training_history.csv"
)
CONFIG_EXP5B_COMBINED = (
    OUTPUT_EXP5B_COMBINED / "training_config.json"
)

# Verify the existing data and label configuration.
assert len(data_exp5_combined["train"]["labels"]) == 2220
assert len(data_exp5_combined["validation"]["labels"]) == 536
assert len(class_counts_exp5_combined) == 8
assert np.all(effective_weights > 0)
assert np.isfinite(effective_weights).all()

# Reproducible fresh initialization.
torch.manual_seed(SEED_EXP5_COMBINED)
np.random.seed(SEED_EXP5_COMBINED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED_EXP5_COMBINED)

model_exp5b_combined = LightweightMultiFeatureCNNExp5(
    num_classes=NUM_CLASSES_EXP5_COMBINED
).to(DEVICE_EXP5_COMBINED)

effective_weights_exp5b = torch.tensor(
    effective_weights,
    dtype=torch.float32,
    device=DEVICE_EXP5_COMBINED,
)

criterion_exp5b_combined = nn.CrossEntropyLoss(
    weight=effective_weights_exp5b
)

optimizer_exp5b_combined = torch.optim.AdamW(
    model_exp5b_combined.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

scheduler_exp5b_combined = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_exp5b_combined,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

config_exp5b_combined = {
    **config_exp5_combined,
    "experiment": "Experiment 5B: effective-number weighted cross-entropy",
    "beta": BETA_EFFECTIVE,
    "loss": "effective-number weighted CrossEntropyLoss",
    "class_weights": effective_weights.tolist(),
    "checkpoint": str(CHECKPOINT_EXP5B_COMBINED),
}

with CONFIG_EXP5B_COMBINED.open("w", encoding="utf-8") as f:
    json.dump(config_exp5b_combined, f, indent=2)

print("=== EXPERIMENT 5B SETUP ===")
print("Device:", DEVICE_EXP5_COMBINED)
print("Trainable parameters:", sum(
    p.numel() for p in model_exp5b_combined.parameters()
    if p.requires_grad
))
print("Training windows:", len(data_exp5_combined["train"]["labels"]))
print("Validation windows:", len(data_exp5_combined["validation"]["labels"]))
print("Beta:", BETA_EFFECTIVE)
print("Weights:", np.round(effective_weights, 4).tolist())
print("Optimizer: AdamW")
print("Learning rate: 0.001")
print("Weight decay: 0.0001")
print("Scheduler: ReduceLROnPlateau monitoring validation macro-F1")
print("Configuration saved:", CONFIG_EXP5B_COMBINED.exists())
print("Output directory:", OUTPUT_EXP5B_COMBINED)

assert sum(
    p.numel() for p in model_exp5b_combined.parameters()
    if p.requires_grad
) == parameter_count_exp5_combined

print("\nPASS: fresh Experiment 5B model initialized.")


In [ ]:

import time
import copy
import pandas as pd
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score

MAX_EPOCHS_EXP5B = 60
PATIENCE_EXP5B = 10

best_f1_exp5b = -1.0
best_epoch_exp5b = 0
patience_count_exp5b = 0
best_state_exp5b = None
history_exp5b = []

start_exp5b = time.time()

def run_epoch_exp5b(training):
    model_exp5b_combined.train() if training else model_exp5b_combined.eval()
    loader = loaders_exp5_combined["train" if training else "validation"]

    total_loss = 0.0
    total_samples = 0
    all_targets = []
    all_predictions = []

    with torch.set_grad_enabled(training):
        for logmel, mfcc, chroma, labels in loader:
            logmel = logmel.to(DEVICE_EXP5_COMBINED)
            mfcc = mfcc.to(DEVICE_EXP5_COMBINED)
            chroma = chroma.to(DEVICE_EXP5_COMBINED)
            labels = labels.to(DEVICE_EXP5_COMBINED)

            if training:
                optimizer_exp5b_combined.zero_grad(set_to_none=True)

            logits = model_exp5b_combined(logmel, mfcc, chroma)
            loss = criterion_exp5b_combined(logits, labels)

            if training:
                loss.backward()
                optimizer_exp5b_combined.step()

            n = labels.size(0)
            total_loss += loss.item() * n
            total_samples += n
            all_targets.extend(labels.detach().cpu().numpy().tolist())
            all_predictions.extend(
                logits.argmax(dim=1).detach().cpu().numpy().tolist()
            )

    y_true = np.asarray(all_targets, dtype=int)
    y_pred = np.asarray(all_predictions, dtype=int)

    return {
        "loss": total_loss / total_samples,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": recall_score(
            y_true,
            y_pred,
            labels=list(range(NUM_CLASSES_EXP5_COMBINED)),
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=list(range(NUM_CLASSES_EXP5_COMBINED)),
            average="macro",
            zero_division=0,
        ),
    }

for epoch in range(1, MAX_EPOCHS_EXP5B + 1):
    train_metrics = run_epoch_exp5b(training=True)
    val_metrics = run_epoch_exp5b(training=False)

    scheduler_exp5b_combined.step(val_metrics["macro_f1"])
    current_lr = optimizer_exp5b_combined.param_groups[0]["lr"]

    improved = val_metrics["macro_f1"] > best_f1_exp5b

    if improved:
        best_f1_exp5b = val_metrics["macro_f1"]
        best_epoch_exp5b = epoch
        patience_count_exp5b = 0
        best_state_exp5b = copy.deepcopy(
            model_exp5b_combined.state_dict()
        )

        torch.save(
            {
                "model_state_dict": best_state_exp5b,
                "num_classes": NUM_CLASSES_EXP5_COMBINED,
                "class_names": CLASS_NAMES_EXP5_COMBINED,
                "best_epoch": best_epoch_exp5b,
                "best_val_macro_f1": best_f1_exp5b,
                "seed": SEED_EXP5_COMBINED,
                "normalization": normalization_exp5_combined,
                "config": config_exp5b_combined,
            },
            CHECKPOINT_EXP5B_COMBINED,
        )
        status = "BEST"
    else:
        patience_count_exp5b += 1
        status = ""

    history_exp5b.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "learning_rate": current_lr,
        "elapsed_seconds": time.time() - start_exp5b,
    })

    pd.DataFrame(history_exp5b).to_csv(
        HISTORY_EXP5B_COMBINED, index=False
    )

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS_EXP5B} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"train_F1={train_metrics['macro_f1']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_acc={val_metrics['accuracy']:.4f} | "
        f"val_bal_acc={val_metrics['balanced_accuracy']:.4f} | "
        f"val_macro_F1={val_metrics['macro_f1']:.4f} | "
        f"lr={current_lr:.2e} {status}"
    )

    if patience_count_exp5b >= PATIENCE_EXP5B:
        print(
            f"\nEarly stopping at epoch {epoch}; "
            f"best epoch was {best_epoch_exp5b}."
        )
        break

if best_state_exp5b is None:
    raise RuntimeError("No valid checkpoint was produced.")

model_exp5b_combined.load_state_dict(best_state_exp5b)

print("\n=== EXPERIMENT 5B TRAINING COMPLETE ===")
print("Best epoch:", best_epoch_exp5b)
print(f"Best validation macro-F1: {best_f1_exp5b:.4f}")
print("Checkpoint exists:", CHECKPOINT_EXP5B_COMBINED.exists())
print("History exists:", HISTORY_EXP5B_COMBINED.exists())
print(f"Elapsed minutes: {(time.time() - start_exp5b) / 60:.2f}")


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
import pandas as pd
import numpy as np
import torch

# Load Experiment 5B's best validation-selected checkpoint.
checkpoint_exp5b = torch.load(
    CHECKPOINT_EXP5B_COMBINED,
    map_location=DEVICE_EXP5_COMBINED,
    weights_only=True,
)

model_exp5b_combined.load_state_dict(
    checkpoint_exp5b["model_state_dict"]
)
model_exp5b_combined.eval()

print("Best epoch:", checkpoint_exp5b["best_epoch"])
print(
    "Best validation macro-F1:",
    round(checkpoint_exp5b["best_val_macro_f1"], 4),
)

results_exp5b = {}
class_ids_exp5b = list(range(NUM_CLASSES_EXP5_COMBINED))

for split_name in ["icbhi_test", "fraiwan_test"]:
    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for logmel, mfcc, chroma, labels in loaders_exp5_combined[split_name]:
            logits = model_exp5b_combined(
                logmel.to(DEVICE_EXP5_COMBINED),
                mfcc.to(DEVICE_EXP5_COMBINED),
                chroma.to(DEVICE_EXP5_COMBINED),
            )
            all_targets.extend(labels.numpy().tolist())
            all_predictions.extend(
                logits.argmax(dim=1).cpu().numpy().tolist()
            )

    y_true = np.asarray(all_targets, dtype=int)
    y_pred = np.asarray(all_predictions, dtype=int)

    results_exp5b[split_name] = {
        "targets": y_true,
        "predictions": y_pred,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy_all8": balanced_accuracy_score(y_true, y_pred),
        "macro_f1_all8": f1_score(
            y_true, y_pred, labels=class_ids_exp5b,
            average="macro", zero_division=0,
        ),
        "macro_f1_supported": f1_score(
            y_true, y_pred, labels=np.unique(y_true),
            average="macro", zero_division=0,
        ),
    }

    metrics = results_exp5b[split_name]

    print(f"\n{'=' * 18} {split_name.upper()} {'=' * 18}")
    print("Windows:", len(y_true))
    print(f"Accuracy:                  {metrics['accuracy']:.4f}")
    print(f"Balanced accuracy (all 8): {metrics['balanced_accuracy_all8']:.4f}")
    print(f"Macro-F1 (all 8):           {metrics['macro_f1_all8']:.4f}")
    print(f"Macro-F1 (supported):       {metrics['macro_f1_supported']:.4f}")

    print("\nPer-class classification report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=class_ids_exp5b,
        target_names=CLASS_NAMES_EXP5_COMBINED,
        digits=4,
        zero_division=0,
    ))

    print("Confusion matrix (rows=true, columns=predicted):")
    display(pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=class_ids_exp5b),
        index=[f"True {name}" for name in CLASS_NAMES_EXP5_COMBINED],
        columns=[f"Pred {name}" for name in CLASS_NAMES_EXP5_COMBINED],
    ))

# Save results separately from Experiment 5 baseline.
prediction_frames_exp5b = []
summary_rows_exp5b = []

for split_name, metrics in results_exp5b.items():
    y_true = metrics["targets"]
    y_pred = metrics["predictions"]

    prediction_frames_exp5b.append(pd.DataFrame({
        "test_split": split_name,
        "true_label": y_true,
        "true_class": [CLASS_NAMES_EXP5_COMBINED[i] for i in y_true],
        "predicted_label": y_pred,
        "predicted_class": [CLASS_NAMES_EXP5_COMBINED[i] for i in y_pred],
    }))

    summary_rows_exp5b.append({
        "test_split": split_name,
        "windows": len(y_true),
        "accuracy": metrics["accuracy"],
        "balanced_accuracy_all8": metrics["balanced_accuracy_all8"],
        "macro_f1_all8": metrics["macro_f1_all8"],
        "macro_f1_supported": metrics["macro_f1_supported"],
    })

pd.concat(prediction_frames_exp5b, ignore_index=True).to_csv(
    OUTPUT_EXP5B_COMBINED / "test_predictions.csv",
    index=False,
)
pd.DataFrame(summary_rows_exp5b).to_csv(
    OUTPUT_EXP5B_COMBINED / "test_metrics_summary.csv",
    index=False,
)

print("\nSaved Experiment 5B test results to:")
print(OUTPUT_EXP5B_COMBINED)


In [ ]:

from pathlib import Path
import pandas as pd

metadata_dir = Path(PROJECT_ROOT) / "data/processed/deep_learning"

metadata_files = [
    metadata_dir / "combined_icbhi_fraiwan_hf_lung_v1" /
    "experiment_b_feature_cache" / "experiment_b_v1_strict_deduplicated" /
    "validation_window_metadata.csv",

    metadata_dir / "combined_icbhi_fraiwan_hf_lung_v1" /
    "experiment_b_feature_cache" / "experiment_b_v1_strict_deduplicated" /
    "icbhi_test_window_metadata.csv",

    metadata_dir / "combined_icbhi_fraiwan_hf_lung_v1" /
    "experiment_b_feature_cache" / "experiment_b_v1_strict_deduplicated" /
    "fraiwan_test_window_metadata.csv",
]

for path in metadata_files:
    print("\n" + "=" * 70)
    print(path)
    print("Exists:", path.exists())

    if path.exists():
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())
        display(df.head(3))


In [ ]:

from pathlib import Path
import numpy as np

DATA_DIR = Path(PROJECT_ROOT) / "data" / "processed" / "deep_learning"

print("=== METADATA FILES ===")
metadata_files = list(DATA_DIR.rglob("*metadata*.csv"))

if metadata_files:
    for path in metadata_files:
        print(path)
else:
    print("No metadata CSV files found under:", DATA_DIR)

print("\n=== CORRECTED CACHE FILES ===")
cache_dir = (
    DATA_DIR
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict_deduplicated"
)

print("Cache directory:", cache_dir)
print("Exists:", cache_dir.exists())

if cache_dir.exists():
    for path in sorted(cache_dir.iterdir()):
        if path.is_file():
            print(f"{path.name}: {path.stat().st_size:,} bytes")

    for filename in [
        "train_features.npz",
        "validation_features.npz",
        "icbhi_test_features.npz",
        "fraiwan_test_features.npz",
    ]:
        path = cache_dir / filename
        if not path.exists():
            print(f"\nMissing: {filename}")
            continue

        print(f"\n--- {filename} ---")
        with np.load(path, allow_pickle=True) as data:
            print("Keys:", data.files)
            for key in data.files:
                arr = data[key]
                if arr.ndim == 1 and len(arr) <= 3000:
                    print(
                        f"{key}: shape={arr.shape}, "
                        f"dtype={arr.dtype}, "
                        f"first values={arr[:3].tolist()}"
                    )
                else:
                    print(
                        f"{key}: shape={arr.shape}, dtype={arr.dtype}"
                    )


In [ ]:

from pathlib import Path
import pandas as pd

DATA_DIR = Path(PROJECT_ROOT) / "data" / "processed" / "deep_learning"

CORRECTED_DIR = (
    DATA_DIR
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_v1_strict_deduplicated"
)

print("Corrected directory:", CORRECTED_DIR)
print("Exists:", CORRECTED_DIR.exists())

metadata_names = [
    "validation_window_metadata.csv",
    "icbhi_test_window_metadata.csv",
    "fraiwan_test_window_metadata.csv",
]

for filename in metadata_names:
    path = CORRECTED_DIR / filename
    print(f"\n{'=' * 65}\n{filename}")
    print("Exists:", path.exists())

    if path.exists():
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())
        display(df.head(5))

        for col in df.columns:
            if any(term in col.lower() for term in [
                "record", "patient", "file", "path", "window", "label", "class"
            ]):
                print(f"{col}: {df[col].nunique(dropna=False)} unique values")


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

DATA_DIR = Path(PROJECT_ROOT) / "data" / "processed" / "deep_learning"
CORRECTED_DIR = (
    DATA_DIR / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_v1_strict_deduplicated"
)

# Metadata and cached features must have matching lengths.
split_files = {
    "icbhi_test": (
        CORRECTED_DIR / "icbhi_test_window_metadata.csv",
        CORRECTED_DIR / "icbhi_test_features.npz",
    ),
    "fraiwan_test": (
        CORRECTED_DIR / "fraiwan_test_window_metadata.csv",
        CORRECTED_DIR / "fraiwan_test_features.npz",
    ),
}

print("=== ALIGNMENT CHECK ===")
metadata_by_split = {}

for split, (metadata_path, features_path) in split_files.items():
    if not metadata_path.exists() or not features_path.exists():
        raise FileNotFoundError(
            f"Missing metadata/features for {split}:\n"
            f"{metadata_path}\n{features_path}"
        )

    metadata = pd.read_csv(metadata_path)
    with np.load(features_path, allow_pickle=True) as archive:
        feature_paths = archive["paths"].astype(str)
        feature_labels = archive["labels"].astype(int)

    print(f"\n{split}:")
    print("Metadata rows:", len(metadata))
    print("Feature rows:", len(feature_labels))
    print("Label order matches:", np.array_equal(
        metadata["label_index"].to_numpy(dtype=int), feature_labels
    ))

    metadata_paths = metadata["recording_path"].astype(str).to_numpy()
    exact_path_match = np.array_equal(metadata_paths, feature_paths)
    print("Recording path order matches exactly:", exact_path_match)

    if len(metadata) != len(feature_labels):
        raise ValueError(f"Row-count mismatch in {split}")

    if not np.array_equal(
        metadata["label_index"].to_numpy(dtype=int), feature_labels
    ):
        raise ValueError(f"Label order mismatch in {split}")

    if not exact_path_match:
        raise ValueError(
            f"Recording path order mismatch in {split}. "
            "Stop before aggregating; inspect cache ordering."
        )

    metadata_by_split[split] = metadata

print("\nPASS: metadata, feature labels, and recording paths align.")

# Evaluate each saved model and aggregate mean class probabilities by recording.
def collect_window_probabilities(model, loader, device):
    model.eval()
    probabilities = []
    labels = []

    with torch.no_grad():
        for logmel, mfcc, chroma, y in loader:
            logits = model(
                logmel.to(device),
                mfcc.to(device),
                chroma.to(device),
            )
            probabilities.append(
                torch.softmax(logits, dim=1).cpu().numpy()
            )
            labels.extend(y.numpy().astype(int).tolist())

    return np.concatenate(probabilities), np.asarray(labels, dtype=int)


def aggregate_recordings(metadata, probabilities, labels):
    if len(metadata) != len(labels) or len(labels) != len(probabilities):
        raise ValueError("Metadata, labels, and probabilities have different lengths.")

    frame = metadata.copy().reset_index(drop=True)
    frame["_true_label"] = labels
    frame["_row"] = np.arange(len(frame))

    if not np.array_equal(
        frame["label_index"].to_numpy(dtype=int), labels
    ):
        raise ValueError("Labels do not align with metadata.")

    # A recording must have one ground-truth diagnosis.
    label_counts = frame.groupby("source_recording_id")["_true_label"].nunique()
    if (label_counts > 1).any():
        raise ValueError("Some recording IDs have multiple ground-truth labels.")

    frame["_prob_row"] = frame["_row"]

    records = []
    for recording_id, group in frame.groupby(
        "source_recording_id", sort=False
    ):
        rows = group["_prob_row"].to_numpy(dtype=int)
        mean_probability = probabilities[rows].mean(axis=0)

        records.append({
            "source_recording_id": recording_id,
            "patient_id": group["patient_id"].iloc[0],
            "true_label": int(group["_true_label"].iloc[0]),
            "predicted_label": int(mean_probability.argmax()),
            "num_windows": len(group),
            "mean_probability": mean_probability,
        })

    return pd.DataFrame(records)


# Reuse the trained in-memory baseline and Experiment 5B models.
# The baseline model variable must still refer to the trained Experiment 5 model.
models_for_recording_eval = {
    "Experiment5_baseline": model_exp5_combined,
    "Experiment5B_effective_weights": model_exp5b_combined,
}

recording_results = []
recording_predictions = {}

for model_name, model in models_for_recording_eval.items():
    for split in ["icbhi_test", "fraiwan_test"]:
        probabilities, labels = collect_window_probabilities(
            model,
            loaders_exp5_combined[split],
            DEVICE_EXP5_COMBINED,
        )

        metadata = metadata_by_split[split]
        records = aggregate_recordings(metadata, probabilities, labels)

        y_true = records["true_label"].to_numpy(dtype=int)
        y_pred = records["predicted_label"].to_numpy(dtype=int)

        metrics = {
            "model": model_name,
            "test_split": split,
            "recordings": len(records),
            "patients": records["patient_id"].nunique(),
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy_supported": balanced_accuracy_score(
                y_true, y_pred
            ),
            "macro_f1_all8": f1_score(
                y_true, y_pred, labels=list(range(8)),
                average="macro", zero_division=0
            ),
            "macro_f1_supported": f1_score(
                y_true, y_pred, labels=np.unique(y_true),
                average="macro", zero_division=0
            ),
        }

        recording_results.append(metrics)
        records.insert(0, "model", model_name)
        records.insert(1, "test_split", split)
        recording_predictions[(model_name, split)] = records

        print(f"\n=== {model_name} | {split} ===")
        print(pd.Series(metrics).to_string())

recording_summary = pd.DataFrame(recording_results)
display(recording_summary)

# Save separate recording-level result files.
recording_output = (
    DATA_DIR / "recording_level_comparison_experiment5_vs_5B"
)
recording_output.mkdir(parents=True, exist_ok=True)

recording_summary.to_csv(
    recording_output / "recording_level_metrics.csv", index=False
)

pd.concat(
    list(recording_predictions.values()), ignore_index=True
).drop(columns=["mean_probability"]).to_csv(
    recording_output / "recording_level_predictions.csv", index=False
)

print("\nSaved recording-level metrics to:", recording_output)


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path(PROJECT_ROOT)
SEARCH_DIRS = [
    ROOT / "data" / "processed" / "deep_learning",
    ROOT / "data" / "processed",
    ROOT / "models",
    ROOT / "artifacts",
]

# Search only existing directories; do not modify anything.
extensions = {
    ".pt", ".pth", ".pkl", ".pickle", ".joblib",
    ".onnx", ".csv", ".json"
}

found = {}

for base in SEARCH_DIRS:
    if not base.exists():
        continue

    for path in base.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in extensions:
            continue

        name = path.name.lower()

        # Keep model artifacts and files likely to contain evaluation results.
        relevant = (
            path.suffix.lower() in {
                ".pt", ".pth", ".pkl", ".pickle",
                ".joblib", ".onnx"
            }
            or any(term in name for term in [
                "metric", "result", "summary", "prediction",
                "history", "report", "classification",
                "experiment", "evaluation", "score"
            ])
        )

        if relevant:
            found[str(path)] = {
                "file": path.name,
                "extension": path.suffix.lower(),
                "size_kb": round(path.stat().st_size / 1024, 1),
                "directory": str(path.parent),
            }

inventory = pd.DataFrame(found.values())

if inventory.empty:
    print("No matching artifacts found in the searched directories.")
else:
    inventory = inventory.sort_values(
        ["directory", "file"]
    ).reset_index(drop=True)

    print("Project:", ROOT)
    print("Relevant artifacts found:", len(inventory))
    display(inventory)

    inventory_path = ROOT / "model_comparison_artifact_inventory.csv"
    inventory.to_csv(inventory_path, index=False)
    print("Inventory saved to:", inventory_path)


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = Path(PROJECT_ROOT)

# Locate relevant existing comparison files.
target_names = {
    "final_model_comparison_summary.json",
    "classical_ml_per_disease_results.csv",
    "classical_ml_predictions.csv",
    "classical_ml_per_disease_results-checkpoint.csv",
}

matches = [
    path for path in ROOT.rglob("*")
    if path.is_file() and path.name in target_names
]

print("=== EXISTING MODEL COMPARISON FILES ===")

for path in matches:
    print(f"\n{'=' * 75}")
    print("File:", path.relative_to(ROOT))

    if path.suffix.lower() == ".json":
        try:
            with path.open("r", encoding="utf-8") as f:
                content = json.load(f)
            print(json.dumps(content, indent=2)[:12000])
        except Exception as exc:
            print("Could not read JSON:", repr(exc))

    elif path.suffix.lower() == ".csv":
        try:
            df = pd.read_csv(path)
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            display(df.head(10))
        except Exception as exc:
            print("Could not read CSV:", repr(exc))

print("\n=== MODEL CHECKPOINTS AND SERIALIZED MODELS ===")

model_extensions = {".pt", ".pth", ".joblib", ".pkl", ".pickle", ".onnx"}

model_files = []
for path in ROOT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in model_extensions:
        continue
    if any(part in {".venv", "venv", "__pycache__"} for part in path.parts):
        continue
    model_files.append(path)

for path in sorted(model_files):
    print(
        f"{path.relative_to(ROOT)} | "
        f"{path.suffix.lower()} | "
        f"{path.stat().st_size / (1024**2):.2f} MB"
    )

print("\nTotal model artifacts:", len(model_files))


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = Path(PROJECT_ROOT)

keywords = [
    "result", "metric", "summary", "comparison", "evaluation",
    "report", "prediction", "per_class", "per_disease",
    "validation", "test", "benchmark", "ablation", "seed",
]

excluded_dirs = {".venv", "venv", "__pycache__", ".git"}

result_files = []
for path in ROOT.rglob("*"):
    if not path.is_file():
        continue
    if any(part in excluded_dirs for part in path.parts):
        continue
    if path.suffix.lower() not in {".csv", ".json"}:
        continue
    if any(word in path.name.lower() for word in keywords):
        result_files.append(path)

print("Potential evaluation/result files:", len(result_files))

# First, list all candidates so we can see which experiments have reports.
for i, path in enumerate(sorted(result_files), 1):
    print(f"{i:03d}. {path.relative_to(ROOT)}")

# Inspect the most useful summary/result files.
print("\n\n========== RESULT CONTENTS ==========")

for path in sorted(result_files):
    name = path.name.lower()

    # Skip large row-level prediction tables for this first inspection.
    if "prediction" in name and "summary" not in name:
        continue

    relevant = (
        path.suffix.lower() == ".json"
        or any(word in name for word in [
            "comparison", "metric", "summary", "per_class",
            "per_disease", "benchmark", "ablation", "report",
        ])
    )
    if not relevant:
        continue

    print(f"\n{'=' * 80}\nFILE: {path.relative_to(ROOT)}")

    try:
        if path.suffix.lower() == ".json":
            with path.open("r", encoding="utf-8") as f:
                obj = json.load(f)
            print(json.dumps(obj, indent=2, default=str)[:7000])
        else:
            df = pd.read_csv(path)
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            display(df.head(8))
    except Exception as exc:
        print("Inspection error:", repr(exc))


In [ ]:

from pathlib import Path
import pandas as pd
import json

ROOT = Path(PROJECT_ROOT)
AUDIT = ROOT / "reports" / "research_audit" / "common_patient_split"
FINAL = ROOT / "reports" / "final_artifacts"

files_to_check = [
    AUDIT / "final_comparison" / "complete_common_test_leaderboard.csv",
    AUDIT / "final_comparison" / "deduplicated_common_test_leaderboard.csv",
    AUDIT / "final_comparison" / "complete_test_alignment_checks.csv",
    AUDIT / "final_comparison" / "complete_common_test_alignment_checks.csv",
    AUDIT / "final_comparison" / "recomputed_additional_model_metrics.csv",
    AUDIT / "overall_metrics.csv",
    AUDIT / "patient_level_overall_metrics.csv",
    FINAL / "prediction_metric_reproduction_check.csv",
    FINAL / "prediction_metric_count_checks.csv",
    FINAL / "final_results_source_inventory.csv",
    FINAL / "comparison_manifest.json",
]

for path in files_to_check:
    if not path.exists():
        print(f"\nNOT FOUND: {path.relative_to(ROOT)}")
        continue

    print(f"\n{'=' * 85}\n{path.relative_to(ROOT)}")

    try:
        if path.suffix.lower() == ".json":
            print(json.dumps(json.loads(path.read_text()), indent=2)[:6000])
        else:
            df = pd.read_csv(path)
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            display(df.head(15))
    except Exception as exc:
        print("Could not inspect:", repr(exc))

# Explicitly inspect five-class test-evaluation availability.
print("\n\n========== FIVE-CLASS EVALUATION FILES ==========")
for path in sorted((ROOT / "data" / "processed" / "deep_learning").glob(
    "lightweight_multifeature_cnn_5class*/**/*"
)):
    if path.is_file() and any(
        term in path.name.lower()
        for term in ["test_metrics", "test_predictions", "classification_report",
                     "experiment_comparison"]
    ):
        print(path.relative_to(ROOT))


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path(PROJECT_ROOT)

search_roots = [
    ROOT / "data/processed/deep_learning",
    ROOT / "models",
    ROOT / "reports",
]

terms = ("5class", "five_class", "fiveclass")
keywords = ("test", "prediction", "metric", "comparison", "summary")

matches = []
for base in search_roots:
    if not base.exists():
        continue
    for path in base.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in {".csv", ".json"}:
            continue
        rel = str(path.relative_to(ROOT)).lower()
        if any(t in rel for t in terms) and any(k in path.name.lower() for k in keywords):
            matches.append(path)

print("Five-class result artifacts:", len(matches))
for path in matches:
    print("\n" + str(path.relative_to(ROOT)))
    if path.suffix.lower() == ".csv":
        try:
            df = pd.read_csv(path)
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            display(df.head(10))
        except Exception as e:
            print("Read error:", repr(e))
    else:
        print(path.read_text()[:2500])


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path(PROJECT_ROOT)
BASE = ROOT / "data" / "processed" / "deep_learning"

dirs = [
    BASE / "lightweight_multifeature_cnn_5class",
    BASE / "lightweight_multifeature_cnn_5class_effective_weights",
    BASE / "lightweight_multifeature_cnn_5class_focal_loss",
    BASE / "lightweight_multifeature_cnn_5class_weighted_sampler",
    BASE / "lightweight_multifeature_cnn_5class_trainonly_norm",
]

terms = ("test", "prediction", "metric", "report", "evaluation")

for directory in dirs:
    print(f"\n{'=' * 70}")
    print(directory.relative_to(ROOT))
    print("Exists:", directory.exists())

    if not directory.exists():
        continue

    files = [
        p for p in directory.rglob("*")
        if p.is_file()
        and any(t in p.name.lower() for t in terms)
        and p.suffix.lower() in {".csv", ".json"}
    ]

    if not files:
        print("No matching test/evaluation files found.")
        continue

    for path in files:
        print("\nFile:", path.relative_to(ROOT))

        if path.suffix.lower() == ".csv":
            try:
                df = pd.read_csv(path)
                print("Shape:", df.shape)
                print("Columns:", df.columns.tolist())
                display(df.head(5))
            except Exception as e:
                print("Read error:", repr(e))
        else:
            print(path.read_text()[:2000])


In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = Path(PROJECT_ROOT)
BASE = ROOT / "data" / "processed" / "deep_learning"

experiment_dirs = [
    BASE / "lightweight_multifeature_cnn_5class",
    BASE / "lightweight_multifeature_cnn_5class_effective_weights",
    BASE / "lightweight_multifeature_cnn_5class_focal_loss",
    BASE / "lightweight_multifeature_cnn_5class_weighted_sampler",
    BASE / "lightweight_multifeature_cnn_5class_trainonly_norm",
]

print("=== CHECKPOINTS AND CONFIGURATION ===")

for directory in experiment_dirs:
    print(f"\n{'=' * 70}")
    print(directory.relative_to(ROOT))

    if not directory.exists():
        print("Directory missing")
        continue

    for path in sorted(directory.iterdir()):
        if path.is_file():
            print(f"{path.name} | {path.stat().st_size:,} bytes")

            if path.suffix.lower() == ".json" and any(
                word in path.name.lower()
                for word in ["config", "manifest", "split"]
            ):
                try:
                    print(json.dumps(
                        json.loads(path.read_text()), indent=2
                    )[:3000])
                except Exception:
                    pass

print("\n=== POSSIBLE FIVE-CLASS SPLIT MANIFESTS ===")

for base in [ROOT / "data", ROOT / "reports"]:
    if not base.exists():
        continue

    for path in base.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in {".csv", ".json"}:
            continue

        name = path.name.lower()
        rel = str(path.relative_to(ROOT)).lower()

        if any(word in name for word in ["split", "manifest", "partition"]):
            if "5class" in rel or "five_class" in rel or "icbhi" in rel:
                print(path.relative_to(ROOT))

                if path.suffix.lower() == ".csv":
                    try:
                        df = pd.read_csv(path)
                        print("Shape:", df.shape)
                        print("Columns:", df.columns.tolist())
                        display(df.head(3))
                    except Exception as e:
                        print("Read error:", repr(e))


In [ ]:

from pathlib import Path
import pandas as pd
import json

ROOT = Path(PROJECT_ROOT)
OUT = ROOT / "reports" / "all_models_comparison"
OUT.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [
    ROOT / "data" / "processed",
    ROOT / "models",
    ROOT / "reports",
]

SKIP_PARTS = {".venv", "venv", ".git", "__pycache__"}
MODEL_EXTS = {".pt", ".pth", ".joblib", ".pkl", ".pickle", ".onnx"}
RESULT_EXTS = {".csv", ".json"}

artifact_rows = []
result_frames = []

for base in SEARCH_ROOTS:
    if not base.exists():
        continue

    for path in base.rglob("*"):
        if not path.is_file():
            continue
        if any(part in SKIP_PARTS for part in path.parts):
            continue

        rel = str(path.relative_to(ROOT))
        name = path.name.lower()
        ext = path.suffix.lower()

        if ext not in MODEL_EXTS | RESULT_EXTS:
            continue

        artifact_rows.append({
            "relative_path": rel,
            "filename": path.name,
            "extension": ext,
            "size_kb": round(path.stat().st_size / 1024, 2),
            "artifact_type": (
                "model_checkpoint" if ext in MODEL_EXTS else "result_or_config"
            ),
        })

        if ext == ".csv":
            # Load result-like CSVs, not raw audio-feature arrays.
            if any(word in name for word in [
                "metric", "result", "summary", "comparison", "leaderboard",
                "prediction", "history", "report", "per_disease", "per_class",
                "experiment_comparison", "overall",
            ]):
                try:
                    df = pd.read_csv(path)
                    result_frames.append({
                        "path": rel,
                        "rows": len(df),
                        "columns": df.columns.tolist(),
                        "preview": df.head(5).to_dict(orient="records"),
                    })
                except Exception as exc:
                    result_frames.append({
                        "path": rel,
                        "error": str(exc),
                    })

        elif ext == ".json" and any(word in name for word in [
            "comparison", "metric", "summary", "manifest", "config",
        ]):
            try:
                with path.open("r", encoding="utf-8") as f:
                    obj = json.load(f)
                result_frames.append({
                    "path": rel,
                    "json_preview": obj,
                })
            except Exception as exc:
                result_frames.append({
                    "path": rel,
                    "error": str(exc),
                })

inventory = pd.DataFrame(artifact_rows).drop_duplicates(
    subset=["relative_path"]
)
inventory = inventory.sort_values(
    ["artifact_type", "relative_path"]
).reset_index(drop=True)

inventory.to_csv(OUT / "all_artifact_inventory.csv", index=False)

with (OUT / "existing_result_file_catalog.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(result_frames, f, indent=2, default=str)

print("=== MASTER ARTIFACT INVENTORY ===")
print("Total artifacts:", len(inventory))
print("Model checkpoints:", (inventory["artifact_type"] == "model_checkpoint").sum())
print("Result/config files:", (inventory["artifact_type"] == "result_or_config").sum())
display(inventory)

print("\n=== EXISTING RESULT FILES ===")
for item in result_frames:
    print("\nFILE:", item["path"])
    if "rows" in item:
        print("Rows:", item["rows"])
        print("Columns:", item["columns"])
        display(pd.DataFrame(item["preview"]))
    elif "json_preview" in item:
        print(json.dumps(item["json_preview"], indent=2, default=str)[:3000])
    else:
        print(item.get("error", ""))

print("\nMaster inventory saved to:", OUT)
